# 09 — App Data Preparation

This notebook converts the **frozen Fold 3 production system** into compact, versioned artifacts for the Streamlit application.

Notebook 09 performs:

* no model training
* no hyperparameter tuning
* no model selection
* no new holdout evaluation
* no SHAP calculation
* no expensive simulation at application runtime

All expensive computation is completed during notebook processing. The Streamlit application reads precomputed artifacts and performs only lightweight deterministic calculations such as filtering, sorting, aggregation, and scenario selection.

## Frozen production system

The application uses the exact system frozen in Notebook 07 and explained in Notebook 08.

| Demand regime | Forecast method   | Inventory method         |
| ------------- | ----------------- | ------------------------ |
| Smooth        | LightGBM Tweedie  | Dynamic policy           |
| Erratic       | LightGBM Tweedie  | Dynamic policy           |
| Intermittent  | TSB               | Dynamic policy           |
| Lumpy         | Historical policy | Historical demand policy |

The application does not create a second deployment model.

## Historical application scope

The M5 data is historical. The application is a **reproducible historical decision-support demonstration**, not a live 2026 retail deployment.

The app must clearly display:

> **Forecast data as of: 2016-01-31**

Validated Fold 3 results must remain distinguishable from user-selected scenario calculations.

## App data architecture

The notebook produces a self-contained application data bundle:

```text
app_data/
├── app_manifest.json
├── app_sku_metadata.parquet
├── app_forecasts.parquet
├── app_regime_forecasts.parquet
├── app_inventory_policy.parquet
├── app_explainability.parquet
├── app_portfolio_risk.parquet
├── app_model_results.parquet
├── app_inventory_results.parquet
├── app_cost_sensitivity.parquet
└── app_validation_report.json
```

The Streamlit application should read this bundle rather than reaching into the modeling notebooks' working files.

## Application guardrails

The app must not present:

* modeled cost savings as realized business savings
* weekly in-stock rate as conventional unit fill rate
* SHAP values as causal effects
* SHAP price contributions as price elasticity
* heuristic forecast-width measures as calibrated confidence scores
* historical M5 results as current operational performance
* 14-day results as validated Fold 3 results

The primary validated inventory configuration is **7-day lead time, 1-week review, q80**.


## Section 0 — Preflight / Frozen Source Contract

Verify that the frozen Fold 3 model, evaluation artifacts, regime assignments, and Notebook 08 explainability artifacts are available before building the application data layer.

This section is **read-only**. It establishes the source-of-truth artifacts that every downstream app dataset must reference.

The final SKU routing assignments are read directly from `sku_regimes_fold3.parquet`, including the locked boundary/reclassification logic from Notebook 07.

No model, feature set, regime assignment, inventory policy, or Fold 3 evaluation result is modified.


In [6]:
# -- 09 Section 0: Preflight + Frozen Artifact Contract -------------------------------
# Read-only validation of the exact frozen Fold 3 system.
# No retraining, tuning, model selection, or policy changes.

import os
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb

warnings.filterwarnings("ignore")

print("=" * 88)
print("SECTION 0: PREFLIGHT + FROZEN ARTIFACT CONTRACT")
print("=" * 88)
print()

# ── Directory contract ------------------------------------------------------------------

PROCESSED_DIR = "../data/processed"

CALIBRATION_DIR = f"{PROCESSED_DIR}/calibration"
FEATURES_DIR = f"{PROCESSED_DIR}/features"
MODELS_DIR = f"{PROCESSED_DIR}/models"
PREDICTIONS_DIR = f"{PROCESSED_DIR}/predictions"
SEGMENTATION_DIR = f"{PROCESSED_DIR}/segmentation"
EXPLAINABILITY_DIR = f"{PREDICTIONS_DIR}/explainability"

APP_DIR = f"{PREDICTIONS_DIR}/app"

os.makedirs(APP_DIR, exist_ok=True)

print(f"Processed directory: {PROCESSED_DIR}")
print(f"App output directory: {APP_DIR}")
print()

# ── Frozen model/configuration artifacts ------------------------------------------------

required_core_artifacts = {
    "tweedie_model": (
        f"{MODELS_DIR}/tweedie_optimized_fold3.txt"
    ),
    "feature_columns": (
        f"{FEATURES_DIR}/feature_cols_v2.pkl"
    ),
    "regime_assignments": (
        f"{SEGMENTATION_DIR}/sku_regimes_fold3.parquet"
    ),
    "fold3_validation_features": (
        f"{FEATURES_DIR}/features_val_v2.parquet"
    ),
}

for name, path in required_core_artifacts.items():

    assert os.path.exists(path), (
        f"Missing required frozen artifact: {path}"
    )

    print(f"✓ {name}: {path}")

print()

# ── Fold 3 evaluation artifacts ---------------------------------------------------------

# Produced by Notebook 07.
# These are consumed by the app as frozen evaluation/results artifacts.

required_fold3_artifacts = {
    "final_predictions": (
        f"{PREDICTIONS_DIR}/final_predictions_fold3.parquet"
    ),
    "dynamic_cost_sensitivity": (
        f"{PREDICTIONS_DIR}/dynamic_cost_sensitivity_fold3.parquet"
    ),
    "static_vs_dynamic": (
        f"{PREDICTIONS_DIR}/static_vs_dynamic_headtohead_fold3.parquet"
    ),
    "stability_check": (
        f"{PREDICTIONS_DIR}/fold3_stability_check.parquet"
    ),
}

print("Notebook 07 evaluation artifacts:")

for name, path in required_fold3_artifacts.items():

    assert os.path.exists(path), (
        f"Missing required Notebook 07 artifact: {path}"
    )

    print(f"✓ {name}: {path}")

print()

# ── Notebook 08 explainability artifacts -----------------------------------------------

required_explainability_artifacts = {
    "global_shap": (
        f"{EXPLAINABILITY_DIR}/shap_global_fold3.parquet"
    ),
    "regime_shap": (
        f"{EXPLAINABILITY_DIR}/shap_by_regime_fold3.parquet"
    ),
    "department_shap": (
        f"{EXPLAINABILITY_DIR}/shap_by_department_fold3.parquet"
    ),
    "per_sku_shap": (
        f"{EXPLAINABILITY_DIR}/shap_per_sku_fold3.parquet"
    ),
    "representative_explainability": (
        f"{EXPLAINABILITY_DIR}/representative_explainability_fold3.parquet"
    ),
    "shap_validation_report": (
        f"{EXPLAINABILITY_DIR}/shap_validation_report.json"
    ),
}

print("Notebook 08 explainability artifacts:")

for name, path in required_explainability_artifacts.items():

    assert os.path.exists(path), (
        f"Missing required Notebook 08 artifact: {path}"
    )

    print(f"✓ {name}: {path}")

print()

# ── Load frozen model --------------------------------------------------------------------

model_path = required_core_artifacts["tweedie_model"]

model_tweedie_fold3 = lgb.Booster(
    model_file=model_path
)

print("Frozen model loaded:")
print(f"  Model: {model_path}")
print(
    f"  Num trees: "
    f"{model_tweedie_fold3.current_iteration():,}"
)
print()

# ── Load frozen feature contract --------------------------------------------------------

feature_cols_path = required_core_artifacts["feature_columns"]

with open(
    feature_cols_path,
    "rb"
) as f:
    FEATURE_COLS = pickle.load(f)

MODEL_FEATURES = list(
    model_tweedie_fold3.feature_name()
)

assert MODEL_FEATURES == list(FEATURE_COLS), (
    "CRITICAL: frozen model feature order does not match "
    "feature_cols_v2.pkl."
)

print(
    f"✓ Frozen feature contract verified: "
    f"{len(FEATURE_COLS):,} features"
)

print()

# ── Load frozen regime assignments -----------------------------------------------------

sku_regimes_fold3 = pd.read_parquet(
    required_core_artifacts["regime_assignments"]
)

assert sku_regimes_fold3["id"].is_unique, (
    "sku_regimes_fold3 contains duplicate SKU IDs."
)

expected_regimes = {
    "smooth",
    "erratic",
    "intermittent",
    "lumpy",
}

actual_regimes = set(
    sku_regimes_fold3["regime"].dropna().unique()
)

assert actual_regimes == expected_regimes, (
    f"Unexpected Fold 3 regimes: {actual_regimes}"
)

assert len(sku_regimes_fold3) == 30_490, (
    "Unexpected frozen Fold 3 SKU count."
)

print(
    f"✓ Frozen regime assignments verified: "
    f"{len(sku_regimes_fold3):,} SKUs"
)

print()
print("Fold 3 regime population:")

regime_summary = (
    sku_regimes_fold3["regime"]
    .value_counts()
    .rename_axis("Regime")
    .reset_index(name="SKUs")
)

regime_summary["Share"] = (
    regime_summary["SKUs"]
    / regime_summary["SKUs"].sum()
)

display(
    regime_summary.assign(
        Share=regime_summary["Share"].map(
            lambda x: f"{x:.1%}"
        )
    )
)

print()

# ── Load Fold 3 validation features ----------------------------------------------------

fold3_val = pd.read_parquet(
    required_core_artifacts["fold3_validation_features"]
)

fold3_val["date"] = pd.to_datetime(
    fold3_val["date"]
)

missing_features = [
    col
    for col in FEATURE_COLS
    if col not in fold3_val.columns
]

assert not missing_features, (
    f"Fold 3 validation data is missing "
    f"{len(missing_features)} production features."
)

assert len(fold3_val) == 11_128_850, (
    "Unexpected Fold 3 validation row count."
)

assert (
    fold3_val["date"].min()
    == pd.Timestamp("2015-02-01")
), (
    "Unexpected Fold 3 validation start date."
)

assert (
    fold3_val["date"].max()
    == pd.Timestamp("2016-01-31")
), (
    "Unexpected Fold 3 validation end date."
)

print(
    f"✓ Fold 3 validation feature contract verified: "
    f"{len(fold3_val):,} rows"
)

print(
    f"  Date range: "
    f"{fold3_val['date'].min().date()} → "
    f"{fold3_val['date'].max().date()}"
)

print()

# ── Validate explainability QA report ---------------------------------------------------

with open(
    required_explainability_artifacts[
        "shap_validation_report"
    ],
    "r",
    encoding="utf-8"
) as f:
    shap_qa = json.load(f)

assert shap_qa.get("status") == "PASS", (
    "Notebook 08 explainability QA report is not PASS."
)

assert shap_qa.get("tweedie_skus") == 9_017, (
    "Unexpected Tweedie SKU count in SHAP QA report."
)

assert shap_qa.get("shap_rows") == 45_085, (
    "Unexpected SHAP row count in SHAP QA report."
)

print(
    "✓ Notebook 08 explainability QA report: PASS"
)

print()

# ── Validate key explainability artifact sizes -----------------------------------------

shap_per_sku = pd.read_parquet(
    required_explainability_artifacts["per_sku_shap"]
)

assert shap_per_sku["id"].nunique() == 9_017, (
    "Unexpected Tweedie SKU count in per-SKU SHAP artifact."
)

assert len(shap_per_sku) == 45_085, (
    "Unexpected per-SKU SHAP row count."
)

print(
    f"✓ Per-SKU SHAP coverage: "
    f"{shap_per_sku['id'].nunique():,} SKUs / "
    f"{len(shap_per_sku):,} rows"
)

print()

# ── Frozen routing contract -------------------------------------------------------------
# These are the locked Syntetos-Boylan thresholds and the resulting
# production forecast path.
#
# IMPORTANT:
# Final SKU assignments are read directly from the frozen
# sku_regimes_fold3 artifact. This table describes the locked threshold
# framework; it does NOT recreate or overwrite the final assignments.
#
# Notebook 07 incorporated the frozen boundary/reclassification logic
# when producing sku_regimes_fold3.parquet.

ROUTING_CONTRACT = {
    "adi_threshold": 1.32,
    "cv2_threshold": 0.49,
    "smooth_forecast": "LightGBM Tweedie",
    "erratic_forecast": "LightGBM Tweedie",
    "intermittent_forecast": "TSB",
    "lumpy_forecast": "Historical policy",
}

print("Frozen routing contract:")

routing_table = pd.DataFrame(
    [
        {
            "Regime": "Smooth",
            "Forecast Method": "LightGBM Tweedie",
            "Locked Threshold Context": "ADI ≤ 1.32, CV² ≤ 0.49",
        },
        {
            "Regime": "Erratic",
            "Forecast Method": "LightGBM Tweedie",
            "Locked Threshold Context": "ADI ≤ 1.32, CV² > 0.49",
        },
        {
            "Regime": "Intermittent",
            "Forecast Method": "TSB",
            "Locked Threshold Context": "ADI > 1.32, CV² ≤ 0.49",
        },
        {
            "Regime": "Lumpy",
            "Forecast Method": "Historical policy",
            "Locked Threshold Context": "ADI > 1.32, CV² > 0.49",
        },
    ]
)

display(routing_table)

print()
print(
    "Final Fold 3 SKU assignments are read directly from "
    "sku_regimes_fold3.parquet, including the locked "
    "boundary/reclassification logic."
)

print()

# ── Verify routing method coverage ------------------------------------------------------

forecast_method_map = {
    "smooth": "LightGBM Tweedie",
    "erratic": "LightGBM Tweedie",
    "intermittent": "TSB",
    "lumpy": "Historical policy",
}

assert set(forecast_method_map.keys()) == actual_regimes

# ── App metadata constants --------------------------------------------------------------

APP_METADATA = {
    "project": "M5 Retail Demand Forecasting",
    "system": "Demand-to-Inventory Decision System",
    "model": "LightGBM Tweedie",
    "model_artifact": "tweedie_optimized_fold3.txt",
    "evaluation_fold": "Fold 3",
    "data_as_of": "2016-01-31",
    "validation_start": "2015-02-01",
    "validation_end": "2016-01-31",
    "feature_count": int(len(FEATURE_COLS)),
    "tweedie_skus": 9_017,
    "total_skus": int(len(sku_regimes_fold3)),
    "regime_thresholds": {
        "adi": 1.32,
        "cv2": 0.49,
    },
    "forecast_methods": forecast_method_map,
}

print("App metadata:")

print(
    json.dumps(
        APP_METADATA,
        indent=2
    )
)

print()
print("=" * 88)
print("✓ SECTION 0 COMPLETE")
print("✓ Frozen model and feature contract verified.")
print("✓ Fold 3 evaluation artifacts verified.")
print("✓ Notebook 08 explainability artifacts verified.")
print("✓ Frozen routing contract recorded.")
print("✓ No model or policy changes made.")
print("=" * 88)

SECTION 0: PREFLIGHT + FROZEN ARTIFACT CONTRACT

Processed directory: ../data/processed
App output directory: ../data/processed/predictions/app

✓ tweedie_model: ../data/processed/models/tweedie_optimized_fold3.txt
✓ feature_columns: ../data/processed/features/feature_cols_v2.pkl
✓ regime_assignments: ../data/processed/segmentation/sku_regimes_fold3.parquet
✓ fold3_validation_features: ../data/processed/features/features_val_v2.parquet

Notebook 07 evaluation artifacts:
✓ final_predictions: ../data/processed/predictions/final_predictions_fold3.parquet
✓ dynamic_cost_sensitivity: ../data/processed/predictions/dynamic_cost_sensitivity_fold3.parquet
✓ static_vs_dynamic: ../data/processed/predictions/static_vs_dynamic_headtohead_fold3.parquet
✓ stability_check: ../data/processed/predictions/fold3_stability_check.parquet

Notebook 08 explainability artifacts:
✓ global_shap: ../data/processed/predictions/explainability/shap_global_fold3.parquet
✓ regime_shap: ../data/processed/predictions/ex

,Regime,SKUs,Share
0,intermittent,17303,56.7%
1,smooth,8111,26.6%
2,lumpy,4170,13.7%
3,erratic,906,3.0%



✓ Fold 3 validation feature contract verified: 11,128,850 rows
  Date range: 2015-02-01 → 2016-01-31

✓ Notebook 08 explainability QA report: PASS

✓ Per-SKU SHAP coverage: 9,017 SKUs / 45,085 rows

Frozen routing contract:


,Regime,Forecast Method,Locked Threshold Context
0,Smooth,LightGBM Tweedie,"ADI ≤ 1.32, CV² ≤ 0.49"
1,Erratic,LightGBM Tweedie,"ADI ≤ 1.32, CV² > 0.49"
2,Intermittent,TSB,"ADI > 1.32, CV² ≤ 0.49"
3,Lumpy,Historical policy,"ADI > 1.32, CV² > 0.49"



Final Fold 3 SKU assignments are read directly from sku_regimes_fold3.parquet, including the locked boundary/reclassification logic.

App metadata:
{
  "project": "M5 Retail Demand Forecasting",
  "system": "Demand-to-Inventory Decision System",
  "model": "LightGBM Tweedie",
  "model_artifact": "tweedie_optimized_fold3.txt",
  "evaluation_fold": "Fold 3",
  "data_as_of": "2016-01-31",
  "validation_start": "2015-02-01",
  "validation_end": "2016-01-31",
  "feature_count": 39,
  "tweedie_skus": 9017,
  "total_skus": 30490,
  "regime_thresholds": {
    "adi": 1.32,
    "cv2": 0.49
  },
  "forecast_methods": {
    "smooth": "LightGBM Tweedie",
    "erratic": "LightGBM Tweedie",
    "intermittent": "TSB",
    "lumpy": "Historical policy"
  }
}

✓ SECTION 0 COMPLETE
✓ Frozen model and feature contract verified.
✓ Fold 3 evaluation artifacts verified.
✓ Notebook 08 explainability artifacts verified.
✓ Frozen routing contract recorded.
✓ No model or policy changes made.


### Results

The frozen Fold 3 system passed the complete preflight contract.

| Check                  |                  Result |
| ---------------------- | ----------------------: |
| Frozen SKUs            |                  30,490 |
| Tweedie-routed SKUs    |                   9,017 |
| Frozen model features  |                      39 |
| LightGBM trees         |                   2,000 |
| Fold 3 validation rows |              11,128,850 |
| Validation period      | 2015-02-01 → 2016-01-31 |
| Notebook 07 artifacts  |             All present |
| Notebook 08 artifacts  |             All present |
| Per-SKU SHAP records   |                  45,085 |
| Explainability QA      |                    PASS |

The application will use the frozen Fold 3 artifacts directly. Final SKU routing is read from `sku_regimes_fold3.parquet`, including the locked boundary/reclassification logic from Notebook 07.

No model, feature set, regime assignment, or inventory policy was modified.

**Status: COMPLETE.**


## Section 1 — App Dataset Manifest

Create the version contract for the Streamlit application.

The manifest records the exact frozen model, feature version, regime thresholds, validated inventory configuration, historical data period, and expected app artifact set.

Downstream artifacts generated later in Notebook 09 are registered here before their final row counts are known. The manifest is finalized after Section 11 once all app artifacts have been generated and validated.

**Output:** `app_manifest.json`


In [5]:
# -- 09 Section 1: App Dataset Manifest -----------------------------------------------
# Establish the application version/configuration contract.
# No model training, prediction, SHAP calculation, or simulation.

print("=" * 88)
print("SECTION 1: APP DATASET MANIFEST")
print("=" * 88)
print()

# ── Required Section 0 state ------------------------------------------------------------

assert "APP_DIR" in globals()
assert "APP_METADATA" in globals()
assert "ROUTING_CONTRACT" in globals()

# ── Application version -----------------------------------------------------------------

APP_PROJECT_VERSION = "1.0.0"
APP_SCHEMA_VERSION = "1.0.0"

MODEL_VERSION = "fold3_final"
MODEL_ARCHITECTURE = "LightGBM Tweedie"

FEATURE_VERSION = "feature_cols_v2"
REGIME_VERSION = "Syntetos-Boylan-Fold3-v1"

# These are locked production configuration values from the frozen system.
SERVICE_LEVELS = [
    0.80,
    0.90,
    0.95,
    0.99,
]

PRIMARY_SERVICE_LEVEL = 0.80

LEAD_TIME_DAYS = 7
REVIEW_PERIOD_WEEKS = 1

# The initial app exposes a 7-day validated production configuration.
# Longer lead times are not represented as validated Fold 3 results.
SUPPORTED_LEAD_TIME_DAYS = [7]

FORECAST_HORIZONS_DAYS = [7, 28]

DATA_AS_OF = "2016-01-31"

VALIDATION_START = "2015-02-01"
VALIDATION_END = "2016-01-31"

# ── Artifact registry -------------------------------------------------------------------

artifact_names = [
    "app_manifest.json",
    "app_sku_metadata.parquet",
    "app_forecasts.parquet",
    "app_regime_forecasts.parquet",
    "app_inventory_policy.parquet",
    "app_explainability.parquet",
    "app_portfolio_risk.parquet",
    "app_model_results.parquet",
    "app_inventory_results.parquet",
    "app_cost_sensitivity.parquet",
    "app_validation_report.json",
]

# Row counts for datasets that have not yet been generated remain None.
# Section 11 will replace these with actual validated counts.
artifact_registry = {
    name: {
        "path": f"app/{name}",
        "row_count": None,
        "status": "pending",
    }
    for name in artifact_names
}

artifact_registry[
    "app_manifest.json"
]["status"] = "generated"

# ── Frozen source artifact registry -----------------------------------------------------

source_artifacts = {
    "model": (
        "../data/processed/models/"
        "tweedie_optimized_fold3.txt"
    ),
    "feature_columns": (
        "../data/processed/features/"
        "feature_cols_v2.pkl"
    ),
    "regime_assignments": (
        "../data/processed/segmentation/"
        "sku_regimes_fold3.parquet"
    ),
    "validation_features": (
        "../data/processed/features/"
        "features_val_v2.parquet"
    ),
    "final_predictions": (
        "../data/processed/predictions/"
        "final_predictions_fold3.parquet"
    ),
    "dynamic_cost_sensitivity": (
        "../data/processed/predictions/"
        "dynamic_cost_sensitivity_fold3.parquet"
    ),
    "static_vs_dynamic": (
        "../data/processed/predictions/"
        "static_vs_dynamic_headtohead_fold3.parquet"
    ),
    "stability_check": (
        "../data/processed/predictions/"
        "fold3_stability_check.parquet"
    ),
    "shap_global": (
        "../data/processed/predictions/"
        "explainability/shap_global_fold3.parquet"
    ),
    "shap_by_regime": (
        "../data/processed/predictions/"
        "explainability/shap_by_regime_fold3.parquet"
    ),
    "shap_by_department": (
        "../data/processed/predictions/"
        "explainability/shap_by_department_fold3.parquet"
    ),
    "shap_per_sku": (
        "../data/processed/predictions/"
        "explainability/shap_per_sku_fold3.parquet"
    ),
    "representative_explainability": (
        "../data/processed/predictions/"
        "explainability/"
        "representative_explainability_fold3.parquet"
    ),
    "shap_validation_report": (
        "../data/processed/predictions/"
        "explainability/shap_validation_report.json"
    ),
}

# ── Manifest ---------------------------------------------------------------------------

manifest = {
    "project": {
        "name": "M5 Retail Demand Forecasting",
        "system": "Demand-to-Inventory Decision System",
        "project_version": APP_PROJECT_VERSION,
    },

    "app": {
        "schema_version": APP_SCHEMA_VERSION,
    },

    "model": {
        "filename": "tweedie_optimized_fold3.txt",
        "version": MODEL_VERSION,
        "architecture": MODEL_ARCHITECTURE,
        "evaluation_fold": "Fold 3",
        "feature_version": FEATURE_VERSION,
    },

    "data": {
        "data_as_of": DATA_AS_OF,
        "validation_start": VALIDATION_START,
        "validation_end": VALIDATION_END,
    },

    "regime": {
        "version": REGIME_VERSION,
        "adi_threshold": ROUTING_CONTRACT["adi_threshold"],
        "cv2_threshold": ROUTING_CONTRACT["cv2_threshold"],
        "assignments_source": (
            "sku_regimes_fold3.parquet"
        ),
    },

    "inventory": {
        "primary_service_level": PRIMARY_SERVICE_LEVEL,
        "supported_service_levels": SERVICE_LEVELS,
        "lead_time_days": LEAD_TIME_DAYS,
        "supported_lead_time_days": (
            SUPPORTED_LEAD_TIME_DAYS
        ),
        "review_period_weeks": REVIEW_PERIOD_WEEKS,
        "validated_configuration": (
            "7-day lead time / "
            "1-week review / "
            "q80"
        ),
    },

    "forecast": {
        "supported_horizons_days": FORECAST_HORIZONS_DAYS,
        "primary_display_horizon_days": 28,
    },

    "sources": source_artifacts,

    "artifacts": artifact_registry,

    # Filled in after Section 11.
    "finalized": False,

    # Generation timestamp is intentionally created now;
    # Section 11 will refresh it for the final manifest.
    "generated_at_utc": (
        pd.Timestamp.utcnow()
        .isoformat()
    ),
}

# ── Save initial manifest ---------------------------------------------------------------

manifest_path = (
    f"{APP_DIR}/app_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

print(
    f"Saved: {manifest_path}"
)

print()
print("── Manifest configuration ──")

manifest_summary = pd.DataFrame(
    [
        {
            "Setting": "Project version",
            "Value": APP_PROJECT_VERSION,
        },
        {
            "Setting": "App schema",
            "Value": APP_SCHEMA_VERSION,
        },
        {
            "Setting": "Model",
            "Value": MODEL_ARCHITECTURE,
        },
        {
            "Setting": "Model version",
            "Value": MODEL_VERSION,
        },
        {
            "Setting": "Feature version",
            "Value": FEATURE_VERSION,
        },
        {
            "Setting": "Regime version",
            "Value": REGIME_VERSION,
        },
        {
            "Setting": "Data as of",
            "Value": DATA_AS_OF,
        },
        {
            "Setting": "Primary service level",
            "Value": "q80",
        },
        {
            "Setting": "Lead time",
            "Value": "7 days",
        },
        {
            "Setting": "Review period",
            "Value": "1 week",
        },
        {
            "Setting": "Primary forecast horizon",
            "Value": "28 days",
        },
    ]
)

display(
    manifest_summary
)

print()
print(
    f"Registered app artifacts: "
    f"{len(artifact_registry):,}"
)

print()
print("=" * 88)
print("✓ SECTION 1 COMPLETE")
print("✓ Application version/configuration contract created.")
print("✓ Frozen model and regime versions recorded.")
print("✓ Validated 7-day/q80 production configuration recorded.")
print("✓ Downstream row counts intentionally remain pending until generated.")
print("=" * 88)

SECTION 1: APP DATASET MANIFEST

Saved: ../data/processed/predictions/app/app_manifest.json

── Manifest configuration ──


,Setting,Value
0,Project version,1.0.0
1,App schema,1.0.0
2,Model,LightGBM Tweedie
3,Model version,fold3_final
4,Feature version,feature_cols_v2
5,Regime version,Syntetos-Boylan-Fold3-v1
6,Data as of,2016-01-31
7,Primary service level,q80
8,Lead time,7 days
9,Review period,1 week



Registered app artifacts: 11

✓ SECTION 1 COMPLETE
✓ Application version/configuration contract created.
✓ Frozen model and regime versions recorded.
✓ Validated 7-day/q80 production configuration recorded.
✓ Downstream row counts intentionally remain pending until generated.


### Results

The application version and configuration contract was created successfully.

| Setting                  | Value                      |
| ------------------------ | -------------------------- |
| Project version          | `1.0.0`                    |
| App schema               | `1.0.0`                    |
| Model                    | LightGBM Tweedie           |
| Model version            | `fold3_final`              |
| Feature version          | `feature_cols_v2`          |
| Regime version           | `Syntetos-Boylan-Fold3-v1` |
| Data as of               | 2016-01-31                 |
| Primary service level    | q80                        |
| Lead time                | 7 days                     |
| Review period            | 1 week                     |
| Primary forecast horizon | 28 days                    |
| Registered app artifacts | 11                         |

The manifest records the frozen model, feature, regime, and validated inventory configuration used by the application.

Downstream artifact row counts remain pending until their respective datasets are generated. The manifest will be finalized and refreshed after Section 11 once all artifacts have passed app-level validation.

**Status: COMPLETE.**

Saved:

`../data/processed/predictions/app/app_manifest.json`


## Section 2 — SKU Master / Metadata Table

Create the primary one-row-per-SKU dimension table used by the application.

The table combines:

* frozen Fold 3 SKU identity
* historical demand statistics
* frozen ADI/CV² regime assignment
* latest available demand
* latest available price
* routing method

Demand statistics are calculated only from historical information available before the Fold 3 validation period. The frozen Fold 3 regime assignment is taken directly from `sku_regimes_fold3.parquet`; the notebook does not recreate the routing logic.

**Output:** `app_sku_metadata.parquet`


In [7]:
# -- 09 Section 2: SKU Master / Metadata -----------------------------------------------
# Build the final one-row-per-SKU application dimension table.
#
# Demand statistics:
#   - use pre-Fold-3 historical demand only
#   - do not recreate frozen routing logic
#
# Regime:
#   - read directly from sku_regimes_fold3.parquet
#
# No model inference or policy changes.

print("=" * 88)
print("SECTION 2: SKU MASTER / METADATA")
print("=" * 88)
print()

# ── Required state ----------------------------------------------------------------------

assert "sku_regimes_fold3" in globals()
assert "APP_DIR" in globals()
assert "FEATURES_DIR" in globals()
assert "SEGMENTATION_DIR" in globals()

# ── Locate historical M5 source files ---------------------------------------------------

from pathlib import Path

DATA_ROOT = Path("../data")

candidate_sales_files = [
    DATA_ROOT / "raw" / "sales_train_validation.csv",
    DATA_ROOT / "raw" / "m5" / "sales_train_validation.csv",
    DATA_ROOT / "raw" / "M5" / "sales_train_validation.csv",
]

candidate_price_files = [
    DATA_ROOT / "raw" / "sell_prices.csv",
    DATA_ROOT / "raw" / "m5" / "sell_prices.csv",
    DATA_ROOT / "raw" / "M5" / "sell_prices.csv",
]

def find_existing_file(candidates, filename):
    for path in candidates:
        if path.exists():
            return path

    matches = list(
        DATA_ROOT.rglob(filename)
    )

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        raise FileExistsError(
            f"Multiple candidate {filename} files found:\n"
            + "\n".join(
                str(p)
                for p in matches
            )
        )

    raise FileNotFoundError(
        f"Could not locate {filename} under {DATA_ROOT.resolve()}"
    )


sales_path = find_existing_file(
    candidate_sales_files,
    "sales_train_validation.csv"
)

price_path = find_existing_file(
    candidate_price_files,
    "sell_prices.csv"
)

print(
    f"Sales source: {sales_path}"
)

print(
    f"Price source: {price_path}"
)

print()

# ── Load historical sales metadata ------------------------------------------------------

sales = pd.read_csv(
    sales_path
)

required_sales_columns = {
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
}

missing_sales_columns = (
    required_sales_columns
    - set(sales.columns)
)

assert not missing_sales_columns, (
    "Sales source is missing required columns: "
    f"{sorted(missing_sales_columns)}"
)

# The validation source contains d_1 ... d_1913.
# Fold 3 training ends at d_1913 / 2015-01-31 in the
# standard M5 calendar alignment.

day_columns = [
    c for c in sales.columns
    if c.startswith("d_")
]

assert day_columns, (
    "No M5 daily demand columns found."
)

print(
    f"Historical demand columns found: "
    f"{len(day_columns):,}"
)

# ── Preserve only frozen SKU population ------------------------------------------------

metadata_ids = set(
    sku_regimes_fold3["id"]
)

sales = sales[
    sales["id"].isin(
        metadata_ids
    )
].copy()

assert len(sales) == 30_490, (
    f"Expected 30,490 sales rows, got {len(sales):,}."
)

assert sales["id"].is_unique, (
    "Historical sales source contains duplicate SKU IDs."
)

# ── Restrict demand history to pre-Fold-3 period ---------------------------------------

# Use all available validation-training history in the supplied
# sales_train_validation artifact. This ends immediately before
# the Fold 3 validation period.

historical_days = day_columns

demand = sales[
    historical_days
].to_numpy(
    dtype=np.float64
)

# ── Weekly demand construction ----------------------------------------------------------

# M5 daily demand is converted to non-overlapping 7-day totals.
n_complete_weeks = demand.shape[1] // 7

weekly_demand = demand[
    :,
    :n_complete_weeks * 7
].reshape(
    demand.shape[0],
    n_complete_weeks,
    7
).sum(
    axis=2
)

# ── Historical weekly statistics --------------------------------------------------------

mean_weekly_demand = np.mean(
    weekly_demand,
    axis=1
)

median_weekly_demand = np.median(
    weekly_demand,
    axis=1
)

zero_demand_rate = np.mean(
    weekly_demand == 0,
    axis=1
)

# Most recent completed weekly demand.
latest_weekly_demand = weekly_demand[:, -1]

# Most recent daily demand.
latest_daily_demand = demand[:, -1]

# ── ADI / CV² ---------------------------------------------------------------------------

# ADI is computed from non-zero demand occurrence intervals.
# CV² uses non-zero demand magnitudes.

adi_values = np.empty(
    demand.shape[0],
    dtype=np.float64
)

cv2_values = np.empty(
    demand.shape[0],
    dtype=np.float64
)

for i in range(demand.shape[0]):

    series = demand[i]

    nonzero_positions = np.flatnonzero(
        series > 0
    )

    nonzero_values = series[
        nonzero_positions
    ]

    if len(nonzero_positions) == 0:

        adi_values[i] = np.inf
        cv2_values[i] = np.inf

        continue

    # Average interval between demand occurrences,
    # including the initial waiting interval.
    intervals = np.diff(
        np.r_[
            -1,
            nonzero_positions
        ]
    )

    adi_values[i] = intervals.mean()

    mean_nonzero = nonzero_values.mean()

    if mean_nonzero > 0 and len(nonzero_values) > 1:

        cv2_values[i] = (
            nonzero_values.std(
                ddof=1
            )
            / mean_nonzero
        ) ** 2

    else:

        cv2_values[i] = 0.0

# ── Latest price ------------------------------------------------------------------------

prices = pd.read_csv(
    price_path
)

required_price_columns = {
    "store_id",
    "item_id",
    "wm_yr_wk",
    "sell_price",
}

missing_price_columns = (
    required_price_columns
    - set(prices.columns)
)

assert not missing_price_columns, (
    "Price source is missing required columns: "
    f"{sorted(missing_price_columns)}"
)

# Latest price available for each store/item pair.
prices = (
    prices
    .sort_values(
        ["store_id", "item_id", "wm_yr_wk"]
    )
    .drop_duplicates(
        ["store_id", "item_id"],
        keep="last"
    )
)

# ── Build metadata base -----------------------------------------------------------------

metadata_base = sales[
    [
        "id",
        "item_id",
        "store_id",
        "dept_id",
        "cat_id",
        "state_id",
    ]
].copy()

metadata_base["mean_weekly_demand"] = (
    mean_weekly_demand
)

metadata_base["median_weekly_demand"] = (
    median_weekly_demand
)

metadata_base["zero_demand_rate"] = (
    zero_demand_rate
)

metadata_base["adi"] = (
    adi_values
)

metadata_base["cv2"] = (
    cv2_values
)

metadata_base["latest_available_demand"] = (
    latest_daily_demand
)

metadata_base["latest_weekly_demand"] = (
    latest_weekly_demand
)

# ── Join latest price --------------------------------------------------------------------

metadata_base = metadata_base.merge(
    prices[
        [
            "store_id",
            "item_id",
            "sell_price",
        ]
    ],
    on=[
        "store_id",
        "item_id",
    ],
    how="left",
    validate="one_to_one",
)

metadata_base = metadata_base.rename(
    columns={
        "sell_price": "price"
    }
)

# ── Join frozen regime assignment -------------------------------------------------------

metadata_base = metadata_base.merge(
    sku_regimes_fold3[
        [
            "id",
            "regime",
        ]
    ],
    on="id",
    how="left",
    validate="one_to_one",
)

# ── Routing method ----------------------------------------------------------------------

routing_method_map = {
    "smooth": "LightGBM Tweedie",
    "erratic": "LightGBM Tweedie",
    "intermittent": "TSB",
    "lumpy": "Historical policy",
}

metadata_base["routing_method"] = (
    metadata_base["regime"]
    .map(routing_method_map)
)

# ── Data-as-of ----------------------------------------------------------------------------

metadata_base["data_as_of"] = (
    pd.Timestamp("2015-01-31")
)

# ── Validate core dimensions ------------------------------------------------------------

assert len(metadata_base) == 30_490

assert metadata_base["id"].is_unique

assert metadata_base["regime"].notna().all()

assert metadata_base["routing_method"].notna().all()

assert (
    set(metadata_base["regime"].unique())
    == {
        "smooth",
        "erratic",
        "intermittent",
        "lumpy",
    }
)

# ── Validate numeric fields --------------------------------------------------------------

required_numeric_columns = [
    "mean_weekly_demand",
    "median_weekly_demand",
    "zero_demand_rate",
    "adi",
    "cv2",
    "latest_available_demand",
    "latest_weekly_demand",
]

for column in required_numeric_columns:

    assert np.isfinite(
        metadata_base[column]
    ).all(), (
        f"Non-finite values detected in {column}."
    )

assert (
    metadata_base["mean_weekly_demand"]
    >= 0
).all()

assert (
    metadata_base["median_weekly_demand"]
    >= 0
).all()

assert (
    metadata_base["zero_demand_rate"]
    .between(0, 1)
    .all()
)

assert (
    metadata_base["latest_available_demand"]
    >= 0
).all()

assert (
    metadata_base["latest_weekly_demand"]
    >= 0
).all()

# ADI/CV² can technically be zero only under degenerate data,
# but valid M5 SKUs should have positive ADI.
assert (
    metadata_base["adi"] > 0
).all()

assert (
    metadata_base["cv2"] >= 0
).all()

# Price should be present for every active M5 SKU.
assert metadata_base["price"].notna().all(), (
    "Missing latest price for at least one SKU."
)

assert np.isfinite(
    metadata_base["price"]
).all()

assert (
    metadata_base["price"] >= 0
).all()

# ── Standardize application column names -----------------------------------------------

sku_metadata_final = metadata_base[
    [
        "id",
        "item_id",
        "store_id",
        "dept_id",
        "cat_id",
        "state_id",
        "mean_weekly_demand",
        "median_weekly_demand",
        "zero_demand_rate",
        "adi",
        "cv2",
        "regime",
        "routing_method",
        "latest_available_demand",
        "latest_weekly_demand",
        "data_as_of",
        "price",
    ]
].copy()

# ── Validate final schema ----------------------------------------------------------------

expected_metadata_columns = [
    "id",
    "item_id",
    "store_id",
    "dept_id",
    "cat_id",
    "state_id",
    "mean_weekly_demand",
    "median_weekly_demand",
    "zero_demand_rate",
    "adi",
    "cv2",
    "regime",
    "routing_method",
    "latest_available_demand",
    "latest_weekly_demand",
    "data_as_of",
    "price",
]

assert (
    list(sku_metadata_final.columns)
    == expected_metadata_columns
), (
    "Unexpected final metadata column order."
)

assert not sku_metadata_final[
    [
        "id",
        "item_id",
        "store_id",
        "dept_id",
        "cat_id",
        "state_id",
        "regime",
        "routing_method",
        "data_as_of",
    ]
].isna().any().any(), (
    "Missing required categorical/date metadata detected."
)

# ── Summary ------------------------------------------------------------------------------

print("── Final SKU master coverage ──")

print(
    f"SKUs: {len(sku_metadata_final):,}"
)

print(
    f"Columns: {len(sku_metadata_final.columns):,}"
)

print()

display(
    sku_metadata_final.head(10)
)

print()

print("── Regime distribution ──")

regime_summary = (
    sku_metadata_final["regime"]
    .value_counts()
    .rename_axis("Regime")
    .reset_index(name="SKUs")
)

regime_summary["Share"] = (
    regime_summary["SKUs"]
    / regime_summary["SKUs"].sum()
)

display(
    regime_summary.assign(
        Share=regime_summary["Share"].map(
            lambda x: f"{x:.1%}"
        )
    )
)

print()

print("── Demand statistics ──")

demand_summary = pd.DataFrame(
    [
        {
            "Metric": "Mean weekly demand — median across SKUs",
            "Value": (
                f"{sku_metadata_final['mean_weekly_demand'].median():.2f}"
            ),
        },
        {
            "Metric": "Median weekly demand — median across SKUs",
            "Value": (
                f"{sku_metadata_final['median_weekly_demand'].median():.2f}"
            ),
        },
        {
            "Metric": "Zero-demand rate — median across SKUs",
            "Value": (
                f"{sku_metadata_final['zero_demand_rate'].median():.1%}"
            ),
        },
        {
            "Metric": "ADI — median across SKUs",
            "Value": (
                f"{sku_metadata_final['adi'].median():.2f}"
            ),
        },
        {
            "Metric": "CV² — median across SKUs",
            "Value": (
                f"{sku_metadata_final['cv2'].median():.2f}"
            ),
        },
    ]
)

display(
    demand_summary
)

print()

# ── Save --------------------------------------------------------------------------------

sku_metadata_path = (
    f"{APP_DIR}/app_sku_metadata.parquet"
)

sku_metadata_final.to_parquet(
    sku_metadata_path,
    index=False
)

# Keep the final dataframe under the common name used downstream.
sku_metadata = sku_metadata_final

print(
    f"Saved: {sku_metadata_path}"
)

print(
    f"Rows: {len(sku_metadata):,}"
)

print(
    f"Columns: {len(sku_metadata.columns):,}"
)

print()
print("=" * 88)
print("✓ SECTION 2 COMPLETE")
print("✓ One row created for every frozen Fold 3 SKU.")
print("✓ Historical demand statistics added.")
print("✓ ADI/CV² metadata calculated from pre-Fold-3 demand history.")
print("✓ Latest demand and price added.")
print("✓ Frozen regime assignments preserved.")
print("✓ No routing logic was recreated or modified.")
print("=" * 88)

SECTION 2: SKU MASTER / METADATA

Sales source: ..\data\raw\sales_train_validation.csv
Price source: ..\data\raw\sell_prices.csv

Historical demand columns found: 1,913
── Final SKU master coverage ──
SKUs: 30,490
Columns: 17



,id,item_id,store_id,dept_id,cat_id,state_id,mean_weekly_demand,median_weekly_demand,zero_demand_rate,adi,cv2,regime,routing_method,latest_available_demand,latest_weekly_demand,data_as_of,price
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,CA_1,HOBBIES_1,HOBBIES,CA,2.190476,1.0,0.490842,4.543943,0.270660,intermittent,TSB,1.0,9.0,2015-01-31,8.38
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,CA_1,HOBBIES_1,HOBBIES,CA,1.805861,1.0,0.263736,4.748756,0.236116,lumpy,Historical policy,0.0,1.0,2015-01-31,3.97
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,CA_1,HOBBIES_1,HOBBIES,CA,1.047619,0.0,0.663004,8.856481,0.293023,intermittent,TSB,1.0,7.0,2015-01-31,2.97
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,CA_1,HOBBIES_1,HOBBIES,CA,12.010989,12.0,0.051282,1.476080,0.585486,smooth,LightGBM Tweedie,2.0,14.0,2015-01-31,4.64
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,CA_1,HOBBIES_1,HOBBIES,CA,6.750916,7.0,0.128205,1.986501,0.406275,smooth,LightGBM Tweedie,4.0,8.0,2015-01-31,2.88
5,HOBBIES_1_006_CA_1_validation,HOBBIES_1_006,CA_1,HOBBIES_1,HOBBIES,CA,6.014652,6.0,0.289377,2.930982,0.484097,intermittent,TSB,0.0,4.0,2015-01-31,0.96
6,HOBBIES_1_007_CA_1_validation,HOBBIES_1_007,CA_1,HOBBIES_1,HOBBIES,CA,1.545788,1.0,0.373626,5.403955,0.142490,intermittent,TSB,1.0,2.0,2015-01-31,7.88
7,HOBBIES_1_008_CA_1_validation,HOBBIES_1_008,CA_1,HOBBIES_1,HOBBIES,CA,50.648352,51.0,0.117216,1.388244,0.865762,smooth,LightGBM Tweedie,1.0,54.0,2015-01-31,0.48
8,HOBBIES_1_009_CA_1_validation,HOBBIES_1_009,CA_1,HOBBIES_1,HOBBIES,CA,8.311355,7.0,0.109890,2.168182,0.787950,smooth,LightGBM Tweedie,0.0,8.0,2015-01-31,1.77
9,HOBBIES_1_010_CA_1_validation,HOBBIES_1_010,CA_1,HOBBIES_1,HOBBIES,CA,5.032967,5.0,0.065934,2.086150,0.266303,smooth,LightGBM Tweedie,2.0,2.0,2015-01-31,2.97



── Regime distribution ──


,Regime,SKUs,Share
0,intermittent,17303,56.7%
1,smooth,8111,26.6%
2,lumpy,4170,13.7%
3,erratic,906,3.0%



── Demand statistics ──


,Metric,Value
0,Mean weekly demand — median across SKUs,3.10
1,Median weekly demand — median across SKUs,2.00
2,Zero-demand rate — median across SKUs,39.2%
3,ADI — median across SKUs,3.77
4,CV² — median across SKUs,0.35



Saved: ../data/processed/predictions/app/app_sku_metadata.parquet
Rows: 30,490
Columns: 17

✓ SECTION 2 COMPLETE
✓ One row created for every frozen Fold 3 SKU.
✓ Historical demand statistics added.
✓ ADI/CV² metadata calculated from pre-Fold-3 demand history.
✓ Latest demand and price added.
✓ Frozen regime assignments preserved.
✓ No routing logic was recreated or modified.


### Results

The frozen Fold 3 SKU population was converted into the primary one-row-per-SKU application dimension table.

| Metric                    |          Result |
| ------------------------- | --------------: |
| Total SKUs                |          30,490 |
| Metadata columns          |              17 |
| Unique SKU IDs            |          30,490 |
| Regime assignments        | 30,490 / 30,490 |
| Data-as-of                |      2015-01-31 |
| Missing required metadata |               0 |

### Frozen regime distribution

| Regime       |   SKUs | Share | Routing           |
| ------------ | -----: | ----: | ----------------- |
| Intermittent | 17,303 | 56.7% | TSB               |
| Smooth       |  8,111 | 26.6% | LightGBM Tweedie  |
| Lumpy        |  4,170 | 13.7% | Historical policy |
| Erratic      |    906 |  3.0% | LightGBM Tweedie  |

### Historical demand summary

| Metric               | Median across SKUs |
| -------------------- | -----------------: |
| Mean weekly demand   |               3.10 |
| Median weekly demand |               2.00 |
| Zero-demand rate     |              39.2% |
| ADI                  |               3.77 |
| CV²                  |               0.35 |

The metadata table also contains the latest available demand, latest weekly demand, and latest available selling price for each SKU.

The demand statistics are calculated from the pre-Fold-3 historical demand history, while the final `regime` assignments are read directly from the frozen `sku_regimes_fold3.parquet` artifact. The routing logic is therefore not recreated or modified in Notebook 09.

**Status: COMPLETE.**

Saved:

`../data/processed/predictions/app/app_sku_metadata.parquet`


## Section 3 — Frozen Production Forecasts

Package the frozen Fold 3 production forecasts into an app-ready forecast table for Smooth and Erratic SKUs.

The point forecasts come directly from the frozen `final_predictions_fold3.parquet` artifact produced in Notebook 07. Forecast uncertainty is supplied by the frozen conformal calibration artifact; no new calibration or model fitting occurs here.

For each forecast date, the application-facing table provides the point forecast and the supported predictive quantiles:

* q50
* q75
* q80
* q90
* q95
* q99

The app supports:

* daily forecast data
* 7-day aggregation
* 28-day aggregation
* weekly aggregation

The primary portfolio display horizon is **28 days**.

The forecast table represents the historical Fold 3 validation period. The application must display:

> **Forecast data as of: 2016-01-31**

These are reproducible historical forecasts, not live 2026 predictions.

**Output:** `app_forecasts.parquet`


In [19]:
# -- 09 Section 3: Frozen Production Forecasts -----------------------------------------
# Package the exact frozen Fold 3 Tweedie trajectory and frozen conformal
# residual distributions into an app-ready forecast dataset.
#
# IMPORTANT:
#   final_predictions_fold3.parquet intentionally contains only the
#   8,863 eligible Smooth/Erratic SKUs used by the frozen dynamic trajectory.
#
#   The remaining 154 frozen Tweedie-routed SKUs consist of:
#       - 32 low-confidence SKUs (< 5 residuals)
#       - 122 SKUs with no monitor_sub residual history
#
#   Notebook 09 does NOT generate replacement predictions for these SKUs.
#
# No retraining, recalibration, model selection, or new model inference.

print("=" * 88)
print("SECTION 3: FROZEN PRODUCTION FORECASTS")
print("=" * 88)
print()

# ── Required state ----------------------------------------------------------------------

assert "APP_DIR" in globals()
assert "PREDICTIONS_DIR" in globals()
assert "CALIBRATION_DIR" in globals()
assert "sku_metadata" in globals()
assert "tweedie_metadata" in globals()

# ── Paths --------------------------------------------------------------------------------

final_predictions_path = (
    f"{PREDICTIONS_DIR}/final_predictions_fold3.parquet"
)

conformal_path = (
    f"{CALIBRATION_DIR}/conformal_residuals_fold3.pkl"
)

assert os.path.exists(final_predictions_path), (
    f"Missing frozen forecast artifact: "
    f"{final_predictions_path}"
)

assert os.path.exists(conformal_path), (
    f"Missing frozen conformal artifact: "
    f"{conformal_path}"
)

# ── Canonical SKU identifier -------------------------------------------------------------

def canonical_sku_id(value):
    value = str(value)

    if value.endswith("_validation"):
        value = value[:-11]

    return value


# ── Frozen point forecasts ---------------------------------------------------------------

final_predictions_fold3 = pd.read_parquet(
    final_predictions_path
).copy()

required_prediction_columns = {
    "id",
    "date",
    "yhat",
}

missing_prediction_columns = (
    required_prediction_columns
    - set(final_predictions_fold3.columns)
)

assert not missing_prediction_columns, (
    "Frozen prediction artifact is missing required columns: "
    f"{sorted(missing_prediction_columns)}"
)

final_predictions_fold3["date"] = pd.to_datetime(
    final_predictions_fold3["date"]
)

final_predictions_fold3["join_id"] = (
    final_predictions_fold3["id"]
    .map(canonical_sku_id)
)

# Frozen trajectory integrity.
assert final_predictions_fold3[
    ["id", "date"]
].duplicated().sum() == 0

assert np.isfinite(
    final_predictions_fold3["yhat"]
).all()

assert (
    final_predictions_fold3["yhat"] >= 0
).all()

assert (
    final_predictions_fold3["date"].min()
    == pd.Timestamp("2015-02-01")
)

assert (
    final_predictions_fold3["date"].max()
    == pd.Timestamp("2016-01-31")
)

forecast_ids = set(
    final_predictions_fold3["join_id"]
)

# ── Frozen Tweedie routing population ----------------------------------------------------

tweedie_metadata = sku_metadata[
    sku_metadata["regime"].isin(
        ["smooth", "erratic"]
    )
][
    [
        "id",
        "item_id",
        "store_id",
        "dept_id",
        "cat_id",
        "state_id",
        "regime",
        "routing_method",
    ]
].copy()

tweedie_metadata["join_id"] = (
    tweedie_metadata["id"]
    .map(canonical_sku_id)
)

tweedie_ids = set(
    tweedie_metadata["join_id"]
)

# The trajectory artifact is intentionally a subset of the
# 9,017 routed population.
missing_forecast_ids = sorted(
    tweedie_ids - forecast_ids
)

extra_forecast_ids = sorted(
    forecast_ids - tweedie_ids
)

assert not extra_forecast_ids, (
    "Frozen prediction artifact contains SKUs outside "
    "the frozen Smooth/Erratic routing population."
)

print("── Frozen Tweedie trajectory population ──")
print(
    f"Frozen Tweedie routing population: "
    f"{len(tweedie_ids):,}"
)
print(
    f"Frozen daily trajectory population: "
    f"{len(forecast_ids):,}"
)
print(
    f"Tweedie SKUs without trajectory: "
    f"{len(missing_forecast_ids):,}"
)

assert len(tweedie_ids) == 9_017
assert len(forecast_ids) == 8_863
assert len(missing_forecast_ids) == 154

print()
print(
    "✓ 154-SKU coverage gap is preserved as a frozen "
    "source-system fact; no replacement predictions are generated."
)
print()

# ── Load frozen conformal residual artifact ---------------------------------------------

with open(
    conformal_path,
    "rb"
) as f:
    conformal_fold3 = pickle.load(f)

assert isinstance(
    conformal_fold3,
    dict
)

assert "sku_residuals" in conformal_fold3

sku_residuals_fold3 = (
    conformal_fold3["sku_residuals"]
)

assert isinstance(
    sku_residuals_fold3,
    dict
)

# Normalize residual keys.
sku_residuals_normalized = {
    canonical_sku_id(k): np.asarray(v, dtype=float)
    for k, v in sku_residuals_fold3.items()
}

residual_ids = set(
    sku_residuals_normalized
)

print("── Frozen conformal coverage ──")

print(
    f"Tweedie routing SKUs: "
    f"{len(tweedie_ids):,}"
)

print(
    f"SKUs with monitor residual data: "
    f"{len(residual_ids & tweedie_ids):,}"
)

print(
    f"SKUs with no monitor residual data: "
    f"{len(tweedie_ids - residual_ids):,}"
)

n_with_5plus_residuals = sum(
    len(
        sku_residuals_normalized.get(
            sid,
            []
        )
    ) >= 5
    for sid in tweedie_ids
)

print(
    f"SKUs with >=5 residuals: "
    f"{n_with_5plus_residuals:,}"
)

# Exact frozen population reconciliation.
n_with_residuals = len(
    residual_ids & tweedie_ids
)

n_missing_residuals = len(
    tweedie_ids - residual_ids
)

n_below_floor = sum(
    0 < len(sku_residuals_normalized.get(sid, [])) < 5
    for sid in tweedie_ids
)

n_valid_residuals = sum(
    len(sku_residuals_normalized.get(sid, [])) >= 5
    for sid in tweedie_ids
)

assert n_with_residuals == 8_895
assert n_missing_residuals == 122
assert n_below_floor == 32
assert n_valid_residuals == 8_863

print()
print(
    "✓ Frozen conformal population reconciled:"
)
print(
    "  8,895 with residual data"
)
print(
    "  122 with no residual data"
)
print(
    "  32 below residual floor"
)
print(
    "  8,863 valid per-SKU residual distributions"
)
print()

# ── Build app daily forecast table -------------------------------------------------------

TARGET_QUANTILES = {
    "q50": 0.50,
    "q75": 0.75,
    "q80": 0.80,
    "q90": 0.90,
    "q95": 0.95,
    "q99": 0.99,
}

# Service-level-specific empirical residual quantiles.
#
# These are daily marginal predictive offsets derived from the
# frozen Fold 3 residual distributions. They are NOT described
# as calibrated confidence intervals for 7/28-day aggregates.

residual_quantiles = []

for sku_id in sorted(forecast_ids):

    residuals = sku_residuals_normalized.get(
        sku_id,
        []
    )

    # Every trajectory SKU must have >=5 residuals.
    assert len(residuals) >= 5, (
        f"Frozen trajectory SKU {sku_id} does not have "
        "the required >=5 conformal residuals."
    )

    row = {
        "join_id": sku_id,
    }

    for q_name, q_value in TARGET_QUANTILES.items():

        row[q_name] = float(
            np.percentile(
                residuals,
                q_value * 100
            )
        )

    residual_quantiles.append(row)

residual_quantiles = pd.DataFrame(
    residual_quantiles
)

assert len(residual_quantiles) == 8_863
assert residual_quantiles["join_id"].is_unique

# ── Join and construct daily predictive quantities -------------------------------------

app_forecasts = (
    final_predictions_fold3[
        [
            "id",
            "join_id",
            "date",
            "yhat",
        ]
    ]
    .merge(
        residual_quantiles,
        on="join_id",
        how="left",
        validate="many_to_one",
    )
)

assert len(app_forecasts) == len(
    final_predictions_fold3
)

app_forecasts = app_forecasts.rename(
    columns={
        "yhat": "point_forecast"
    }
)

for q_name in TARGET_QUANTILES:

    app_forecasts[q_name] = (
        app_forecasts["point_forecast"]
        + app_forecasts[q_name]
    ).clip(
        lower=0
    )

# ── Quantile monotonicity ----------------------------------------------------------------

quantile_order = [
    "q50",
    "q75",
    "q80",
    "q90",
    "q95",
    "q99",
]

for lower_q, upper_q in zip(
    quantile_order[:-1],
    quantile_order[1:],
):

    assert (
        app_forecasts[upper_q]
        >= app_forecasts[lower_q]
    ).all(), (
        f"Quantile monotonicity violated: "
        f"{upper_q} < {lower_q}."
    )

print(
    "✓ Daily predictive quantities are monotonic:"
)

print(
    "  q50 ≤ q75 ≤ q80 ≤ q90 ≤ q95 ≤ q99"
)

# ── Join frozen SKU metadata ------------------------------------------------------------

app_forecasts = (
    app_forecasts
    .merge(
        tweedie_metadata[
            [
                "join_id",
                "item_id",
                "store_id",
                "dept_id",
                "cat_id",
                "state_id",
                "regime",
                "routing_method",
            ]
        ],
        on="join_id",
        how="left",
        validate="many_to_one",
    )
)

assert app_forecasts["regime"].notna().all()

# ── Daily schema -------------------------------------------------------------------------

app_forecasts["forecast_horizon_days"] = 1

app_forecasts_daily = app_forecasts[
    [
        "id",
        "date",
        "forecast_horizon_days",
        "item_id",
        "store_id",
        "dept_id",
        "cat_id",
        "state_id",
        "regime",
        "routing_method",
        "point_forecast",
        "q50",
        "q75",
        "q80",
        "q90",
        "q95",
        "q99",
    ]
].copy()

# ── 7-day / 28-day point summaries ------------------------------------------------------
#
# IMPORTANT:
# Only point forecasts are aggregated here.
# We do NOT sum marginal quantiles and label the result
# as a calibrated 7-day/28-day predictive quantile.

daily = app_forecasts_daily.copy()

daily["week_start"] = (
    daily["date"]
    - pd.to_timedelta(
        daily["date"].dt.dayofweek,
        unit="D",
    )
)

weekly_forecasts = (
    daily
    .groupby(
        [
            "id",
            "week_start",
            "regime",
            "routing_method",
        ],
        observed=True,
    )
    .agg(
        point_forecast=("point_forecast", "sum"),
    )
    .reset_index()
)

weekly_forecasts["forecast_horizon_days"] = 7

# 28-day rolling point forecast.
forecast_28_parts = []

for sku_id, group in daily.groupby(
    "id",
    observed=True,
):

    group = (
        group
        .sort_values("date")
        .copy()
    )

    group["point_forecast"] = (
        group["point_forecast"]
        .rolling(
            window=28,
            min_periods=28,
        )
        .sum()
    )

    group = group.dropna(
        subset=["point_forecast"]
    )

    group["forecast_horizon_days"] = 28

    forecast_28_parts.append(
        group[
            [
                "id",
                "date",
                "regime",
                "routing_method",
                "forecast_horizon_days",
                "point_forecast",
            ]
        ]
    )

forecast_28 = pd.concat(
    forecast_28_parts,
    ignore_index=True,
)

# ── Save daily app forecast --------------------------------------------------------------

daily_output = (
    f"{APP_DIR}/app_forecasts.parquet"
)

app_forecasts_daily.to_parquet(
    daily_output,
    index=False,
)

# Keep downstream objects in memory.
app_forecasts_daily = app_forecasts_daily
app_forecasts_weekly = weekly_forecasts
app_forecasts_28d = forecast_28

# ── Coverage summary ---------------------------------------------------------------------

print()
print("── Frozen forecast coverage ──")

forecast_summary = pd.DataFrame(
    [
        {
            "Metric": "Frozen Tweedie routing SKUs",
            "Value": f"{len(tweedie_ids):,}",
        },
        {
            "Metric": "Frozen daily trajectory SKUs",
            "Value": f"{len(forecast_ids):,}",
        },
        {
            "Metric": "Tweedie SKUs without trajectory",
            "Value": f"{len(missing_forecast_ids):,}",
        },
        {
            "Metric": "Daily forecast rows",
            "Value": f"{len(app_forecasts_daily):,}",
        },
        {
            "Metric": "7-day point-summary rows",
            "Value": f"{len(app_forecasts_weekly):,}",
        },
        {
            "Metric": "28-day point-summary rows",
            "Value": f"{len(app_forecasts_28d):,}",
        },
        {
            "Metric": "Forecast data as of",
            "Value": "2016-01-31",
        },
    ]
)

display(
    forecast_summary
)

print()

display(
    app_forecasts_daily.head(10)
)

print()

# ── Final validation ----------------------------------------------------------------------

assert (
    app_forecasts_daily["id"].nunique()
    == 8_863
)

assert (
    len(app_forecasts_daily)
    == 3_234_995
)

assert (
    app_forecasts_daily["date"].min()
    == pd.Timestamp("2015-02-01")
)

assert (
    app_forecasts_daily["date"].max()
    == pd.Timestamp("2016-01-31")
)

assert (
    app_forecasts_daily[
        ["id", "date"]
    ].duplicated().sum()
    == 0
)

for column in [
    "point_forecast",
    "q50",
    "q75",
    "q80",
    "q90",
    "q95",
    "q99",
]:

    assert np.isfinite(
        app_forecasts_daily[column]
    ).all(), (
        f"Non-finite values in {column}."
    )

    assert (
        app_forecasts_daily[column] >= 0
    ).all(), (
        f"Negative values in {column}."
    )

print()
print(
    f"Saved: {daily_output}"
)

print(
    f"Rows: {len(app_forecasts_daily):,}"
)

print(
    f"SKUs: {app_forecasts_daily['id'].nunique():,}"
)

print(
    f"Columns: {len(app_forecasts_daily.columns):,}"
)

print()
print("=" * 88)
print("✓ SECTION 3 COMPLETE")
print("✓ Frozen Fold 3 eligible Tweedie trajectories packaged.")
print("✓ Frozen Fold 3 conformal residuals converted to daily predictive quantities.")
print("✓ 7-day and 28-day point forecasts prepared.")
print("✓ No replacement predictions generated for the 154 unavailable trajectories.")
print("✓ Historical forecast-as-of date preserved: 2016-01-31.")
print("=" * 88)

SECTION 3: FROZEN PRODUCTION FORECASTS

── Frozen Tweedie trajectory population ──
Frozen Tweedie routing population: 9,017
Frozen daily trajectory population: 8,863
Tweedie SKUs without trajectory: 154

✓ 154-SKU coverage gap is preserved as a frozen source-system fact; no replacement predictions are generated.

── Frozen conformal coverage ──
Tweedie routing SKUs: 9,017
SKUs with monitor residual data: 8,895
SKUs with no monitor residual data: 122
SKUs with >=5 residuals: 8,863

✓ Frozen conformal population reconciled:
  8,895 with residual data
  122 with no residual data
  32 below residual floor
  8,863 valid per-SKU residual distributions

✓ Daily predictive quantities are monotonic:
  q50 ≤ q75 ≤ q80 ≤ q90 ≤ q95 ≤ q99

── Frozen forecast coverage ──


,Metric,Value
0,Frozen Tweedie routing SKUs,"9,017"
1,Frozen daily trajectory SKUs,"8,863"
2,Tweedie SKUs without trajectory,154
3,Daily forecast rows,"3,234,995"
4,7-day point-summary rows,"469,739"
5,28-day point-summary rows,"2,995,694"
6,Forecast data as of,2016-01-31


,id,date,forecast_horizon_days,item_id,store_id,dept_id,cat_id,state_id,regime,routing_method,point_forecast,q50,q75,q80,q90,q95,q99
0,FOODS_1_001_CA_1_validation,2015-02-01,1,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,smooth,LightGBM Tweedie,0.969247,0.811463,1.385907,1.807755,2.824573,3.092989,5.331968
1,FOODS_1_001_CA_1_validation,2015-02-02,1,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,smooth,LightGBM Tweedie,0.743832,0.586048,1.160492,1.582340,2.599158,2.867574,5.106553
2,FOODS_1_001_CA_1_validation,2015-02-03,1,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,smooth,LightGBM Tweedie,0.712354,0.554570,1.129014,1.550862,2.567680,2.836096,5.075075
3,FOODS_1_001_CA_1_validation,2015-02-04,1,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,smooth,LightGBM Tweedie,0.680088,0.522304,1.096748,1.518596,2.535414,2.803830,5.042809
4,FOODS_1_001_CA_1_validation,2015-02-05,1,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,smooth,LightGBM Tweedie,0.674572,0.516788,1.091232,1.513080,2.529898,2.798314,5.037293
5,FOODS_1_001_CA_1_validation,2015-02-06,1,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,smooth,LightGBM Tweedie,0.726625,0.568841,1.143285,1.565133,2.581951,2.850367,5.089346
6,FOODS_1_001_CA_1_validation,2015-02-07,1,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,smooth,LightGBM Tweedie,0.829969,0.672185,1.246629,1.668476,2.685295,2.953711,5.192690
7,FOODS_1_001_CA_1_validation,2015-02-08,1,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,smooth,LightGBM Tweedie,0.167886,0.010102,0.584546,1.006394,2.023212,2.291628,4.530607
8,FOODS_1_001_CA_1_validation,2015-02-09,1,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,smooth,LightGBM Tweedie,0.778572,0.620788,1.195232,1.617080,2.633898,2.902314,5.141293
9,FOODS_1_001_CA_1_validation,2015-02-10,1,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,smooth,LightGBM Tweedie,0.687994,0.530210,1.104654,1.526502,2.543320,2.811736,5.050715




Saved: ../data/processed/predictions/app/app_forecasts.parquet
Rows: 3,234,995
SKUs: 8,863
Columns: 17

✓ SECTION 3 COMPLETE
✓ Frozen Fold 3 eligible Tweedie trajectories packaged.
✓ Frozen Fold 3 conformal residuals converted to daily predictive quantities.
✓ 7-day and 28-day point forecasts prepared.
✓ No replacement predictions generated for the 154 unavailable trajectories.
✓ Historical forecast-as-of date preserved: 2016-01-31.


## Section 3 — Frozen Production Forecasts

Package the frozen Fold 3 LightGBM Tweedie trajectories and frozen conformal uncertainty into an application-ready forecast dataset.

The point forecasts come directly from `final_predictions_fold3.parquet`. This artifact intentionally contains **8,863 eligible Smooth/Erratic SKUs × 365 validation days**. It does not contain replacement predictions for the remaining 154 routed SKUs.

The frozen population reconciliation is:

| Population                           |  SKUs |
| ------------------------------------ | ----: |
| Smooth/Erratic routed                | 9,017 |
| With monitor residual data           | 8,895 |
| Valid per-SKU residual distributions | 8,863 |
| Below five-residual floor            |    32 |
| No monitor residual history          |   122 |
| Frozen daily Tweedie trajectories    | 8,863 |

For the 8,863 eligible SKUs, daily predictive quantities are constructed from the frozen Fold 3 point forecasts and per-SKU residual distributions.

The application supports:

* daily frozen forecasts
* 7-day point-forecast aggregation
* 28-day point-forecast aggregation
* weekly point-forecast summaries

Only the **point forecast** is aggregated for the 7-day and 28-day summaries. Daily marginal quantiles are not summed and presented as calibrated aggregate predictive quantiles.

The 154 unavailable trajectories are preserved as a frozen source-system fact. Notebook 09 does not generate replacement model predictions for them.

The historical application scope remains:

> **Forecast data as of: 2016-01-31**

These are reproducible historical Fold 3 forecasts, not live 2026 predictions.

**Output:** `app_forecasts.parquet`


## Section 4 — Regime-Specific Forecasts

Create a unified one-row-per-SKU forecast table covering all four frozen demand regimes.

The table uses only frozen Notebook 07 artifacts and previously calculated application metadata. No new model predictions, recalibration, or routing decisions are made.

Forecast source by regime:

| Regime       | Frozen forecast source       | Uncertainty / interpretation           |
| ------------ | ---------------------------- | -------------------------------------- |
| Smooth       | LightGBM Tweedie Fold 3      | Frozen conformal residual distribution |
| Erratic      | LightGBM Tweedie Fold 3      | Frozen conformal residual distribution |
| Intermittent | TSB / block-bootstrap Fold 3 | Frozen bootstrap uncertainty           |
| Lumpy        | Historical policy            | Historical demand-policy estimate      |

For Smooth and Erratic SKUs, the forecast is represented on the validated **7-day lead-time basis** used by the frozen inventory system.

The 8,863 eligible Smooth/Erratic SKUs use their frozen conformal/Tweedie results. The remaining 154 Smooth/Erratic SKUs did not receive a frozen daily trajectory in Notebook 07:

* 32 had fewer than five monitor residuals and used the frozen pooled fallback for inventory calculations.
* 122 had no monitor-window residual history and therefore have no frozen Tweedie inventory forecast.

These 154 SKUs are **not given new model predictions in Notebook 09**. Where a finite application value is required, their latest historical weekly demand is retained as an explicitly labeled historical proxy rather than presented as a new ML forecast.

This table is an application-facing representation of the frozen production system, not a new forecasting model.

**Output:** `app_regime_forecasts.parquet`


In [22]:
# -- 09 Section 4: Regime-Specific Forecasts --------------------------------------------
# Build one application-facing forecast record for every frozen Fold 3 SKU.
#
# No new model predictions, recalibration, routing, or policy fitting.
#
# Smooth / Erratic:
#   - 8,863 eligible frozen Tweedie trajectories
#   - 32 low-confidence SKUs using the frozen pooled conformal fallback
#   - 122 SKUs with no frozen monitor-window residual history;
#     latest weekly demand is retained as an explicitly labeled historical proxy
#
# Intermittent:
#   - frozen TSB + block-bootstrap artifact
#
# Lumpy:
#   - frozen historical-policy artifact

print("=" * 88)
print("SECTION 4: REGIME-SPECIFIC FORECASTS")
print("=" * 88)
print()

# ── Required state ----------------------------------------------------------------------

assert "APP_DIR" in globals()
assert "PREDICTIONS_DIR" in globals()
assert "sku_metadata" in globals()
assert "PRIMARY_SERVICE_LEVEL" in globals()

# ── Frozen source paths ------------------------------------------------------------------

final_reorder_path = (
    f"{PREDICTIONS_DIR}/final_reorder_params_fold3.parquet"
)

tsb_path = (
    f"{PREDICTIONS_DIR}/tsb_safety_stock_fold3.parquet"
)

lumpy_path = (
    f"{PREDICTIONS_DIR}/lumpy_policy_params_fold3.parquet"
)

for path in [
    final_reorder_path,
    tsb_path,
    lumpy_path,
]:

    assert os.path.exists(path), (
        f"Missing frozen Notebook 07 artifact: {path}"
    )

# ── Load frozen artifacts ----------------------------------------------------------------

final_reorder_params_fold3 = pd.read_parquet(
    final_reorder_path
)

tsb_safety_stock_fold3 = pd.read_parquet(
    tsb_path
)

lumpy_policy_params_fold3 = pd.read_parquet(
    lumpy_path
)

# ── Validate source populations ----------------------------------------------------------

assert len(sku_metadata) == 30_490
assert sku_metadata["id"].is_unique

assert final_reorder_params_fold3["id"].is_unique
assert tsb_safety_stock_fold3["id"].is_unique
assert lumpy_policy_params_fold3["id"].is_unique

# ── Base SKU population ------------------------------------------------------------------

base = sku_metadata[
    [
        "id",
        "regime",
        "latest_weekly_demand",
    ]
].copy()

assert len(base) == 30_490
assert base["id"].is_unique

# ==============================================================================
# SMOOTH / ERRATIC
# ==============================================================================

print("── Smooth / Erratic ──")

smooth_erratic_base = base[
    base["regime"].isin(
        ["smooth", "erratic"]
    )
][
    [
        "id",
        "regime",
        "latest_weekly_demand",
    ]
].copy()

assert len(smooth_erratic_base) == 9_017

smooth_erratic_frozen = final_reorder_params_fold3[
    final_reorder_params_fold3["regime"].isin(
        ["smooth", "erratic"]
    )
][
    [
        "id",
        "expected_lead_time_demand",
        "safety_buffer",
        "low_confidence",
        "fallback_required",
    ]
].copy()

assert len(smooth_erratic_frozen) == 8_895
assert smooth_erratic_frozen["id"].is_unique

smooth_erratic = smooth_erratic_base.merge(
    smooth_erratic_frozen,
    on="id",
    how="left",
    validate="one_to_one",
)

assert len(smooth_erratic) == 9_017

has_frozen_forecast = (
    smooth_erratic[
        "expected_lead_time_demand"
    ].notna()
)

has_no_frozen_forecast = (
    smooth_erratic[
        "expected_lead_time_demand"
    ].isna()
)

assert has_frozen_forecast.sum() == 8_895
assert has_no_frozen_forecast.sum() == 122

# ── Start application fields -------------------------------------------------------------

smooth_erratic["forecast"] = (
    smooth_erratic[
        "expected_lead_time_demand"
    ]
)

smooth_erratic["uncertainty_value"] = (
    smooth_erratic[
        "safety_buffer"
    ]
)

smooth_erratic["forecast_method"] = (
    "LightGBM Tweedie"
)

smooth_erratic["forecast_horizon_days"] = 7
smooth_erratic["service_level"] = PRIMARY_SERVICE_LEVEL

smooth_erratic["uncertainty_method"] = (
    "Per-SKU conformal residuals"
)

smooth_erratic["forecast_source"] = (
    "Notebook 07 frozen reorder parameters"
)

smooth_erratic["forecast_status"] = (
    "Frozen production forecast"
)

# ── 32 low-confidence fallback cases ----------------------------------------------------

low_confidence_mask = (
    smooth_erratic["low_confidence"]
    == True
)

assert low_confidence_mask.sum() == 32

# These 32 have frozen expected lead-time demand,
# but their uncertainty uses the frozen pooled-regime fallback.
smooth_erratic.loc[
    low_confidence_mask,
    "uncertainty_method"
] = (
    "Pooled regime conformal fallback"
)

smooth_erratic.loc[
    low_confidence_mask,
    "forecast_status"
] = (
    "Frozen fallback forecast"
)

# ── 122 with no frozen forecast ----------------------------------------------------------

missing_forecast_mask = (
    smooth_erratic[
        "forecast"
    ].isna()
)

assert missing_forecast_mask.sum() == 122

# Do not generate a new ML prediction.
# Use the already-available historical weekly demand only as
# an explicitly labeled proxy for application completeness.
smooth_erratic.loc[
    missing_forecast_mask,
    "forecast"
] = (
    smooth_erratic.loc[
        missing_forecast_mask,
        "latest_weekly_demand",
    ]
)

smooth_erratic.loc[
    missing_forecast_mask,
    "uncertainty_value"
] = 0.0

smooth_erratic.loc[
    missing_forecast_mask,
    "uncertainty_method"
] = (
    "Unavailable — no frozen conformal history"
)

smooth_erratic.loc[
    missing_forecast_mask,
    "forecast_source"
] = (
    "Latest historical weekly demand proxy"
)

smooth_erratic.loc[
    missing_forecast_mask,
    "forecast_status"
] = (
    "Historical proxy — not ML forecast"
)

assert smooth_erratic["forecast"].notna().all()

# ── Final Smooth / Erratic output --------------------------------------------------------

smooth_erratic_output = smooth_erratic[
    [
        "id",
        "regime",
        "forecast_method",
        "forecast",
        "forecast_horizon_days",
        "uncertainty_value",
        "uncertainty_method",
        "service_level",
        "forecast_source",
        "forecast_status",
    ]
].copy()

assert len(smooth_erratic_output) == 9_017
assert smooth_erratic_output["id"].is_unique

print(
    f"Smooth / Erratic SKUs: "
    f"{len(smooth_erratic_output):,}"
)

print(
    f"  Frozen production: "
    f"{(smooth_erratic_output['forecast_status'] == 'Frozen production forecast').sum():,}"
)

print(
    f"  Frozen fallback: "
    f"{(smooth_erratic_output['forecast_status'] == 'Frozen fallback forecast').sum():,}"
)

print(
    f"  Historical proxy: "
    f"{(smooth_erratic_output['forecast_status'] == 'Historical proxy — not ML forecast').sum():,}"
)

# ==============================================================================
# INTERMITTENT
# ==============================================================================

print()
print("── Intermittent ──")

intermittent_base = base[
    base["regime"] == "intermittent"
][
    [
        "id",
        "regime",
    ]
].copy()

assert len(intermittent_base) == 17_303

intermittent = intermittent_base.merge(
    tsb_safety_stock_fold3[
        [
            "id",
            "mean_lead_time_demand",
            "safety_stock_q80",
            "low_confidence",
            "fallback_required",
        ]
    ],
    on="id",
    how="left",
    validate="one_to_one",
)

assert len(intermittent) == 17_303

assert (
    intermittent[
        "mean_lead_time_demand"
    ].notna().all()
)

assert (
    intermittent[
        "safety_stock_q80"
    ].notna().all()
)

intermittent_output = pd.DataFrame(
    {
        "id": intermittent["id"],
        "regime": intermittent["regime"],
        "forecast_method": "TSB",
        "forecast": intermittent[
            "mean_lead_time_demand"
        ],
        "forecast_horizon_days": 7,
        "uncertainty_value": intermittent[
            "safety_stock_q80"
        ],
        "uncertainty_method": (
            "TSB + block-bootstrap q80"
        ),
        "service_level": PRIMARY_SERVICE_LEVEL,
        "forecast_source": (
            "Notebook 07 frozen TSB / bootstrap artifact"
        ),
        "forecast_status": (
            "Frozen production forecast"
        ),
    }
)

assert len(intermittent_output) == 17_303
assert intermittent_output["id"].is_unique

print(
    f"Intermittent SKUs: "
    f"{len(intermittent_output):,}"
)

# ==============================================================================
# LUMPY
# ==============================================================================

print()
print("── Lumpy ──")

lumpy_base = base[
    base["regime"] == "lumpy"
][
    [
        "id",
        "regime",
    ]
].copy()

assert len(lumpy_base) == 4_170

lumpy = lumpy_base.merge(
    lumpy_policy_params_fold3[
        [
            "id",
            "avg_weekly",
            "reorder_point",
        ]
    ],
    on="id",
    how="left",
    validate="one_to_one",
)

assert len(lumpy) == 4_170

assert (
    lumpy[
        "avg_weekly"
    ].notna().all()
)

assert (
    lumpy[
        "reorder_point"
    ].notna().all()
)

lumpy_output = pd.DataFrame(
    {
        "id": lumpy["id"],
        "regime": lumpy["regime"],
        "forecast_method": "Historical policy",
        "forecast": lumpy[
            "avg_weekly"
        ],
        "forecast_horizon_days": 7,
        "uncertainty_value": (
            lumpy["reorder_point"]
            - lumpy["avg_weekly"]
        ),
        "uncertainty_method": (
            "Historical min-max policy buffer"
        ),
        "service_level": PRIMARY_SERVICE_LEVEL,
        "forecast_source": (
            "Notebook 07 frozen historical policy"
        ),
        "forecast_status": (
            "Historical policy estimate"
        ),
    }
)

assert len(lumpy_output) == 4_170
assert lumpy_output["id"].is_unique

print(
    f"Lumpy SKUs: "
    f"{len(lumpy_output):,}"
)

# ==============================================================================
# COMBINE
# ==============================================================================

regime_forecasts = pd.concat(
    [
        smooth_erratic_output,
        intermittent_output,
        lumpy_output,
    ],
    ignore_index=True,
)

# ==============================================================================
# FINAL CONTRACT
# ==============================================================================

print()
print("── Final regime forecast contract ──")

assert len(regime_forecasts) == 30_490
assert regime_forecasts["id"].is_unique

expected_regime_counts = {
    "smooth": 8_111,
    "erratic": 906,
    "intermittent": 17_303,
    "lumpy": 4_170,
}

actual_regime_counts = (
    regime_forecasts["regime"]
    .value_counts()
    .to_dict()
)

assert actual_regime_counts == expected_regime_counts, (
    f"Unexpected regime counts: "
    f"{actual_regime_counts}"
)

# Required text fields.
for column in [
    "forecast_method",
    "uncertainty_method",
    "forecast_source",
    "forecast_status",
]:

    assert (
        regime_forecasts[column].notna().all()
    ), (
        f"Missing values detected in {column}."
    )

# Required numeric fields.
for column in [
    "forecast",
    "uncertainty_value",
    "service_level",
]:

    assert np.isfinite(
        regime_forecasts[column]
    ).all(), (
        f"Non-finite values detected in {column}."
    )

assert (
    regime_forecasts["forecast"] >= 0
).all()

assert (
    regime_forecasts["uncertainty_value"] >= 0
).all()

assert (
    regime_forecasts["service_level"]
    == PRIMARY_SERVICE_LEVEL
).all()

assert (
    regime_forecasts["forecast_horizon_days"]
    == 7
).all()

# Forecast-method contract.
expected_methods = {
    "smooth": "LightGBM Tweedie",
    "erratic": "LightGBM Tweedie",
    "intermittent": "TSB",
    "lumpy": "Historical policy",
}

for regime, expected_method in expected_methods.items():

    values = regime_forecasts.loc[
        regime_forecasts["regime"] == regime,
        "forecast_method",
    ]

    assert (
        values == expected_method
    ).all(), (
        f"Forecast method mismatch for regime {regime}."
    )

# Exact Smooth / Erratic status reconciliation.
se_status_counts = (
    smooth_erratic_output[
        "forecast_status"
    ]
    .value_counts()
    .to_dict()
)

assert se_status_counts == {
    "Frozen production forecast": 8_863,
    "Frozen fallback forecast": 32,
    "Historical proxy — not ML forecast": 122,
}

# Intermittent and Lumpy status reconciliation.
assert (
    intermittent_output["forecast_status"]
    == "Frozen production forecast"
).all()

assert (
    lumpy_output["forecast_status"]
    == "Historical policy estimate"
).all()

# ── Summary ------------------------------------------------------------------------------

print()
print("── Regime distribution ──")

display(
    pd.DataFrame(
        [
            {
                "Regime": regime,
                "SKUs": int(
                    (
                        regime_forecasts["regime"]
                        == regime
                    ).sum()
                ),
                "Forecast Method": method,
            }
            for regime, method
            in expected_methods.items()
        ]
    )
)

print()
print("── Forecast status ──")

display(
    regime_forecasts[
        "forecast_status"
    ]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="SKUs")
)

# ── Save --------------------------------------------------------------------------------

regime_forecast_output = (
    f"{APP_DIR}/app_regime_forecasts.parquet"
)

regime_forecasts.to_parquet(
    regime_forecast_output,
    index=False
)

app_regime_forecasts = regime_forecasts

print()
print(
    f"Saved: {regime_forecast_output}"
)

print(
    f"Rows: {len(regime_forecasts):,}"
)

print(
    f"Columns: {len(regime_forecasts.columns):,}"
)

print()
print("=" * 88)
print("✓ SECTION 4 COMPLETE")
print("✓ One forecast record created for all 30,490 frozen SKUs.")
print("✓ Smooth/Erratic frozen Tweedie results preserved.")
print("✓ 32 low-confidence pooled conformal fallbacks preserved.")
print("✓ 122 unavailable Tweedie forecasts explicitly labeled as historical proxies.")
print("✓ Intermittent TSB / bootstrap results preserved.")
print("✓ Lumpy historical-policy estimates preserved.")
print("✓ No new model predictions or policy fitting performed.")
print("=" * 88)

SECTION 4: REGIME-SPECIFIC FORECASTS

── Smooth / Erratic ──
Smooth / Erratic SKUs: 9,017
  Frozen production: 8,863
  Frozen fallback: 32
  Historical proxy: 122

── Intermittent ──
Intermittent SKUs: 17,303

── Lumpy ──
Lumpy SKUs: 4,170

── Final regime forecast contract ──

── Regime distribution ──


,Regime,SKUs,Forecast Method
0,smooth,8111,LightGBM Tweedie
1,erratic,906,LightGBM Tweedie
2,intermittent,17303,TSB
3,lumpy,4170,Historical policy



── Forecast status ──


,Status,SKUs
0,Frozen production forecast,26166
1,Historical policy estimate,4170
2,Historical proxy — not ML forecast,122
3,Frozen fallback forecast,32



Saved: ../data/processed/predictions/app/app_regime_forecasts.parquet
Rows: 30,490
Columns: 10

✓ SECTION 4 COMPLETE
✓ One forecast record created for all 30,490 frozen SKUs.
✓ Smooth/Erratic frozen Tweedie results preserved.
✓ 32 low-confidence pooled conformal fallbacks preserved.
✓ 122 unavailable Tweedie forecasts explicitly labeled as historical proxies.
✓ Intermittent TSB / bootstrap results preserved.
✓ Lumpy historical-policy estimates preserved.
✓ No new model predictions or policy fitting performed.


## Section 4 — Regime-Specific Forecasts

Create a unified one-row-per-SKU forecast table covering the complete **30,490-SKU frozen Fold 3 population**.

The application preserves the frozen forecasting architecture:

| Regime       | Forecast method   | Uncertainty / policy source        |
| ------------ | ----------------- | ---------------------------------- |
| Smooth       | LightGBM Tweedie  | Frozen conformal residuals         |
| Erratic      | LightGBM Tweedie  | Frozen conformal residuals         |
| Intermittent | TSB               | Frozen block-bootstrap uncertainty |
| Lumpy        | Historical policy | Frozen historical demand policy    |

The Smooth/Erratic population contains 9,017 SKUs:

* **8,863** have eligible frozen Tweedie trajectories.
* **32** have fewer than five monitor residuals and use the frozen pooled-regime conformal fallback.
* **122** have no monitor-window residual history and therefore have no frozen Tweedie forecast. These are represented only by an explicitly labeled latest-weekly-demand historical proxy.

The Intermittent and Lumpy records are taken directly from their frozen Notebook 07 artifacts.

The resulting application table contains one record for every frozen SKU and records the forecast method, forecast basis, uncertainty method, service level, source artifact, and forecast status.

No model predictions, routing decisions, recalibration, or policy fitting are performed.

**Output:** `app_regime_forecasts.parquet`


## Section 5 — Inventory Policy

Package the frozen Fold 3 inventory policy into an application-ready table.

The primary validated inventory configuration is:

> **7-day lead time / 1-week review / q80**

The frozen Fold 3 inventory population contains **30,368 SKUs**. The remaining 122 Smooth/Erratic SKUs had no monitor-window residual history and therefore did not receive a frozen Smooth/Erratic inventory policy in Notebook 07.

Notebook 07 evaluated six frozen service-level scenarios:

* q50
* q75
* q80
* q90
* q95
* q99

The application exposes the validated q80 configuration and the higher-service-level alternatives q90, q95, and q99.

For each supported service level, the application records:

* SKU and regime
* inventory method
* service level
* 7-day lead time
* 1-week review period
* expected lead-time demand
* reorder point
* order quantity
* whether the row represents the validated q80 configuration

The service-level policy values are derived deterministically from the frozen Notebook 07 residual, TSB, and historical-policy artifacts using the same formulas used in the frozen Fold 3 sweep. No model training, recalibration, or inventory simulation is performed.

The 122 SKUs without a frozen inventory policy remain outside this table rather than receiving fabricated policy values.

**Output:** `app_inventory_policy.parquet`


In [29]:
# -- 09 Section 5: Inventory Policy -----------------------------------------------------
# Package the frozen Fold 3 inventory-policy configurations for the
# supported application service levels:
#
#   Primary: q80
#   Alternatives: q90, q95, q99
#
# Notebook 07 froze six service-level calculations:
#   q50, q75, q80, q90, q95, q99
#
# Notebook 09 uses only the supported application levels above.
#
# IMPORTANT:
#   - q80 is copied directly from the frozen final_reorder_params_fold3
#     artifact so the production configuration is preserved exactly.
#   - q90/q95/q99 are deterministic scenario alternatives reconstructed
#     from the frozen Notebook 07 ingredients/formulas.
#   - No model training, recalibration, or inventory simulation is performed.

print("=" * 88)
print("SECTION 5: INVENTORY POLICY")
print("=" * 88)
print()

# ── Required state ----------------------------------------------------------------------

assert "APP_DIR" in globals()
assert "PREDICTIONS_DIR" in globals()
assert "CALIBRATION_DIR" in globals()
assert "sku_metadata" in globals()
assert "PRIMARY_SERVICE_LEVEL" in globals()
assert "LEAD_TIME_DAYS" in globals()
assert "REVIEW_PERIOD_WEEKS" in globals()

# ── Frozen artifact paths ----------------------------------------------------------------

final_reorder_path = (
    f"{PREDICTIONS_DIR}/final_reorder_params_fold3.parquet"
)

service_sweep_path = (
    f"{PREDICTIONS_DIR}/service_level_sweep_fold3.parquet"
)

tsb_path = (
    f"{PREDICTIONS_DIR}/tsb_safety_stock_fold3.parquet"
)

conformal_path = (
    f"{CALIBRATION_DIR}/conformal_residuals_fold3.pkl"
)

lumpy_policy_path = (
    f"{PREDICTIONS_DIR}/lumpy_policy_params_fold3.parquet"
)

required_paths = [
    final_reorder_path,
    service_sweep_path,
    tsb_path,
    conformal_path,
    lumpy_policy_path,
]

for path in required_paths:

    assert os.path.exists(path), (
        f"Missing frozen Notebook 07 artifact: {path}"
    )

# ── Load frozen artifacts ----------------------------------------------------------------

final_reorder_params_fold3 = pd.read_parquet(
    final_reorder_path
)

service_level_sweep_fold3 = pd.read_parquet(
    service_sweep_path
)

tsb_safety_stock_fold3 = pd.read_parquet(
    tsb_path
)

lumpy_policy_params_fold3 = pd.read_parquet(
    lumpy_policy_path
)

with open(
    conformal_path,
    "rb"
) as f:
    conformal_fold3 = pickle.load(f)

# ── Exact frozen population contract -----------------------------------------------------

assert len(final_reorder_params_fold3) == 30_368
assert final_reorder_params_fold3["id"].is_unique

assert len(service_level_sweep_fold3) == 182_208
assert (
    service_level_sweep_fold3["id"].nunique()
    == 30_368
)

assert len(tsb_safety_stock_fold3) == 17_303
assert tsb_safety_stock_fold3["id"].is_unique

assert isinstance(
    conformal_fold3,
    dict
)

assert "sku_residuals" in conformal_fold3

print("── Frozen inventory population ──")

print(
    f"Frozen q80 reorder population: "
    f"{len(final_reorder_params_fold3):,}"
)

print(
    f"Frozen service-level sweep rows: "
    f"{len(service_level_sweep_fold3):,}"
)

print(
    f"Frozen sweep SKU population: "
    f"{service_level_sweep_fold3['id'].nunique():,}"
)

print()

# ── Verify frozen sweep structure --------------------------------------------------------

expected_sweep_labels = {
    "q50",
    "q75",
    "q80",
    "q90",
    "q95",
    "q99",
}

actual_sweep_labels = set(
    service_level_sweep_fold3[
        "service_level"
    ]
    .astype(str)
    .unique()
)

assert actual_sweep_labels == expected_sweep_labels, (
    f"Unexpected frozen service-level sweep labels: "
    f"{actual_sweep_labels}"
)

expected_rows_per_level = 30_368

sweep_counts = (
    service_level_sweep_fold3[
        "service_level"
    ]
    .value_counts()
    .to_dict()
)

for level in sorted(expected_sweep_labels):

    assert (
        sweep_counts[level]
        == expected_rows_per_level
    ), (
        f"{level} has unexpected frozen coverage: "
        f"{sweep_counts[level]:,}"
    )

# ── Supported application service levels -----------------------------------------------

APP_SERVICE_LEVELS = [
    0.80,
    0.90,
    0.95,
    0.99,
]

APP_SERVICE_LEVEL_LABELS = {
    0.80: "q80",
    0.90: "q90",
    0.95: "q95",
    0.99: "q99",
}

assert PRIMARY_SERVICE_LEVEL == 0.80

# ── Canonical SKU population -------------------------------------------------------------

frozen_policy_ids = set(
    final_reorder_params_fold3["id"]
)

metadata_ids = set(
    sku_metadata["id"]
)

missing_policy_ids = (
    metadata_ids
    - frozen_policy_ids
)

extra_policy_ids = (
    frozen_policy_ids
    - metadata_ids
)

assert len(missing_policy_ids) == 122
assert len(extra_policy_ids) == 0

print(
    f"Complete frozen SKU population: "
    f"{len(metadata_ids):,}"
)

print(
    f"SKUs with frozen inventory policy: "
    f"{len(frozen_policy_ids):,}"
)

print(
    f"SKUs without frozen inventory policy: "
    f"{len(missing_policy_ids):,}"
)

print()

# ── Frozen conformal residual distributions --------------------------------------------

sku_residuals_fold3 = {
    str(k): np.asarray(
        v,
        dtype=float
    )
    for k, v
    in conformal_fold3[
        "sku_residuals"
    ].items()
}

regime_lookup = (
    sku_metadata
    .set_index("id")[
        "regime"
    ]
    .to_dict()
)

policy_ids = sorted(
    frozen_policy_ids
)

# ── Build pooled residual distributions exactly as Notebook 07 --------------------------

pooled_residuals_by_regime = {}

for regime in [
    "smooth",
    "erratic",
]:

    regime_ids = [
        sku_id
        for sku_id in policy_ids
        if regime_lookup[sku_id] == regime
    ]

    pooled_arrays = [
        np.asarray(
            sku_residuals_fold3[sku_id],
            dtype=float
        )
        for sku_id in regime_ids
        if (
            sku_id in sku_residuals_fold3
            and len(
                sku_residuals_fold3[sku_id]
            ) > 0
        )
    ]

    assert pooled_arrays, (
        f"No frozen residuals found for regime {regime}."
    )

    pooled_residuals_by_regime[regime] = (
        np.concatenate(
            pooled_arrays
        )
    )

# ── Frozen service-level ingredients ----------------------------------------------------

# Smooth / Erratic q80 frozen population
se_q80 = final_reorder_params_fold3[
    final_reorder_params_fold3["regime"].isin(
        ["smooth", "erratic"]
    )
][
    [
        "id",
        "regime",
        "method",
        "expected_lead_time_demand",
        "reorder_point",
        "order_qty",
        "low_confidence",
        "fallback_required",
    ]
].copy()

expected_smooth_erratic_policy_count = (
    sku_metadata[
        "regime"
    ].isin(
        ["smooth", "erratic"]
    ).sum()
    - 122
)

assert len(se_q80) == (
    expected_smooth_erratic_policy_count
)

# Recover the native daily point forecast from the frozen q80
# expected lead-time demand.
#
# This is used only for q90/q95/q99 scenario reconstruction.
se_q80["native_daily_point_forecast"] = (
    se_q80[
        "expected_lead_time_demand"
    ]
    / LEAD_TIME_DAYS
)

# ── Intermittent frozen TSB quantities ---------------------------------------------------

required_tsb_quantiles = [
    "safety_stock_q80",
    "safety_stock_q90",
    "safety_stock_q95",
    "safety_stock_q99",
]

for column in required_tsb_quantiles:

    assert column in (
        tsb_safety_stock_fold3.columns
    ), (
        f"Missing frozen TSB column: {column}"
    )

tsb_indexed = (
    tsb_safety_stock_fold3
    .set_index("id")
)

# Every intermittent SKU in metadata should have a TSB row.
intermittent_ids = (
    sku_metadata.loc[
        sku_metadata["regime"] == "intermittent",
        "id",
    ]
    .tolist()
)

assert len(intermittent_ids) == 17_303

assert set(intermittent_ids) == set(
    tsb_safety_stock_fold3["id"]
)

# ── Lumpy frozen policy -----------------------------------------------------------------

lumpy_q80 = final_reorder_params_fold3[
    final_reorder_params_fold3["regime"] == "lumpy"
][
    [
        "id",
        "regime",
        "method",
        "expected_lead_time_demand",
        "reorder_point",
        "order_qty",
    ]
].copy()

assert len(lumpy_q80) == 4_170

# Notebook 07 reused the frozen lumpy reorder point at every
# service-level sweep level.
#
# Keep the exact q80 values for the primary configuration.
# The avg_weekly quantity is used only to make the app-facing
# expected-demand field interpretable for all service levels.

lumpy_expected = lumpy_policy_params_fold3[
    [
        "id",
        "avg_weekly",
    ]
].copy()

assert len(lumpy_expected) == 4_170
assert lumpy_expected["id"].is_unique

# ── Construct supported inventory-policy scenarios -------------------------------------

inventory_records = []

for service_level in APP_SERVICE_LEVELS:

    label = APP_SERVICE_LEVEL_LABELS[
        service_level
    ]

    # ==================================================================
    # Smooth / Erratic
    # ==================================================================

    for row in se_q80.itertuples(
        index=False
    ):

        sku_id = row.id
        regime = row.regime

        # --------------------------------------------------------------
        # PRIMARY q80
        #
        # Copy the frozen artifact values exactly.
        # No max(), recomputation, or rounding is applied.
        # --------------------------------------------------------------

        if service_level == PRIMARY_SERVICE_LEVEL:

            inventory_records.append(
                {
                    "id": sku_id,
                    "regime": regime,
                    "method": row.method,
                    "service_level": service_level,
                    "lead_time_days": LEAD_TIME_DAYS,
                    "review_period_weeks": REVIEW_PERIOD_WEEKS,
                    "expected_lead_time_demand": (
                        float(
                            row.expected_lead_time_demand
                        )
                    ),
                    "safety_buffer": (
                        float(
                            row.reorder_point
                            - row.expected_lead_time_demand
                        )
                    ),
                    "reorder_point": (
                        float(
                            row.reorder_point
                        )
                    ),
                    "order_qty": (
                        float(
                            row.order_qty
                        )
                    ),
                    "uncertainty_method": (
                        "Frozen Fold 3 conformal policy"
                    ),
                    "validated_configuration": True,
                }
            )

            continue

        # --------------------------------------------------------------
        # q90 / q95 / q99
        #
        # Deterministic reconstruction from the frozen Notebook 07
        # point forecast + frozen residual distributions.
        # --------------------------------------------------------------

        residuals = sku_residuals_fold3.get(
            sku_id,
            []
        )

        if len(residuals) < 5:

            residual_q = np.percentile(
                pooled_residuals_by_regime[
                    regime
                ],
                service_level * 100,
            )

            uncertainty_method = (
                "Pooled regime conformal fallback"
            )

        else:

            residual_q = np.percentile(
                residuals,
                service_level * 100,
            )

            uncertainty_method = (
                "Per-SKU conformal residuals"
            )

        residual_q = max(
            float(residual_q),
            0.0,
        )

        expected_lead_time_demand = (
            row.native_daily_point_forecast
            * LEAD_TIME_DAYS
        )

        safety_buffer = (
            residual_q
            * np.sqrt(
                LEAD_TIME_DAYS
            )
        )

        reorder_point = (
            expected_lead_time_demand
            + safety_buffer
        )

        # IMPORTANT:
        # q90/q95/q99 retain the exact frozen q80 order quantity.
        # Notebook 07 does not tune order quantity by service level.
        order_qty = float(
            row.order_qty
        )

        inventory_records.append(
            {
                "id": sku_id,
                "regime": regime,
                "method": row.method,
                "service_level": service_level,
                "lead_time_days": LEAD_TIME_DAYS,
                "review_period_weeks": REVIEW_PERIOD_WEEKS,
                "expected_lead_time_demand": (
                    expected_lead_time_demand
                ),
                "safety_buffer": safety_buffer,
                "reorder_point": reorder_point,
                "order_qty": order_qty,
                "uncertainty_method": (
                    uncertainty_method
                ),
                "validated_configuration": False,
            }
        )

    # ==================================================================
    # Intermittent
    # ==================================================================

    for sku_id in intermittent_ids:

        row = tsb_indexed.loc[
            sku_id
        ]

        safety_column = (
            f"safety_stock_{label}"
        )

        assert safety_column in row.index, (
            f"Missing frozen TSB safety-stock "
            f"column {safety_column}."
        )

        mean_lead_time_demand = float(
            row["mean_lead_time_demand"]
        )

        reorder_point = float(
            row[safety_column]
        )

        # q80 must preserve the exact frozen q80 order quantity
        # from final_reorder_params_fold3.
        if service_level == PRIMARY_SERVICE_LEVEL:

            frozen_row = final_reorder_params_fold3.loc[
                final_reorder_params_fold3["id"] == sku_id
            ]

            assert len(frozen_row) == 1

            frozen_row = frozen_row.iloc[0]

            expected_lead_time_demand = float(
                frozen_row[
                    "expected_lead_time_demand"
                ]
            )

            reorder_point = float(
                frozen_row[
                    "reorder_point"
                ]
            )

            order_qty = float(
                frozen_row[
                    "order_qty"
                ]
            )

        else:

            expected_lead_time_demand = (
                mean_lead_time_demand
            )

            # Order quantity is frozen across service levels.
            frozen_row = final_reorder_params_fold3.loc[
                final_reorder_params_fold3["id"] == sku_id
            ]

            assert len(frozen_row) == 1

            order_qty = float(
                frozen_row.iloc[0][
                    "order_qty"
                ]
            )

        safety_buffer = (
            reorder_point
            - expected_lead_time_demand
        )

        inventory_records.append(
            {
                "id": sku_id,
                "regime": "intermittent",
                "method": "block_bootstrap_tsb",
                "service_level": service_level,
                "lead_time_days": LEAD_TIME_DAYS,
                "review_period_weeks": REVIEW_PERIOD_WEEKS,
                "expected_lead_time_demand": (
                    expected_lead_time_demand
                ),
                "safety_buffer": safety_buffer,
                "reorder_point": reorder_point,
                "order_qty": order_qty,
                "uncertainty_method": (
                    "TSB + block-bootstrap"
                ),
                "validated_configuration": (
                    service_level
                    == PRIMARY_SERVICE_LEVEL
                ),
            }
        )

    # ==================================================================
    # Lumpy
    # ==================================================================

    for row in lumpy_q80.itertuples(
        index=False
    ):

        # Pull the frozen average weekly quantity only for the
        # human-readable expected-demand field.
        lumpy_row = lumpy_expected.loc[
            lumpy_expected["id"] == row.id
        ]

        assert len(lumpy_row) == 1

        avg_weekly = float(
            lumpy_row.iloc[0][
                "avg_weekly"
            ]
        )

        # q80: preserve exact frozen q80 fields.
        if service_level == PRIMARY_SERVICE_LEVEL:

            expected_lead_time_demand = (
                float(
                    row.expected_lead_time_demand
                )
                if pd.notna(
                    row.expected_lead_time_demand
                )
                else avg_weekly
            )

            reorder_point = float(
                row.reorder_point
            )

            order_qty = float(
                row.order_qty
            )

        else:

            # Notebook 07 kept the lumpy reorder point fixed
            # across the service-level sweep.
            expected_lead_time_demand = avg_weekly
            reorder_point = float(
                row.reorder_point
            )
            order_qty = float(
                row.order_qty
            )

        safety_buffer = (
            reorder_point
            - expected_lead_time_demand
        )

        inventory_records.append(
            {
                "id": row.id,
                "regime": "lumpy",
                "method": "historical_policy",
                "service_level": service_level,
                "lead_time_days": LEAD_TIME_DAYS,
                "review_period_weeks": REVIEW_PERIOD_WEEKS,
                "expected_lead_time_demand": (
                    expected_lead_time_demand
                ),
                "safety_buffer": safety_buffer,
                "reorder_point": reorder_point,
                "order_qty": order_qty,
                "uncertainty_method": (
                    "Historical min-max policy"
                ),
                "validated_configuration": (
                    service_level
                    == PRIMARY_SERVICE_LEVEL
                ),
            }
        )

# ── Convert to DataFrame ----------------------------------------------------------------

inventory_policy = pd.DataFrame(
    inventory_records
)

# ── Join SKU metadata -------------------------------------------------------------------

inventory_policy = inventory_policy.merge(
    sku_metadata[
        [
            "id",
            "item_id",
            "store_id",
            "dept_id",
            "cat_id",
            "state_id",
        ]
    ],
    on="id",
    how="left",
    validate="many_to_one",
)

# ── Population contract -----------------------------------------------------------------

expected_rows = (
    30_368
    * len(APP_SERVICE_LEVELS)
)

assert len(inventory_policy) == expected_rows

assert (
    inventory_policy["id"]
    .nunique()
    == 30_368
)

assert (
    inventory_policy[
        [
            "id",
            "service_level",
        ]
    ]
    .duplicated()
    .sum()
    == 0
)

# ── Regime coverage ----------------------------------------------------------------------

expected_policy_sku_counts = {
    "smooth": 8_003,
    "erratic": 892,
    "intermittent": 17_303,
    "lumpy": 4_170,
}

actual_policy_sku_counts = (
    inventory_policy[
        [
            "id",
            "regime",
        ]
    ]
    .drop_duplicates("id")
    ["regime"]
    .value_counts()
    .to_dict()
)

assert (
    actual_policy_sku_counts
    == expected_policy_sku_counts
), (
    f"Unexpected frozen inventory SKU populations: "
    f"{actual_policy_sku_counts}"
)

expected_policy_row_counts = {
    regime: count * len(APP_SERVICE_LEVELS)
    for regime, count
    in expected_policy_sku_counts.items()
}

actual_policy_row_counts = (
    inventory_policy[
        "regime"
    ]
    .value_counts()
    .to_dict()
)

assert (
    actual_policy_row_counts
    == expected_policy_row_counts
), (
    f"Unexpected inventory-policy row counts: "
    f"{actual_policy_row_counts}"
)

assert (
    len(inventory_policy)
    == 121_472
)

# ── Numeric validation -------------------------------------------------------------------

for column in [
    "service_level",
    "lead_time_days",
    "review_period_weeks",
    "expected_lead_time_demand",
    "safety_buffer",
    "reorder_point",
    "order_qty",
]:

    assert np.isfinite(
        inventory_policy[column]
    ).all(), (
        f"Non-finite values detected in {column}."
    )

assert (
    inventory_policy[
        "service_level"
    ].isin(
        APP_SERVICE_LEVELS
    )
).all()

assert (
    inventory_policy[
        "expected_lead_time_demand"
    ]
    >= 0
).all()

assert (
    inventory_policy[
        "reorder_point"
    ]
    >= 0
).all()

assert (
    inventory_policy[
        "order_qty"
    ]
    > 0
).all()

assert (
    inventory_policy[
        "lead_time_days"
    ]
    == 7
).all()

assert (
    inventory_policy[
        "review_period_weeks"
    ]
    == 1
).all()

# ── Primary q80 validation ---------------------------------------------------------------

q80_inventory = inventory_policy[
    inventory_policy[
        "service_level"
    ]
    == PRIMARY_SERVICE_LEVEL
].copy()

assert len(q80_inventory) == 30_368

q80_reference = final_reorder_params_fold3[
    [
        "id",
        "regime",
        "method",
        "expected_lead_time_demand",
        "reorder_point",
        "order_qty",
    ]
].copy()

assert q80_reference["id"].is_unique

q80_check = q80_inventory.merge(
    q80_reference,
    on="id",
    how="inner",
    suffixes=(
        "_app",
        "_reference",
    ),
    validate="one_to_one",
)

assert len(q80_check) == 30_368

# q80 production fields must match the frozen artifact exactly.

for app_col, reference_col in [
    (
        "expected_lead_time_demand_app",
        "expected_lead_time_demand_reference",
    ),
    (
        "reorder_point_app",
        "reorder_point_reference",
    ),
    (
        "order_qty_app",
        "order_qty_reference",
    ),
]:

    app_values = q80_check[
        app_col
    ].to_numpy(
        dtype=float
    )

    reference_values = q80_check[
        reference_col
    ].to_numpy(
        dtype=float
    )

    differences = np.abs(
        app_values - reference_values
    )

    max_diff = np.nanmax(
        differences
    )

    assert np.allclose(
        app_values,
        reference_values,
        rtol=0.0,
        atol=1e-8,
        equal_nan=True,
    ), (
        f"q80 mismatch in {app_col}: "
        f"max difference = {max_diff}"
    )

print(
    "✓ q80 application policy exactly reconciled "
    "to final_reorder_params_fold3.parquet."
)

# ── Additional exact q80 identity checks -------------------------------------------------

assert (
    q80_check[
        "regime_app"
    ]
    == q80_check[
        "regime_reference"
    ]
).all()

assert (
    q80_check[
        "method_app"
    ]
    == q80_check[
        "method_reference"
    ]
).all()

# ── Primary / alternatives ---------------------------------------------------------------

assert (
    inventory_policy[
        "validated_configuration"
    ].sum()
    == 30_368
)

print()
print("── Inventory-policy coverage ──")

display(
    pd.DataFrame(
        [
            {
                "Service Level": (
                    APP_SERVICE_LEVEL_LABELS[
                        level
                    ]
                ),
                "SKUs": int(
                    (
                        inventory_policy[
                            "service_level"
                        ]
                        == level
                    ).sum()
                ),
                "Validated Fold 3": (
                    "Yes"
                    if level == PRIMARY_SERVICE_LEVEL
                    else "No — scenario alternative"
                ),
            }
            for level in APP_SERVICE_LEVELS
        ]
    )
)

print()
print(
    f"Primary q80 rows: "
    f"{len(q80_inventory):,}"
)

print(
    "Primary configuration: "
    "7-day lead time / 1-week review / q80"
)

# ── Save --------------------------------------------------------------------------------

inventory_policy_output = (
    f"{APP_DIR}/app_inventory_policy.parquet"
)

inventory_policy.to_parquet(
    inventory_policy_output,
    index=False,
)

app_inventory_policy = inventory_policy

print()
print(
    f"Saved: {inventory_policy_output}"
)

print(
    f"Rows: {len(inventory_policy):,}"
)

print(
    f"SKUs: "
    f"{inventory_policy['id'].nunique():,}"
)

print(
    f"Columns: "
    f"{len(inventory_policy.columns):,}"
)

print()
print("=" * 88)
print("✓ SECTION 5 COMPLETE")
print("✓ Frozen inventory-policy scenarios packaged.")
print("✓ Primary 7-day / 1-week / q80 configuration preserved exactly.")
print("✓ q90 / q95 / q99 alternatives preserved.")
print("✓ 122 SKUs without frozen inventory policy remain excluded.")
print("✓ q80 reconciled exactly to the frozen Fold 3 reorder artifact.")
print("✓ No model training, recalibration, or inventory simulation performed.")
print("=" * 88)

SECTION 5: INVENTORY POLICY

── Frozen inventory population ──
Frozen q80 reorder population: 30,368
Frozen service-level sweep rows: 182,208
Frozen sweep SKU population: 30,368

Complete frozen SKU population: 30,490
SKUs with frozen inventory policy: 30,368
SKUs without frozen inventory policy: 122

✓ q80 application policy exactly reconciled to final_reorder_params_fold3.parquet.

── Inventory-policy coverage ──


,Service Level,SKUs,Validated Fold 3
0,q80,30368,Yes
1,q90,30368,No — scenario alternative
2,q95,30368,No — scenario alternative
3,q99,30368,No — scenario alternative



Primary q80 rows: 30,368
Primary configuration: 7-day lead time / 1-week review / q80

Saved: ../data/processed/predictions/app/app_inventory_policy.parquet
Rows: 121,472
SKUs: 30,368
Columns: 17

✓ SECTION 5 COMPLETE
✓ Frozen inventory-policy scenarios packaged.
✓ Primary 7-day / 1-week / q80 configuration preserved exactly.
✓ q90 / q95 / q99 alternatives preserved.
✓ 122 SKUs without frozen inventory policy remain excluded.
✓ q80 reconciled exactly to the frozen Fold 3 reorder artifact.
✓ No model training, recalibration, or inventory simulation performed.


## Section 5: Inventory Policy

Package the frozen Fold 3 inventory-policy configurations for the application.

The application exposes four service-level configurations:

* **q80** — primary production configuration
* **q90** — scenario alternative
* **q95** — scenario alternative
* **q99** — scenario alternative

The frozen Fold 3 inventory-policy population contains **30,368 SKUs**. This excludes 122 Smooth/Erratic SKUs that do not have a frozen Notebook 07 inventory-policy configuration because they lack monitor-set conformal residual history.

The primary q80 configuration is copied directly from `final_reorder_params_fold3.parquet` and reconciled exactly against that frozen artifact.

The q90/q95/q99 configurations are packaged as deterministic scenario alternatives using the frozen Notebook 07 inventory-policy ingredients and formulas. They are not presented as separately validated production configurations.

The underlying Notebook 07 service-level sweep remains available at q50, q75, q80, q90, q95, and q99; Notebook 09 exposes only q80/q90/q95/q99 in the application.

No model training, recalibration, or inventory simulation is performed in this section.

**Output:** `app_inventory_policy.parquet`

**Final coverage:** 30,368 SKUs × 4 application service levels = 121,472 rows.


## Section 6: Explainability

Package the frozen Notebook 08 explainability artifacts into a single application-ready table covering all 30,490 Fold 3 SKUs.

The application uses two explanation paths:

* **Smooth / Erratic:** five precomputed SHAP contributions from the frozen LightGBM Tweedie model.
* **Intermittent / Lumpy:** routing and policy explanations based on the frozen demand-regime assignment and forecasting method.

For Tweedie-routed SKUs, the application retains the five features with the largest absolute SHAP contributions from the latest available Fold 3 validation snapshot. SHAP values are treated as **predictive contributions**, not causal effects or price-elasticity estimates.

Intermittent and Lumpy SKUs do not receive SHAP explanations because those regimes are handled by TSB and the historical policy respectively. Instead, the application receives one routing/policy explanation record per SKU.

The Notebook 08 representative explainability artifact and final SHAP QA report are also checked before packaging. No SHAP values are recomputed and no model, feature pipeline, or routing decision is modified.

**Output:** `app_explainability.parquet`

**Coverage:**

* 9,017 Tweedie-routed SKUs
* 45,085 frozen SHAP explanation rows
* 17,303 Intermittent SKUs with routing explanations
* 4,170 Lumpy SKUs with routing explanations
* 66,558 total application explanation rows

This table is intended to power the application's SKU-level **"Why is this forecast changing?"** and **"Why was this SKU routed this way?"** views without performing expensive explainability calculations at runtime.


In [32]:
# -- 09 Section 6: Explainability ---------------------------------------------------------
# Package the frozen Notebook 08 explainability artifacts into one
# application-ready table covering all 30,490 Fold 3 SKUs.
#
# Production explanation paths:
#
#   Smooth / Erratic:
#       Frozen LightGBM Tweedie
#       -> top-5 precomputed SHAP contributions
#
#   Intermittent:
#       TSB
#       -> routing / policy explanation
#
#   Lumpy:
#       Historical policy
#       -> routing / policy explanation
#
# No SHAP recomputation, model inference, retraining, tuning,
# recalibration, or routing changes are performed here.

print("=" * 88)
print("SECTION 6: EXPLAINABILITY")
print("=" * 88)
print()

# ── Required state ----------------------------------------------------------------------

assert "APP_DIR" in globals()
assert "SHAP_DIR" in globals()
assert "sku_metadata" in globals()
assert "sku_regimes_fold3" in globals()

# ── Frozen Notebook 08 artifacts --------------------------------------------------------

sku_shap_path = (
    f"{SHAP_DIR}/shap_per_sku_fold3.parquet"
)

department_shap_path = (
    f"{SHAP_DIR}/shap_by_department_fold3.parquet"
)

representative_path = (
    f"{SHAP_DIR}/representative_explainability_fold3.parquet"
)

qa_report_path = (
    f"{SHAP_DIR}/shap_validation_report.json"
)

required_paths = [
    sku_shap_path,
    department_shap_path,
    representative_path,
    qa_report_path,
]

for path in required_paths:

    assert os.path.exists(path), (
        f"Missing frozen Notebook 08 artifact: {path}"
    )

# ── Load frozen Notebook 08 artifacts --------------------------------------------------

shap_per_sku_fold3 = pd.read_parquet(
    sku_shap_path
)

shap_by_department_fold3 = pd.read_parquet(
    department_shap_path
)

representative_explainability_fold3 = pd.read_parquet(
    representative_path
)

with open(
    qa_report_path,
    "r",
    encoding="utf-8"
) as f:

    shap_validation_report = json.load(
        f
    )

print("── Frozen explainability artifacts ──")

print(
    f"Per-SKU SHAP rows: "
    f"{len(shap_per_sku_fold3):,}"
)

print(
    f"Per-SKU SHAP SKUs: "
    f"{shap_per_sku_fold3['id'].nunique():,}"
)

print(
    f"Department SHAP rows: "
    f"{len(shap_by_department_fold3):,}"
)

print(
    f"Representative cases: "
    f"{len(representative_explainability_fold3):,}"
)

print(
    f"SHAP QA status: "
    f"{shap_validation_report.get('status')}"
)

print()

# ── Frozen Notebook 08 QA contract ------------------------------------------------------

assert shap_validation_report.get(
    "status"
) == "PASS"

assert (
    shap_validation_report["tweedie_skus"]
    == 9_017
)

assert (
    shap_validation_report["shap_skus"]
    == 9_017
)

assert (
    shap_validation_report["shap_rows"]
    == 45_085
)

assert (
    shap_validation_report["features_per_sku"]
    == 5
)

assert (
    shap_validation_report["frozen_feature_count"]
    == 39
)

# ── Validate frozen per-SKU SHAP structure ----------------------------------------------

required_shap_columns = {
    "id",
    "date",
    "regime",
    "feature",
    "feature_value",
    "feature_value_missing",
    "shap_value",
    "abs_shap",
    "direction",
    "rank",
}

missing_shap_columns = (
    required_shap_columns
    - set(
        shap_per_sku_fold3.columns
    )
)

assert not missing_shap_columns, (
    "Frozen SHAP artifact is missing required "
    f"columns: {sorted(missing_shap_columns)}"
)

assert len(
    shap_per_sku_fold3
) == 45_085

assert (
    shap_per_sku_fold3[
        "id"
    ].nunique()
    == 9_017
)

assert (
    shap_per_sku_fold3[
        "rank"
    ]
    .between(1, 5)
    .all()
)

assert np.isfinite(
    shap_per_sku_fold3[
        "shap_value"
    ]
).all()

assert np.isfinite(
    shap_per_sku_fold3[
        "abs_shap"
    ]
).all()

assert np.allclose(
    shap_per_sku_fold3[
        "abs_shap"
    ].to_numpy(),
    np.abs(
        shap_per_sku_fold3[
            "shap_value"
        ].to_numpy()
    ),
    rtol=1e-10,
    atol=1e-12,
)

assert (
    shap_per_sku_fold3[
        "direction"
    ]
    .isin(
        [
            "positive",
            "negative",
            "neutral",
        ]
    )
    .all()
)

# Every Tweedie SKU must have exactly five rows.
shap_rows_per_sku = (
    shap_per_sku_fold3
    .groupby(
        "id",
        observed=True
    )
    .size()
)

assert (
    shap_rows_per_sku
    .eq(5)
    .all()
)

# Every Tweedie SKU must contain ranks 1-5.
shap_rank_sets = (
    shap_per_sku_fold3
    .groupby(
        "id",
        observed=True
    )[
        "rank"
    ]
    .apply(set)
)

assert (
    shap_rank_sets
    .apply(
        lambda x: x == {1, 2, 3, 4, 5}
    )
    .all()
)

# Every Tweedie SKU must have five distinct explained features.
shap_feature_counts = (
    shap_per_sku_fold3
    .groupby(
        "id",
        observed=True
    )[
        "feature"
    ]
    .nunique()
)

assert (
    shap_feature_counts
    .eq(5)
    .all()
)

print(
    "✓ Frozen per-SKU SHAP artifact passed structural checks."
)

# ── Frozen regime population ------------------------------------------------------------

regime_lookup = (
    sku_metadata
    .set_index("id")[
        "regime"
    ]
    .to_dict()
)

all_sku_ids = set(
    sku_metadata["id"]
)

tweedie_ids = set(
    sku_metadata.loc[
        sku_metadata["regime"].isin(
            [
                "smooth",
                "erratic",
            ]
        ),
        "id",
    ]
)

intermittent_ids = set(
    sku_metadata.loc[
        sku_metadata["regime"] == "intermittent",
        "id",
    ]
)

lumpy_ids = set(
    sku_metadata.loc[
        sku_metadata["regime"] == "lumpy",
        "id",
    ]
)

assert len(all_sku_ids) == 30_490
assert len(tweedie_ids) == 9_017
assert len(intermittent_ids) == 17_303
assert len(lumpy_ids) == 4_170

assert (
    len(
        tweedie_ids
        | intermittent_ids
        | lumpy_ids
    )
    == 30_490
)

assert (
    len(
        tweedie_ids
        & intermittent_ids
    )
    == 0
)

assert (
    len(
        tweedie_ids
        & lumpy_ids
    )
    == 0
)

assert (
    len(
        intermittent_ids
        & lumpy_ids
    )
    == 0
)

print("── Frozen demand-regime population ──")

print(
    f"Tweedie-routed SKUs: "
    f"{len(tweedie_ids):,}"
)

print(
    f"Intermittent SKUs: "
    f"{len(intermittent_ids):,}"
)

print(
    f"Lumpy SKUs: "
    f"{len(lumpy_ids):,}"
)

print(
    f"Total SKUs: "
    f"{len(all_sku_ids):,}"
)

print()

# ── Verify SHAP coverage against frozen regime assignments -----------------------------

shap_ids = set(
    shap_per_sku_fold3[
        "id"
    ].unique()
)

assert shap_ids == tweedie_ids

artifact_regimes = (
    shap_per_sku_fold3[
        "id"
    ]
    .map(regime_lookup)
)

assert artifact_regimes.notna().all()

assert (
    artifact_regimes.to_numpy()
    ==
    shap_per_sku_fold3[
        "regime"
    ].to_numpy()
).all()

print(
    "✓ SHAP coverage exactly matches the "
    "9,017 frozen Tweedie-routed SKUs."
)

# ── Normalize frozen SHAP rows for application ------------------------------------------

shap_app = shap_per_sku_fold3[
    [
        "id",
        "date",
        "regime",
        "feature",
        "feature_value",
        "feature_value_missing",
        "shap_value",
        "abs_shap",
        "direction",
        "rank",
    ]
].copy()

shap_app[
    "explanation_type"
] = "shap"

shap_app[
    "forecast_method"
] = "LightGBM Tweedie"

shap_app[
    "explanation_text"
] = (
    "Predictive contribution from the frozen "
    "LightGBM Tweedie model."
)

# ── Build Intermittent routing explanations --------------------------------------------

intermittent_metadata = (
    sku_metadata[
        sku_metadata["regime"] == "intermittent"
    ][
        [
            "id",
            "data_as_of",
        ]
    ].copy()
)

assert len(
    intermittent_metadata
) == 17_303

intermittent_app = pd.DataFrame(
    {
        "id": intermittent_metadata[
            "id"
        ],
        "date": intermittent_metadata[
            "data_as_of"
        ],
        "regime": "intermittent",
        "feature": pd.Series(
            [None] * len(intermittent_metadata),
            dtype="object",
        ),
        "feature_value": np.nan,
        "feature_value_missing": False,
        "shap_value": np.nan,
        "abs_shap": np.nan,
        "direction": "neutral",
        "rank": 1,
        "explanation_type": "routing",
        "forecast_method": "TSB",
        "explanation_text": (
            "SKU was classified as intermittent demand "
            "and routed to the TSB forecasting policy "
            "rather than the LightGBM Tweedie model."
        ),
    }
)

# ── Build Lumpy routing explanations ----------------------------------------------------

lumpy_metadata = (
    sku_metadata[
        sku_metadata["regime"] == "lumpy"
    ][
        [
            "id",
            "data_as_of",
        ]
    ].copy()
)

assert len(
    lumpy_metadata
) == 4_170

lumpy_app = pd.DataFrame(
    {
        "id": lumpy_metadata[
            "id"
        ],
        "date": lumpy_metadata[
            "data_as_of"
        ],
        "regime": "lumpy",
        "feature": pd.Series(
            [None] * len(lumpy_metadata),
            dtype="object",
        ),
        "feature_value": np.nan,
        "feature_value_missing": False,
        "shap_value": np.nan,
        "abs_shap": np.nan,
        "direction": "neutral",
        "rank": 1,
        "explanation_type": "routing",
        "forecast_method": "Historical policy",
        "explanation_text": (
            "SKU was classified as lumpy demand "
            "and routed to the historical demand policy "
            "rather than the LightGBM Tweedie model."
        ),
    }
)

# ── Combine all application explanations -----------------------------------------------

explainability_columns = [
    "id",
    "date",
    "regime",
    "forecast_method",
    "explanation_type",
    "feature",
    "feature_value",
    "feature_value_missing",
    "shap_value",
    "abs_shap",
    "direction",
    "rank",
    "explanation_text",
]

app_explainability = pd.concat(
    [
        shap_app[
            explainability_columns
        ],
        intermittent_app[
            explainability_columns
        ],
        lumpy_app[
            explainability_columns
        ],
    ],
    ignore_index=True,
)

# ── Population validation ----------------------------------------------------------------

expected_shap_rows = (
    9_017
    * 5
)

expected_routing_rows = (
    17_303
    + 4_170
)

expected_total_rows = (
    expected_shap_rows
    + expected_routing_rows
)

assert expected_shap_rows == 45_085
assert expected_routing_rows == 21_473
assert expected_total_rows == 66_558

assert (
    len(app_explainability)
    == expected_total_rows
)

assert (
    app_explainability[
        "id"
    ].nunique()
    == 30_490
)

# Every SKU must have at least one explanation row.
sku_explanation_counts = (
    app_explainability
    .groupby(
        "id",
        observed=True
    )
    .size()
)

assert (
    sku_explanation_counts
    .index
    .isin(all_sku_ids)
    .all()
)

assert (
    len(sku_explanation_counts)
    == 30_490
)

# ── Explanation-type population ---------------------------------------------------------

shap_rows = app_explainability[
    app_explainability[
        "explanation_type"
    ] == "shap"
]

routing_rows = app_explainability[
    app_explainability[
        "explanation_type"
    ] == "routing"
]

assert len(shap_rows) == 45_085
assert len(routing_rows) == 21_473

assert (
    shap_rows[
        "regime"
    ]
    .isin(
        [
            "smooth",
            "erratic",
        ]
    )
    .all()
)

assert (
    routing_rows[
        "regime"
    ]
    .isin(
        [
            "intermittent",
            "lumpy",
        ]
    )
    .all()
)

# ── SHAP-specific application validation -----------------------------------------------

assert (
    shap_rows[
        "rank"
    ]
    .between(1, 5)
    .all()
)

assert np.isfinite(
    shap_rows[
        "shap_value"
    ]
).all()

assert np.isfinite(
    shap_rows[
        "abs_shap"
    ]
).all()

assert np.allclose(
    shap_rows[
        "abs_shap"
    ].to_numpy(),
    np.abs(
        shap_rows[
            "shap_value"
        ].to_numpy()
    ),
    rtol=1e-10,
    atol=1e-12,
)

assert (
    shap_rows[
        "direction"
    ]
    .isin(
        [
            "positive",
            "negative",
            "neutral",
        ]
    )
    .all()
)

# SHAP rows contain exactly five explanations per Tweedie SKU.
assert (
    shap_rows
    .groupby(
        "id",
        observed=True
    )
    .size()
    .eq(5)
    .all()
)

# No SHAP rows for Intermittent or Lumpy.
assert (
    set(
        shap_rows[
            "id"
        ]
    )
    ==
    tweedie_ids
)

# ── Routing-specific validation ---------------------------------------------------------

assert (
    routing_rows[
        "feature"
    ].isna()
).all()

assert (
    routing_rows[
        "shap_value"
    ].isna()
).all()

assert (
    routing_rows[
        "abs_shap"
    ].isna()
).all()

assert (
    routing_rows[
        "direction"
    ]
    == "neutral"
).all()

assert (
    routing_rows[
        "rank"
    ]
    == 1
).all()

assert (
    routing_rows[
        "id"
    ].nunique()
    == 21_473
)

assert (
    set(
        routing_rows[
            "id"
        ]
    )
    ==
    (
        intermittent_ids
        | lumpy_ids
    )
)

# ── Forecast-method validation ----------------------------------------------------------

assert (
    shap_rows[
        "forecast_method"
    ]
    == "LightGBM Tweedie"
).all()

assert (
    routing_rows.loc[
        routing_rows["regime"] == "intermittent",
        "forecast_method",
    ]
    == "TSB"
).all()

assert (
    routing_rows.loc[
        routing_rows["regime"] == "lumpy",
        "forecast_method",
    ]
    == "Historical policy"
).all()

# ── Date / historical snapshot validation -----------------------------------------------

assert (
    shap_rows[
        "date"
    ].notna()
).all()

assert (
    routing_rows[
        "date"
    ].notna()
).all()

# The application's historical snapshot is 2016-01-31.
assert (
    sku_metadata[
        "data_as_of"
    ].astype(str).eq(
        "2016-01-31"
    )
).all()

# ── Representative artifact validation --------------------------------------------------

expected_representative_regimes = {
    "Smooth",
    "Erratic",
    "Intermittent",
    "Lumpy",
}

assert set(
    representative_explainability_fold3[
        "Regime"
    ]
) == expected_representative_regimes

assert (
    representative_explainability_fold3[
        "Representative SKU"
    ]
    .nunique()
    == 4
)

print(
    "✓ Notebook 08 representative explainability "
    "artifact validated."
)

# ── Department SHAP artifact validation -------------------------------------------------

required_department_columns = {
    "department",
    "feature",
    "n_rows",
    "mean_abs_shap",
    "mean_signed_shap",
    "importance_share",
    "rank",
    "global_rank",
    "rank_delta_vs_global",
}

missing_department_columns = (
    required_department_columns
    - set(
        shap_by_department_fold3.columns
    )
)

assert not missing_department_columns, (
    "Department SHAP artifact is missing required "
    f"columns: {sorted(missing_department_columns)}"
)

assert (
    shap_by_department_fold3[
        "feature"
    ].notna()
).all()

assert np.isfinite(
    shap_by_department_fold3[
        [
            "mean_abs_shap",
            "mean_signed_shap",
            "importance_share",
        ]
    ].to_numpy()
).all()

print(
    "✓ Notebook 08 department-level SHAP artifact validated."
)

# ── Application table schema ------------------------------------------------------------

expected_app_columns = [
    "id",
    "date",
    "regime",
    "forecast_method",
    "explanation_type",
    "feature",
    "feature_value",
    "feature_value_missing",
    "shap_value",
    "abs_shap",
    "direction",
    "rank",
    "explanation_text",
]

assert (
    list(app_explainability.columns)
    == expected_app_columns
)

# ── Save --------------------------------------------------------------------------------

app_explainability_output = (
    f"{APP_DIR}/app_explainability.parquet"
)

app_explainability.to_parquet(
    app_explainability_output,
    index=False,
)

print()
print("── Application explainability coverage ──")

display(
    pd.DataFrame(
        [
            {
                "Explanation Type": "SHAP",
                "Regimes": "Smooth / Erratic",
                "SKUs": int(
                    shap_rows[
                        "id"
                    ].nunique()
                ),
                "Rows": int(
                    len(shap_rows)
                ),
                "Rows per SKU": "5",
            },
            {
                "Explanation Type": "Routing",
                "Regimes": "Intermittent / Lumpy",
                "SKUs": int(
                    routing_rows[
                        "id"
                    ].nunique()
                ),
                "Rows": int(
                    len(routing_rows)
                ),
                "Rows per SKU": "1",
            },
        ]
    )
)

print()
print(
    f"Saved: {app_explainability_output}"
)

print(
    f"Rows: {len(app_explainability):,}"
)

print(
    f"SKUs: "
    f"{app_explainability['id'].nunique():,}"
)

print(
    f"Columns: "
    f"{len(app_explainability.columns):,}"
)

print()
print("=" * 88)
print("✓ SECTION 6 COMPLETE")
print("✓ Frozen Notebook 08 explainability artifacts packaged.")
print("✓ 9,017 Tweedie SKUs have 5 precomputed SHAP explanations each.")
print("✓ 17,303 Intermittent SKUs have routing/policy explanations.")
print("✓ 4,170 Lumpy SKUs have routing/policy explanations.")
print("✓ All 30,490 SKUs have application explainability coverage.")
print("✓ No SHAP recomputation or model inference performed.")
print("✓ Frozen production model and routing logic unchanged.")
print("=" * 88)

SECTION 6: EXPLAINABILITY



AssertionError: 

In [44]:
# -- 09 Section 6: Explainability ---------------------------------------------------------

print("=" * 88)
print("SECTION 6: EXPLAINABILITY")
print("=" * 88)
print()

# ── Required state ----------------------------------------------------------------------

assert all(
    x in globals()
    for x in [
        "APP_DIR",
        "EXPLAINABILITY_DIR",
        "DATA_AS_OF",
        "sku_metadata",
        "sku_regimes_fold3",
    ]
)

# ── Load frozen Notebook 08 artifacts ---------------------------------------------------

paths = {
    "global": f"{EXPLAINABILITY_DIR}/shap_global_fold3.parquet",
    "regime": f"{EXPLAINABILITY_DIR}/shap_by_regime_fold3.parquet",
    "department": f"{EXPLAINABILITY_DIR}/shap_by_department_fold3.parquet",
    "sku": f"{EXPLAINABILITY_DIR}/shap_per_sku_fold3.parquet",
    "representative": (
        f"{EXPLAINABILITY_DIR}/representative_explainability_fold3.parquet"
    ),
    "qa": f"{EXPLAINABILITY_DIR}/shap_validation_report.json",
}

assert all(
    os.path.exists(p) for p in paths.values()
), "One or more frozen Notebook 08 artifacts are missing."

global_shap_fold3 = pd.read_parquet(paths["global"])
regime_shap_fold3 = pd.read_parquet(paths["regime"])
department_shap_fold3 = pd.read_parquet(paths["department"])
shap_per_sku_fold3 = pd.read_parquet(paths["sku"])
representative_explainability_fold3 = pd.read_parquet(paths["representative"])

with open(paths["qa"], "r", encoding="utf-8") as f:
    shap_validation_report = json.load(f)

print("── Frozen explainability artifacts ──")
print(f"Global SHAP rows: {len(global_shap_fold3):,}")
print(f"Regime SHAP rows: {len(regime_shap_fold3):,}")
print(f"Department SHAP rows: {len(department_shap_fold3):,}")
print(f"Per-SKU SHAP rows: {len(shap_per_sku_fold3):,}")
print(f"Per-SKU SHAP SKUs: {shap_per_sku_fold3['id'].nunique():,}")
print(f"Representative cases: {len(representative_explainability_fold3):,}")
print(f"SHAP QA status: {shap_validation_report.get('status')}")
print()

# ── Frozen SHAP QA ----------------------------------------------------------------------

assert shap_validation_report.get("status") == "PASS"
assert shap_validation_report["tweedie_skus"] == 9_017
assert shap_validation_report["shap_skus"] == 9_017
assert shap_validation_report["shap_rows"] == 45_085
assert shap_validation_report["features_per_sku"] == 5
assert shap_validation_report["frozen_feature_count"] == 39

required_shap_columns = {
    "id", "date", "regime", "feature", "feature_value",
    "feature_value_missing", "shap_value", "abs_shap",
    "direction", "rank",
}

assert required_shap_columns <= set(shap_per_sku_fold3.columns)
assert len(shap_per_sku_fold3) == 45_085
assert shap_per_sku_fold3["id"].nunique() == 9_017
assert shap_per_sku_fold3["rank"].between(1, 5).all()
assert np.isfinite(shap_per_sku_fold3["shap_value"]).all()
assert np.isfinite(shap_per_sku_fold3["abs_shap"]).all()

assert np.allclose(
    shap_per_sku_fold3["abs_shap"],
    np.abs(shap_per_sku_fold3["shap_value"]),
    rtol=1e-10,
    atol=1e-12,
)

assert shap_per_sku_fold3["direction"].isin(
    ["positive", "negative", "neutral"]
).all()

assert (
    shap_per_sku_fold3.groupby("id", observed=True)
    .size()
    .eq(5)
    .all()
)

assert (
    shap_per_sku_fold3.groupby("id", observed=True)["rank"]
    .apply(lambda x: set(x) == {1, 2, 3, 4, 5})
    .all()
)

assert (
    shap_per_sku_fold3.groupby("id", observed=True)["feature"]
    .nunique()
    .eq(5)
    .all()
)

shap_dates = (
    pd.to_datetime(
        shap_per_sku_fold3["date"],
        errors="raise",
    )
    .dt.strftime("%Y-%m-%d")
    .unique()
)

assert len(shap_dates) == 1
SHAP_AS_OF = shap_dates[0]

print("✓ Frozen per-SKU SHAP artifact passed structural checks.")
print(f"✓ Frozen SHAP snapshot date: {SHAP_AS_OF}")

# ── Frozen regime population ------------------------------------------------------------

regime_lookup = sku_metadata.set_index("id")["regime"]

all_sku_ids = set(sku_metadata["id"])

tweedie_ids = set(
    sku_metadata.loc[
        sku_metadata["regime"].isin(["smooth", "erratic"]),
        "id",
    ]
)

intermittent_ids = set(
    sku_metadata.loc[
        sku_metadata["regime"] == "intermittent",
        "id",
    ]
)

lumpy_ids = set(
    sku_metadata.loc[
        sku_metadata["regime"] == "lumpy",
        "id",
    ]
)

assert len(all_sku_ids) == 30_490
assert len(tweedie_ids) == 9_017
assert len(intermittent_ids) == 17_303
assert len(lumpy_ids) == 4_170
assert len(tweedie_ids | intermittent_ids | lumpy_ids) == 30_490
assert not (tweedie_ids & intermittent_ids)
assert not (tweedie_ids & lumpy_ids)
assert not (intermittent_ids & lumpy_ids)

print("── Frozen demand-regime population ──")
print(f"Tweedie-routed SKUs: {len(tweedie_ids):,}")
print(f"Intermittent SKUs: {len(intermittent_ids):,}")
print(f"Lumpy SKUs: {len(lumpy_ids):,}")
print(f"Total SKUs: {len(all_sku_ids):,}")
print()

# ── SHAP coverage -----------------------------------------------------------------------

assert set(shap_per_sku_fold3["id"]) == tweedie_ids

assert (
    shap_per_sku_fold3["id"].map(regime_lookup).to_numpy()
    == shap_per_sku_fold3["regime"].to_numpy()
).all()

print("✓ SHAP coverage exactly matches the 9,017 frozen Tweedie-routed SKUs.")

# ── Controlled feature descriptions -----------------------------------------------------

FEATURE_DESCRIPTIONS = {
    "lag_1": "Demand one day ago.",
    "lag_7": "Demand seven days ago.",
    "lag_14": "Demand fourteen days ago.",
    "rolling_mean_7": "Seven-day rolling average demand.",
    "rolling_mean_28": "Twenty-eight-day rolling average demand.",
    "rolling_std_7": "Seven-day rolling demand variability.",
    "day_of_week": "Day-of-week calendar effect.",
    "sell_price": "Historical selling price.",
    "price_vs_item_mean": (
        "Selling price relative to the item's historical average."
    ),
    "price_change_pct": "Recent percentage change in selling price.",
    "dept_rolling_7": "Seven-day rolling average demand for the department.",
    "snap_proximity": "Proximity to SNAP-related calendar effects.",
}

for feature in shap_per_sku_fold3["feature"].dropna().unique():
    FEATURE_DESCRIPTIONS.setdefault(
        feature,
        f"Frozen model feature: {feature.replace('_', ' ')}.",
    )

assert set(
    shap_per_sku_fold3["feature"].dropna()
) <= set(FEATURE_DESCRIPTIONS)

# ── Application SHAP rows ---------------------------------------------------------------

shap_app = shap_per_sku_fold3[
    [
        "id",
        "date",
        "regime",
        "feature",
        "feature_value",
        "feature_value_missing",
        "shap_value",
        "abs_shap",
        "direction",
        "rank",
    ]
].copy()

shap_app["date"] = pd.to_datetime(
    shap_app["date"],
    errors="raise",
)

shap_app["feature_description"] = (
    shap_app["feature"].map(FEATURE_DESCRIPTIONS)
)

assert shap_app["feature_description"].notna().all()

shap_app["forecast_method"] = "LightGBM Tweedie"
shap_app["explanation_type"] = "shap"
shap_app["explanation_text"] = (
    "Predictive contribution from the frozen LightGBM Tweedie model."
)

# ── Routing rows ------------------------------------------------------------------------

def build_routing_rows(regime, ids, method, text):
    meta = (
        sku_metadata.loc[
            sku_metadata["id"].isin(ids),
            ["id"],
        ]
        .sort_values("id")
        .reset_index(drop=True)
    )

    assert len(meta) == len(ids)

    n = len(meta)

    return pd.DataFrame({
        "id": meta["id"].to_numpy(),
        "date": np.repeat(pd.Timestamp(DATA_AS_OF), n),
        "regime": np.repeat(regime, n),
        "forecast_method": np.repeat(method, n),
        "explanation_type": np.repeat("routing", n),
        "feature": np.full(n, None, dtype=object),
        "feature_description": np.full(n, None, dtype=object),
        "feature_value": np.full(n, np.nan),
        "feature_value_missing": np.full(n, False),
        "shap_value": np.full(n, np.nan),
        "abs_shap": np.full(n, np.nan),
        "direction": np.repeat("neutral", n),
        "rank": np.ones(n, dtype=int),
        "explanation_text": np.repeat(text, n),
    })


intermittent_app = build_routing_rows(
    "intermittent",
    intermittent_ids,
    "TSB",
    (
        "SKU was classified as intermittent demand and routed to "
        "the TSB forecasting policy rather than the LightGBM "
        "Tweedie model."
    ),
)

lumpy_app = build_routing_rows(
    "lumpy",
    lumpy_ids,
    "Historical policy",
    (
        "SKU was classified as lumpy demand and routed to the "
        "historical demand policy rather than the LightGBM "
        "Tweedie model."
    ),
)

# ── Combine ------------------------------------------------------------------------------

app_columns = [
    "id",
    "date",
    "regime",
    "forecast_method",
    "explanation_type",
    "feature",
    "feature_description",
    "feature_value",
    "feature_value_missing",
    "shap_value",
    "abs_shap",
    "direction",
    "rank",
    "explanation_text",
]

app_explainability = pd.concat(
    [
        shap_app[app_columns],
        intermittent_app[app_columns],
        lumpy_app[app_columns],
    ],
    ignore_index=True,
)

app_explainability["date"] = pd.to_datetime(
    app_explainability["date"],
    errors="raise",
)

# ── Final population contract -----------------------------------------------------------

assert len(shap_app) == 45_085
assert len(intermittent_app) == 17_303
assert len(lumpy_app) == 4_170
assert len(app_explainability) == 66_558
assert app_explainability["id"].nunique() == 30_490

explanation_counts = (
    app_explainability.groupby("id", observed=True).size()
)

assert len(explanation_counts) == 30_490
assert set(explanation_counts.index) == all_sku_ids

shap_rows = app_explainability[
    app_explainability["explanation_type"] == "shap"
]

routing_rows = app_explainability[
    app_explainability["explanation_type"] == "routing"
]

assert len(shap_rows) == 45_085
assert len(routing_rows) == 21_473

assert set(shap_rows["regime"]) <= {"smooth", "erratic"}
assert set(routing_rows["regime"]) <= {"intermittent", "lumpy"}

assert (
    shap_rows.groupby("id", observed=True)
    .size()
    .eq(5)
    .all()
)

assert set(shap_rows["id"]) == tweedie_ids
assert set(routing_rows["id"]) == (
    intermittent_ids | lumpy_ids
)

assert shap_rows["feature_description"].notna().all()
assert routing_rows["feature_description"].isna().all()

assert routing_rows["feature"].isna().all()
assert routing_rows["feature_value"].isna().all()
assert routing_rows["shap_value"].isna().all()
assert routing_rows["abs_shap"].isna().all()
assert (routing_rows["direction"] == "neutral").all()
assert (routing_rows["rank"] == 1).all()

assert (
    shap_rows["forecast_method"] == "LightGBM Tweedie"
).all()

assert (
    routing_rows.loc[
        routing_rows["regime"] == "intermittent",
        "forecast_method",
    ]
    == "TSB"
).all()

assert (
    routing_rows.loc[
        routing_rows["regime"] == "lumpy",
        "forecast_method",
    ]
    == "Historical policy"
).all()

assert (
    shap_rows["date"]
    .dt.strftime("%Y-%m-%d")
    .eq(SHAP_AS_OF)
    .all()
)

assert (
    routing_rows["date"]
    .dt.strftime("%Y-%m-%d")
    .eq(
        pd.Timestamp(DATA_AS_OF).strftime("%Y-%m-%d")
    )
    .all()
)

# ── Supporting Notebook 08 checks -------------------------------------------------------

assert set(
    representative_explainability_fold3["Regime"]
) == {
    "Smooth",
    "Erratic",
    "Intermittent",
    "Lumpy",
}

assert (
    representative_explainability_fold3["Representative SKU"]
    .nunique()
    == 4
)

department_columns = {
    "department",
    "feature",
    "n_rows",
    "mean_abs_shap",
    "mean_signed_shap",
    "importance_share",
    "rank",
    "global_rank",
    "rank_delta_vs_global",
}

assert department_columns <= set(
    department_shap_fold3.columns
)

assert department_shap_fold3["feature"].notna().all()

assert np.isfinite(
    department_shap_fold3[
        [
            "mean_abs_shap",
            "mean_signed_shap",
            "importance_share",
        ]
    ].to_numpy()
).all()

assert "feature" in global_shap_fold3.columns
assert "feature" in regime_shap_fold3.columns
assert global_shap_fold3["feature"].notna().all()
assert regime_shap_fold3["feature"].notna().all()

# ── Final schema -------------------------------------------------------------------------

expected_app_columns = [
    "id",
    "date",
    "regime",
    "forecast_method",
    "explanation_type",
    "feature",
    "feature_description",
    "feature_value",
    "feature_value_missing",
    "shap_value",
    "abs_shap",
    "direction",
    "rank",
    "explanation_text",
]

assert list(app_explainability.columns) == expected_app_columns
assert len(app_explainability.columns) == 14
assert pd.api.types.is_datetime64_any_dtype(
    app_explainability["date"]
)

# ── Save --------------------------------------------------------------------------------

app_explainability_output = (
    f"{APP_DIR}/app_explainability.parquet"
)

app_explainability.to_parquet(
    app_explainability_output,
    index=False,
)

print()
print("── Application explainability coverage ──")

display(
    pd.DataFrame([
        {
            "Explanation Type": "SHAP",
            "Regimes": "Smooth / Erratic",
            "SKUs": shap_rows["id"].nunique(),
            "Rows": len(shap_rows),
            "Rows per SKU": 5,
        },
        {
            "Explanation Type": "Routing",
            "Regimes": "Intermittent / Lumpy",
            "SKUs": routing_rows["id"].nunique(),
            "Rows": len(routing_rows),
            "Rows per SKU": 1,
        },
    ])
)

print()
print(f"Frozen SHAP snapshot: {SHAP_AS_OF}")
print(f"Application snapshot: {DATA_AS_OF}")
print(f"Saved: {app_explainability_output}")
print(f"Rows: {len(app_explainability):,}")
print(f"SKUs: {app_explainability['id'].nunique():,}")
print(f"Columns: {len(app_explainability.columns):,}")
print()
print("=" * 88)
print("✓ SECTION 6 COMPLETE")
print("✓ Frozen Notebook 08 explainability artifacts packaged.")
print("✓ 9,017 Tweedie SKUs have 5 precomputed SHAP explanations each.")
print("✓ Controlled feature descriptions packaged with SHAP rows.")
print("✓ 17,303 Intermittent SKUs have routing/policy explanations.")
print("✓ 4,170 Lumpy SKUs have routing/policy explanations.")
print("✓ All 30,490 SKUs have application explainability coverage.")
print("✓ No SHAP recomputation or model inference performed.")
print("✓ Frozen production model and routing logic unchanged.")
print("=" * 88)

SECTION 6: EXPLAINABILITY

── Frozen explainability artifacts ──
Global SHAP rows: 39
Regime SHAP rows: 78
Department SHAP rows: 117
Per-SKU SHAP rows: 45,085
Per-SKU SHAP SKUs: 9,017
Representative cases: 4
SHAP QA status: PASS

✓ Frozen per-SKU SHAP artifact passed structural checks.
✓ Frozen SHAP snapshot date: 2016-01-31
── Frozen demand-regime population ──
Tweedie-routed SKUs: 9,017
Intermittent SKUs: 17,303
Lumpy SKUs: 4,170
Total SKUs: 30,490

✓ SHAP coverage exactly matches the 9,017 frozen Tweedie-routed SKUs.

── Application explainability coverage ──


,Explanation Type,Regimes,SKUs,Rows,Rows per SKU
0,SHAP,Smooth / Erratic,9017,45085,5
1,Routing,Intermittent / Lumpy,21473,21473,1



Frozen SHAP snapshot: 2016-01-31
Application snapshot: 2016-01-31
Saved: ../data/processed/predictions/app/app_explainability.parquet
Rows: 66,558
SKUs: 30,490
Columns: 14

✓ SECTION 6 COMPLETE
✓ Frozen Notebook 08 explainability artifacts packaged.
✓ 9,017 Tweedie SKUs have 5 precomputed SHAP explanations each.
✓ Controlled feature descriptions packaged with SHAP rows.
✓ 17,303 Intermittent SKUs have routing/policy explanations.
✓ 4,170 Lumpy SKUs have routing/policy explanations.
✓ All 30,490 SKUs have application explainability coverage.
✓ No SHAP recomputation or model inference performed.
✓ Frozen production model and routing logic unchanged.


## Section 6: Explainability

Package the frozen Notebook 08 explainability artifacts into a single application-ready table covering the full **30,490-SKU frozen Fold 3 population**.

The application uses two explanation paths:

* **Smooth / Erratic:** five precomputed SHAP contributions from the frozen LightGBM Tweedie model for each of the 9,017 Tweedie-routed SKUs.
* **Intermittent / Lumpy:** one routing/policy explanation per SKU based on the frozen demand-regime assignment and forecasting method.

The frozen Notebook 08 per-SKU SHAP artifact contains **45,085 rows across 9,017 Tweedie-routed SKUs**, with exactly five ranked features per SKU. The Notebook 08 SHAP QA report passed, and the frozen SHAP snapshot date is **2016-01-31**.

A controlled feature-description dictionary is packaged with the SHAP rows so the application can display human-readable descriptions without generating explanations dynamically at runtime. SHAP values represent **predictive contribution**, not causal effects or price elasticity.

Intermittent and Lumpy SKUs do not receive SHAP explanations because those regimes use TSB and the historical policy respectively. Instead, they receive explicit routing explanations. These records are application-generated explanatory metadata and are not presented as SHAP results.

No SHAP values were recomputed, and no model, feature pipeline, or routing decision was modified.

**Output:** `app_explainability.parquet`

**Final coverage:**

* 9,017 Tweedie-routed SKUs
* 45,085 frozen SHAP rows
* 17,303 Intermittent routing rows
* 4,170 Lumpy routing rows
* 21,473 routing rows total
* 66,558 total explanation rows
* 30,490 unique SKUs
* 14 application columns

The resulting artifact provides the Streamlit application with precomputed SKU-level explanations while keeping expensive explainability calculations offline.

## Section 7: Portfolio Risk

Precompute the full SKU-level monitoring table used by the application for portfolio-level risk and operational review.

The table combines the frozen SKU master, regime-specific forecast information, and frozen q80 inventory-policy outputs. Risk fields are deterministic and derived only from information supported by the frozen application artifacts.

The portfolio table should include:

* current/latest inventory input placeholder
* forecast
* lead-time demand
* reorder point
* safety stock
* days/weeks of supply
* demand regime
* routing method
* recent forecast error
* recent demand volatility
* stockout-risk flag
* deterministic risk category
* regime-change flag

The M5 dataset does not provide an observed live on-hand inventory position. Therefore, the application must clearly distinguish **unavailable observed inventory inputs** from derived forecast and inventory-policy quantities. No fabricated inventory balance or simulated stock position should be presented as observed retailer inventory.

The frozen inventory-policy population contains **30,368 SKUs**. The remaining **122 Smooth/Erratic SKUs** do not have a frozen Notebook 07 inventory-policy artifact and must remain explicitly identifiable rather than being assigned fabricated reorder-point or order-quantity values.

Risk categories must be deterministic and supported by available inputs. Inventory-dependent classifications such as projected stockout within lead time should only be populated when an appropriate inventory input exists. Otherwise, the application should indicate that the risk determination is unavailable rather than imply a measured operational risk.

The table should support filtering by:

* store
* department
* category
* state
* regime
* risk tier

No new forecasting, model inference, inventory simulation, or recalibration is performed in this section.

**Output:** `app_portfolio_risk.parquet`


In [46]:
# -- 09 Section 7: Portfolio Risk --------------------------------------------------------

print("=" * 88)
print("SECTION 7: PORTFOLIO RISK")
print("=" * 88)
print()

# ── Required state ----------------------------------------------------------------------

assert all(
    x in globals()
    for x in [
        "APP_DIR",
        "sku_metadata",
        "app_regime_forecasts",
        "app_inventory_policy",
        "sku_regimes_fold3",
    ]
)

# ── Base SKU population -----------------------------------------------------------------

risk = sku_metadata[
    [
        "id",
        "item_id",
        "store_id",
        "dept_id",
        "cat_id",
        "state_id",
        "regime",
        "routing_method",
        "mean_weekly_demand",
        "median_weekly_demand",
        "zero_demand_rate",
        "adi",
        "cv2",
        "latest_available_demand",
        "latest_weekly_demand",
        "data_as_of",
        "price",
    ]
].copy()

assert len(risk) == 30_490
assert risk["id"].is_unique

# ── Unified regime forecast --------------------------------------------------------------

forecast = app_regime_forecasts[
    [
        "id",
        "forecast_method",
        "forecast",
        "uncertainty_method",
        "service_level",
        "forecast_status",
    ]
].drop_duplicates("id")

assert len(forecast) == 30_490
assert forecast["id"].is_unique

risk = risk.merge(
    forecast,
    on="id",
    how="left",
    validate="one_to_one",
)

assert risk["forecast_method"].notna().all()
assert risk["forecast_status"].notna().all()

# ── Frozen q80 inventory policy ----------------------------------------------------------

q80 = (
    app_inventory_policy.loc[
        app_inventory_policy["service_level"] == PRIMARY_SERVICE_LEVEL,
        [
            "id",
            "expected_lead_time_demand",
            "safety_buffer",
            "reorder_point",
            "order_qty",
            "validated_configuration",
        ],
    ]
    .drop_duplicates("id")
)

assert len(q80) == 30_368
assert q80["id"].is_unique

risk = risk.merge(
    q80,
    on="id",
    how="left",
    validate="one_to_one",
)

risk["inventory_policy_available"] = (
    risk["reorder_point"].notna()
)

assert (
    risk["inventory_policy_available"].sum()
    == 30_368
)

# ── Recent forecast error from frozen Fold 3 predictions -------------------------------

prediction_path = (
    "../data/processed/predictions/final_predictions_fold3.parquet"
)

pred = pd.read_parquet(prediction_path)

assert {"id", "date", "yhat"} <= set(pred.columns)

actual_col = next(
    (
        c for c in ["y", "actual", "demand", "sales"]
        if c in pred.columns
    ),
    None,
)

if actual_col is not None:
    pred["date"] = pd.to_datetime(
        pred["date"],
        errors="raise",
    )

    recent = (
        pred.sort_values(["id", "date"])
        .groupby("id", observed=True)
        .tail(28)
        .copy()
    )

    recent["abs_error"] = (
        recent[actual_col] - recent["yhat"]
    ).abs()

    recent["bias_component"] = (
        recent["yhat"] - recent[actual_col]
    )

    error = (
        recent.groupby("id", observed=True)
        .agg(
            recent_wape=(
                "abs_error",
                "sum",
            ),
            recent_actual=(
                actual_col,
                "sum",
            ),
            recent_bias=(
                "bias_component",
                "mean",
            ),
        )
        .reset_index()
    )

    error["recent_wape"] = np.where(
        error["recent_actual"] > 0,
        error["recent_wape"] / error["recent_actual"],
        np.nan,
    )

    error = error[
        [
            "id",
            "recent_wape",
            "recent_bias",
        ]
    ]

else:
    error = pd.DataFrame(
        {
            "id": risk["id"],
            "recent_wape": np.nan,
            "recent_bias": np.nan,
        }
    )

risk = risk.merge(
    error,
    on="id",
    how="left",
    validate="one_to_one",
)

# ── Recent demand volatility ------------------------------------------------------------

risk["recent_demand_volatility"] = np.where(
    risk["mean_weekly_demand"] > 0,
    risk["cv2"],
    np.nan,
)

risk["demand_volatility_band"] = pd.cut(
    risk["cv2"],
    bins=[-np.inf, 0.49, 1.0, np.inf],
    labels=["low", "moderate", "high"],
)

# ── Forecast deterioration --------------------------------------------------------------

risk["forecast_to_history_ratio"] = np.where(
    risk["mean_weekly_demand"] > 0,
    risk["forecast"] / risk["mean_weekly_demand"],
    np.nan,
)

risk["forecast_deterioration_flag"] = np.select(
    [
        risk["forecast"].isna(),
        risk["forecast_to_history_ratio"] >= 1.50,
        risk["forecast_to_history_ratio"] <= 0.50,
    ],
    [
        "unavailable",
        "elevated_forecast",
        "reduced_forecast",
    ],
    default="none",
)

# ── Inventory / stockout fields ---------------------------------------------------------

# M5 does not provide observed current on-hand inventory.
risk["current_inventory"] = np.nan
risk["current_inventory_status"] = "not_observed_in_M5"

risk["days_of_supply"] = np.nan
risk["weeks_of_supply"] = np.nan

risk["stockout_risk_flag"] = "not_determinable"

# ── Deterministic portfolio risk category ----------------------------------------------

risk["risk_category"] = np.select(
    [
        risk["current_inventory"].notna()
        & (
            risk["current_inventory"]
            < risk["expected_lead_time_demand"]
        ),

        risk["current_inventory"].notna()
        & (
            risk["current_inventory"]
            < risk["reorder_point"]
        ),

        risk["forecast_deterioration_flag"].isin(
            [
                "elevated_forecast",
                "reduced_forecast",
            ]
        ),

        ~risk["inventory_policy_available"],
    ],
    [
        "Critical",
        "Watch",
        "Watch",
        "Not determinable",
    ],
    default="Normal",
)

# With no observed inventory, no SKU can legitimately receive
# an observed-stockout classification.
assert (
    risk.loc[
        risk["current_inventory"].isna(),
        "stockout_risk_flag",
    ]
    == "not_determinable"
).all()

# ── Regime-change flag ------------------------------------------------------------------

# No independent observed regime-change history is created here.
# The frozen Fold 3 regime assignment remains the authoritative state.
risk["regime_change_flag"] = "not_available"

# ── Policy-monitoring status ------------------------------------------------------------

risk["policy_monitor_status"] = np.select(
    [
        ~risk["inventory_policy_available"],
        risk["forecast_deterioration_flag"].isin(
            [
                "elevated_forecast",
                "reduced_forecast",
            ]
        ),
    ],
    [
        "no_frozen_inventory_policy",
        "forecast_monitoring_required",
    ],
    default="normal_policy_coverage",
)

# ── Population / numeric QA -------------------------------------------------------------

assert len(risk) == 30_490
assert risk["id"].is_unique

assert (
    risk["regime"].value_counts().to_dict()
    == {
        "intermittent": 17_303,
        "smooth": 8_111,
        "lumpy": 4_170,
        "erratic": 906,
    }
)

assert (
    risk["inventory_policy_available"].sum()
    == 30_368
)

assert (
    (~risk["inventory_policy_available"]).sum()
    == 122
)

for column in [
    "mean_weekly_demand",
    "median_weekly_demand",
    "zero_demand_rate",
    "adi",
    "cv2",
    "latest_available_demand",
    "latest_weekly_demand",
    "price",
    "forecast",
]:
    values = risk[column].to_numpy(dtype=float)
    assert np.isfinite(values).all(), (
        f"Non-finite values found in {column}."
    )

# ── Final schema -------------------------------------------------------------------------

risk_columns = [
    "id",
    "item_id",
    "store_id",
    "dept_id",
    "cat_id",
    "state_id",
    "regime",
    "routing_method",
    "mean_weekly_demand",
    "median_weekly_demand",
    "zero_demand_rate",
    "adi",
    "cv2",
    "latest_available_demand",
    "latest_weekly_demand",
    "price",
    "forecast_method",
    "forecast",
    "uncertainty_method",
    "service_level",
    "forecast_status",
    "expected_lead_time_demand",
    "safety_buffer",
    "reorder_point",
    "order_qty",
    "validated_configuration",
    "inventory_policy_available",
    "recent_wape",
    "recent_bias",
    "recent_demand_volatility",
    "demand_volatility_band",
    "forecast_to_history_ratio",
    "forecast_deterioration_flag",
    "current_inventory",
    "current_inventory_status",
    "days_of_supply",
    "weeks_of_supply",
    "stockout_risk_flag",
    "risk_category",
    "regime_change_flag",
    "policy_monitor_status",
]

risk = risk[risk_columns]

assert risk["id"].is_unique
assert list(risk.columns) == risk_columns

# ── Save --------------------------------------------------------------------------------

portfolio_risk_output = (
    f"{APP_DIR}/app_portfolio_risk.parquet"
)

risk.to_parquet(
    portfolio_risk_output,
    index=False,
)

app_portfolio_risk = risk

print()
print("── Portfolio risk coverage ──")

display(
    pd.DataFrame([
        {
            "Metric": "Frozen SKU population",
            "Value": len(risk),
        },
        {
            "Metric": "Frozen q80 policy SKUs",
            "Value": int(
                risk["inventory_policy_available"].sum()
            ),
        },
        {
            "Metric": "No frozen inventory policy",
            "Value": int(
                (~risk["inventory_policy_available"]).sum()
            ),
        },
        {
            "Metric": "Observed current inventory",
            "Value": "Unavailable in M5",
        },
        {
            "Metric": "Stockout risk",
            "Value": "Not determinable without observed inventory",
        },
    ])
)

print()
print(f"Saved: {portfolio_risk_output}")
print(f"Rows: {len(risk):,}")
print(f"SKUs: {risk['id'].nunique():,}")
print(f"Columns: {len(risk.columns):,}")
print()
print("=" * 88)
print("✓ SECTION 7 COMPLETE")
print("✓ Full 30,490-SKU portfolio monitoring table packaged.")
print("✓ Frozen q80 inventory-policy coverage preserved for 30,368 SKUs.")
print("✓ 122 SKUs without frozen inventory policy remain explicit.")
print("✓ Recent forecast-error diagnostics use frozen Fold 3 predictions when available.")
print("✓ Demand volatility is derived from frozen SKU statistics.")
print("✓ Current inventory was not fabricated.")
print("✓ Stockout risk is not claimed where observed inventory is unavailable.")
print("✓ No new forecasting, simulation, or model training performed.")
print("=" * 88)

SECTION 7: PORTFOLIO RISK


── Portfolio risk coverage ──


,Metric,Value
0,Frozen SKU population,30490
1,Frozen q80 policy SKUs,30368
2,No frozen inventory policy,122
3,Observed current inventory,Unavailable in M5
4,Stockout risk,Not determinable without observed inventory



Saved: ../data/processed/predictions/app/app_portfolio_risk.parquet
Rows: 30,490
SKUs: 30,490
Columns: 41

✓ SECTION 7 COMPLETE
✓ Full 30,490-SKU portfolio monitoring table packaged.
✓ Frozen q80 inventory-policy coverage preserved for 30,368 SKUs.
✓ 122 SKUs without frozen inventory policy remain explicit.
✓ Recent forecast-error diagnostics use frozen Fold 3 predictions when available.
✓ Demand volatility is derived from frozen SKU statistics.
✓ Current inventory was not fabricated.
✓ Stockout risk is not claimed where observed inventory is unavailable.
✓ No new forecasting, simulation, or model training performed.


## Section 7: Portfolio Risk

Package a full SKU-level portfolio monitoring table for the application using the frozen Fold 3 SKU master, regime-specific forecast outputs, and frozen q80 inventory-policy data.

The resulting table covers all **30,490 frozen Fold 3 SKUs** and preserves the distinction between the full SKU population and the **30,368 SKUs** with a frozen Notebook 07 inventory-policy configuration. The remaining **122 SKUs** without frozen inventory-policy artifacts remain explicitly identifiable.

The table includes forecast, lead-time demand, reorder-point, safety-buffer, demand-volatility, forecast-monitoring, inventory-policy coverage, and portfolio-risk fields.

Because the historical M5 dataset does not contain an observed current on-hand inventory position, current inventory, days of supply, weeks of supply, and inventory-dependent stockout risk are not represented as observed operational measurements. No inventory balance was fabricated, and stockout risk is therefore not claimed where the required inventory input is unavailable.

Recent forecast-error diagnostics use the frozen Fold 3 prediction artifact where the required actual-demand fields are available. Demand volatility is derived from the frozen SKU-level demand statistics.

The regime-change field remains explicitly unavailable because the frozen Fold 3 segmentation artifact provides the final regime assignment rather than a longitudinal regime history. No independent regime-reclassification system is introduced in Notebook 09.

**Output:** `app_portfolio_risk.parquet`

**Final coverage:**

* 30,490 rows
* 30,490 unique SKUs
* 30,368 SKUs with frozen q80 inventory-policy coverage
* 122 SKUs without frozen inventory-policy coverage
* 41 application columns

No new forecasting, model training, inventory simulation, or recalibration is performed.

## Section 8: Historical Model Performance

Package compact application-facing historical model and inventory performance tables from the existing frozen Notebook 07 and earlier evaluation artifacts.

No expensive model evaluation is recomputed in Notebook 09. The section reuses previously generated results so the Streamlit application can display historical performance without requiring notebooks or model execution at runtime.

### Forecasting performance

Include the previously evaluated forecasting approaches:

* Naive
* SARIMA
* Prophet
* XGBoost
* LightGBM / Tweedie

Expose the previously computed forecasting metrics, including:

* log-RMSE
* WAPE
* bias
* median
* p90

### Fold 3 inventory performance

Package the previously computed Fold 3 inventory results, including:

* median WAPE by demand regime
* p90 WAPE by demand regime
* regime distribution
* stockout rate
* weekly in-stock rate
* unit fill rate
* average inventory
* modeled annual cost

These are historical M5 evaluation results and must not be presented as current retailer performance or realized business outcomes.

The application should preserve the distinction between validated Fold 3 production results and scenario analyses. In particular, the validated production inventory configuration remains **7-day lead time / 1-week review / q80**.

**Outputs:**

* `app_model_results.parquet`
* `app_inventory_results.parquet`

No model training, prediction generation, or expensive evaluation is performed in this section.


In [61]:
# =============================================================================
# SECTION 8: HISTORICAL MODEL PERFORMANCE
# =============================================================================

from pathlib import Path
import pickle
import numpy as np
import pandas as pd

PREDICTIONS_DIR = Path("../data/processed/predictions")
CALIBRATION_DIR = Path("../data/processed/calibration")
APP_DIR = PREDICTIONS_DIR / "app"
APP_DIR.mkdir(parents=True, exist_ok=True)

print("═" * 100)
print("SECTION 8: HISTORICAL MODEL PERFORMANCE")
print("═" * 100)

# -----------------------------------------------------------------------------
# Load frozen artifacts
# -----------------------------------------------------------------------------

regime_summary = pd.read_parquet(
    PREDICTIONS_DIR / "dynamic_policy_regime_summary_fold3.parquet"
)

pooled = pd.read_parquet(
    PREDICTIONS_DIR / "dynamic_policy_pooled_fold3.parquet"
)

full_population = pd.read_parquet(
    PREDICTIONS_DIR / "dynamic_policy_full_population_fold3.parquet"
)

naive = pd.read_parquet(
    PREDICTIONS_DIR / "naive_baseline_fold3.parquet"
)

service_levels = pd.read_parquet(
    PREDICTIONS_DIR / "service_level_sweep_fold3.parquet"
)

simulation = pd.read_parquet(
    PREDICTIONS_DIR / "simulation_results_fold3.parquet"
)

cost_sensitivity = pd.read_parquet(
    PREDICTIONS_DIR / "dynamic_cost_sensitivity_fold3.parquet"
)

head_to_head = pd.read_parquet(
    PREDICTIONS_DIR / "static_vs_dynamic_headtohead_fold3.parquet"
)

stability = pd.read_parquet(
    PREDICTIONS_DIR / "fold3_stability_check.parquet"
)

with open(CALIBRATION_DIR / "winner_decision_fold2.pkl", "rb") as f:
    winner = pickle.load(f)

# -----------------------------------------------------------------------------
# Artifact diagnostics
# -----------------------------------------------------------------------------

print("Frozen performance artifacts")
print(f"Regime summary rows: {len(regime_summary):,}")
print(f"Pooled inventory rows: {len(pooled):,}")
print(f"Full-population rows: {len(full_population):,}")
print(f"Naive baseline rows: {len(naive):,}")
print(f"Service-level rows: {len(service_levels):,}")
print(f"Simulation rows: {len(simulation):,}")
print(f"Cost-sensitivity scenarios: {len(cost_sensitivity):,}")
print(f"Static-vs-dynamic scenarios: {len(head_to_head):,}")
print(f"Stability rows: {len(stability):,}")

# -----------------------------------------------------------------------------
# Frozen validation
# -----------------------------------------------------------------------------

assert len(regime_summary) == 4
assert set(regime_summary["regime"]) == {
    "smooth",
    "erratic",
    "intermittent",
    "lumpy",
}

assert len(pooled) == 4
assert len(full_population) == 29_671
assert len(naive) == 30_368
assert len(service_levels) == 182_208
assert len(simulation) == 30_368
assert len(cost_sensitivity) == 24
assert len(head_to_head) == 24
assert len(stability) == 12

assert service_levels["id"].nunique() == 30_368
assert simulation["id"].nunique() == 30_368

# Service-level sweep uses categorical q-labels.
expected_levels = {"q50", "q75", "q80", "q90", "q95", "q99"}

assert set(service_levels["service_level"].unique()) == expected_levels

for level in expected_levels:
    assert (
        service_levels.loc[
            service_levels["service_level"] == level,
            "id",
        ].nunique()
        == 30_368
    )

# Simulation artifact is q80 only and stores service level numerically.
assert set(simulation["service_level"].unique()) == {0.8}

# -----------------------------------------------------------------------------
# 8A. Historical forecasting-model performance
#
# Classic statistical models:
#   Notebook 02 / 03 aggregate held-out benchmark.
#
# XGBoost / Tweedie:
#   Frozen Fold 2 model-selection artifact.
#
# Evaluation scope stays explicit because these were not all evaluated on one
# identical population/metric basis.
# -----------------------------------------------------------------------------

model_rows = [
    {
        "result_type": "forecast_model",
        "model": "Naive",
        "variant": "Baseline",
        "selected": False,
        "evaluation_scope": "Aggregate held-out 15-month period",
        "evaluation_basis": "Notebook 02",
        "rmse": 380_051.0,
        "wape": np.nan,
        "mape": 8.96,
        "median_wape": np.nan,
        "p90_wape": np.nan,
        "bias": np.nan,
        "aggregate_ratio": np.nan,
        "median_ratio": np.nan,
        "pct_tight_ratio": np.nan,
        "notes": "Aggregate historical benchmark; MAPE as originally reported.",
    },
    {
        "result_type": "forecast_model",
        "model": "SARIMA",
        "variant": "Baseline",
        "selected": False,
        "evaluation_scope": "Aggregate held-out 15-month period",
        "evaluation_basis": "Notebook 02",
        "rmse": 277_220.0,
        "wape": np.nan,
        "mape": 6.91,
        "median_wape": np.nan,
        "p90_wape": np.nan,
        "bias": np.nan,
        "aggregate_ratio": np.nan,
        "median_ratio": np.nan,
        "pct_tight_ratio": np.nan,
        "notes": "Aggregate historical benchmark; MAPE as originally reported.",
    },
    {
        "result_type": "forecast_model",
        "model": "Prophet",
        "variant": "Baseline",
        "selected": False,
        "evaluation_scope": "Aggregate held-out 15-month period",
        "evaluation_basis": "Notebook 03",
        "rmse": 209_726.0,
        "wape": np.nan,
        "mape": 5.02,
        "median_wape": np.nan,
        "p90_wape": np.nan,
        "bias": np.nan,
        "aggregate_ratio": np.nan,
        "median_ratio": np.nan,
        "pct_tight_ratio": np.nan,
        "notes": "Aggregate historical benchmark; MAPE as originally reported.",
    },
]

# Exact frozen Fold 2 XGB/Tweedie variants.
for variant_name in [
    "XGB-A Raw",
    "XGB-B Suppressed",
    "XGB-C Calibrated",
    "TW-A Raw",
    "TW-B Suppressed",
    "TW-C Calibrated",
]:
    agg = winner["fold2_agg"][variant_name]
    wape_dist = winner["fold2_wape_dist"][variant_name]
    ratio_dist = winner["fold2_ratio_dist"][variant_name]

    model = (
        "XGBoost"
        if variant_name.startswith("XGB")
        else "LightGBM / Tweedie"
    )

    variant = variant_name.split(" ", 1)[1]

    model_rows.append(
        {
            "result_type": "model_variant",
            "model": model,
            "variant": variant,
            "selected": variant_name == "TW-A Raw",
            "evaluation_scope": "Fold 2 SKU-level model selection",
            "evaluation_basis": "winner_decision_fold2.pkl",
            "rmse": float(agg["rmse"]),
            "wape": float(agg["wape"]),
            "mape": float(agg["mape"]),
            "median_wape": float(wape_dist["median"]),
            "p90_wape": np.nan,
            "bias": float(agg["bias"]),
            "aggregate_ratio": float(agg["ratio"]),
            "median_ratio": float(ratio_dist["median"]),
            "pct_tight_ratio": float(ratio_dist["pct_tight"]),
            "notes": (
                "Frozen Fold 2 evaluation. "
                "P90 WAPE is not stored in winner_decision_fold2."
            ),
        }
    )

app_model_results = pd.DataFrame(model_rows)

# -----------------------------------------------------------------------------
# 8B. Fold 2 / Fold 3 stability
# -----------------------------------------------------------------------------

stability_rows = stability.rename(
    columns={
        "Metric": "metric",
        "Fold 2": "fold2",
        "Fold 3": "fold3",
        "Delta": "delta",
        "Tolerance": "tolerance",
        "Stable?": "stable",
    }
).copy()

stability_rows.insert(0, "result_type", "stability")
stability_rows["model"] = "Final production system"
stability_rows["variant"] = "Fold 3"
stability_rows["selected"] = False
stability_rows["evaluation_scope"] = "Fold 2 vs Fold 3"
stability_rows["evaluation_basis"] = "fold3_stability_check.parquet"
stability_rows["notes"] = (
    "Existing frozen stability check; not a separate model selection."
)

# Common metadata.
app_model_results["metric_period"] = "historical"
app_model_results["forecast_data_as_of"] = "2016-01-31"
stability_rows["metric_period"] = "historical"
stability_rows["forecast_data_as_of"] = "2016-01-31"

# -----------------------------------------------------------------------------
# 8C. Save model results
# -----------------------------------------------------------------------------

model_columns = [
    "result_type",
    "model",
    "variant",
    "selected",
    "evaluation_scope",
    "evaluation_basis",
    "rmse",
    "wape",
    "mape",
    "median_wape",
    "p90_wape",
    "bias",
    "aggregate_ratio",
    "median_ratio",
    "pct_tight_ratio",
    "metric",
    "fold2",
    "fold3",
    "delta",
    "tolerance",
    "stable",
    "notes",
    "metric_period",
    "forecast_data_as_of",
]

for col in model_columns:
    if col not in app_model_results.columns:
        app_model_results[col] = np.nan

for col in model_columns:
    if col not in stability_rows.columns:
        stability_rows[col] = np.nan

app_model_results = pd.concat(
    [
        app_model_results[model_columns],
        stability_rows[model_columns],
    ],
    ignore_index=True,
    sort=False,
)

app_model_results.to_parquet(
    APP_DIR / "app_model_results.parquet",
    index=False,
)

# -----------------------------------------------------------------------------
# 8D. Service-level inventory scenarios
#
# service_level_sweep_fold3:
#   q50, q75, q80, q90, q95, q99
#
# simulation_results_fold3:
#   q80 only, stored as numeric 0.8
# -----------------------------------------------------------------------------

service_results = (
    service_levels
    .groupby("service_level", as_index=False)
    .agg(
        n_skus=("id", "nunique"),
        stockout_rate=("stockout_rate", "mean"),
        avg_inventory=("avg_inventory", "mean"),
        fill_rate=("fill_rate", "mean"),
        units_short=("units_short", "sum"),
    )
)

service_results["weekly_in_stock_rate"] = (
    1.0 - service_results["stockout_rate"]
)

service_results["result_type"] = "service_level"
service_results["regime"] = "all"
service_results["service_level"] = (
    service_results["service_level"].astype(str)
)
service_results["is_primary"] = (
    service_results["service_level"] == "q80"
)
service_results["median_wape"] = np.nan
service_results["p90_wape"] = np.nan
service_results["naive_cost_yr"] = np.nan
service_results["static_cost_yr"] = np.nan
service_results["dynamic_cost_yr"] = np.nan
service_results["static_vs_naive_%"] = np.nan
service_results["dynamic_vs_static_%"] = np.nan
service_results["modeled_annual_cost"] = np.nan
service_results["notes"] = (
    "Frozen six-level inventory sweep across 30,368 policy-covered SKUs. "
    "Weekly in-stock = 1 - simulated stockout rate; "
    "fill rate is a separate unit-based metric."
)

# -----------------------------------------------------------------------------
# 8E. Fold 3 regime-level q80 inventory performance
# -----------------------------------------------------------------------------

q80_sim = simulation[
    simulation["service_level"] == 0.8
].copy()

assert len(q80_sim) == 30_368
assert q80_sim["id"].nunique() == 30_368

regime_inventory = (
    q80_sim
    .groupby("regime", as_index=False)
    .agg(
        n_skus=("id", "nunique"),
        stockout_rate=("stockout_rate", "mean"),
        avg_inventory=("avg_inventory", "mean"),
        fill_rate=("fill_rate", "mean"),
        units_short=("units_short", "sum"),
    )
)

regime_inventory["weekly_in_stock_rate"] = (
    1.0 - regime_inventory["stockout_rate"]
)

regime_inventory = regime_inventory.merge(
    regime_summary[
        [
            "regime",
            "median_wape",
            "p90_wape",
            "n_real_forecast",
            "win_rate_pct",
            "median_dyn_vs_static_pct",
            "mean_static_stockout_rate",
            "mean_dynamic_stockout_rate",
            "mean_static_avg_inventory",
            "mean_dynamic_avg_inventory",
        ]
    ],
    on="regime",
    how="left",
)

regime_inventory = regime_inventory.merge(
    pooled[
        [
            "regime",
            "naive_cost_yr",
            "static_cost_yr",
            "dynamic_cost_yr",
            "static_vs_naive_%",
            "dynamic_vs_static_%",
        ]
    ],
    on="regime",
    how="left",
)

regime_inventory["result_type"] = "regime_performance"
regime_inventory["service_level"] = "q80"
regime_inventory["is_primary"] = True
regime_inventory["modeled_annual_cost"] = (
    regime_inventory["dynamic_cost_yr"]
)
regime_inventory["notes"] = (
    "Fold 3 frozen q80 regime-level inventory evaluation. "
    "Lumpy WAPE is unavailable because n_real_forecast = 0."
)

# -----------------------------------------------------------------------------
# 8F. Pooled q80 modeled cost
#
# This pooled artifact covers the regimes represented in the frozen pooled
# policy result, not all 30,368 inventory-policy SKUs.
# -----------------------------------------------------------------------------

pooled_policy_skus = int(pooled["n_skus"].sum())
pooled_dynamic_cost = float(pooled["dynamic_cost_yr"].sum())

overall_cost = pd.DataFrame(
    [
        {
            "result_type": "overall_policy",
            "regime": "all",
            "service_level": "q80",
            "n_skus": pooled_policy_skus,
            "stockout_rate": float(q80_sim["stockout_rate"].mean()),
            "weekly_in_stock_rate": float(
                1.0 - q80_sim["stockout_rate"].mean()
            ),
            "avg_inventory": float(q80_sim["avg_inventory"].mean()),
            "fill_rate": float(q80_sim["fill_rate"].mean()),
            "units_short": float(q80_sim["units_short"].sum()),
            "median_wape": np.nan,
            "p90_wape": np.nan,
            "naive_cost_yr": float(pooled["naive_cost_yr"].sum()),
            "static_cost_yr": float(pooled["static_cost_yr"].sum()),
            "dynamic_cost_yr": pooled_dynamic_cost,
            "static_vs_naive_%": np.nan,
            "dynamic_vs_static_%": np.nan,
            "modeled_annual_cost": pooled_dynamic_cost,
            "is_primary": True,
            "notes": (
                f"Frozen pooled q80 modeled annual policy cost across "
                f"{pooled_policy_skus:,} SKUs represented in the pooled "
                f"regime-policy artifact; this is distinct from the "
                f"30,368-SKU inventory-policy coverage."
            ),
        }
    ]
)

# -----------------------------------------------------------------------------
# 8G. Final regime distribution
# -----------------------------------------------------------------------------

sku_metadata = pd.read_parquet(
    APP_DIR / "app_sku_metadata.parquet"
)

assert len(sku_metadata) == 30_490
assert sku_metadata["id"].nunique() == 30_490

regime_distribution = (
    sku_metadata["regime"]
    .value_counts()
    .rename_axis("regime")
    .reset_index(name="n_skus")
)

regime_distribution["result_type"] = "regime_distribution"
regime_distribution["service_level"] = "q80"
regime_distribution["stockout_rate"] = np.nan
regime_distribution["weekly_in_stock_rate"] = np.nan
regime_distribution["avg_inventory"] = np.nan
regime_distribution["fill_rate"] = np.nan
regime_distribution["units_short"] = np.nan
regime_distribution["median_wape"] = np.nan
regime_distribution["p90_wape"] = np.nan
regime_distribution["naive_cost_yr"] = np.nan
regime_distribution["static_cost_yr"] = np.nan
regime_distribution["dynamic_cost_yr"] = np.nan
regime_distribution["static_vs_naive_%"] = np.nan
regime_distribution["dynamic_vs_static_%"] = np.nan
regime_distribution["modeled_annual_cost"] = np.nan
regime_distribution["is_primary"] = True
regime_distribution["notes"] = (
    "Frozen final 30,490-SKU regime distribution."
)

regime_distribution["regime_share"] = (
    regime_distribution["n_skus"] / 30_490
)

# -----------------------------------------------------------------------------
# 8H. Coverage row
# -----------------------------------------------------------------------------

coverage = pd.DataFrame(
    [
        {
            "result_type": "coverage",
            "regime": "all",
            "service_level": "q80",
            "n_skus": 30_368,
            "stockout_rate": np.nan,
            "weekly_in_stock_rate": np.nan,
            "avg_inventory": np.nan,
            "fill_rate": np.nan,
            "units_short": np.nan,
            "median_wape": np.nan,
            "p90_wape": np.nan,
            "naive_cost_yr": np.nan,
            "static_cost_yr": np.nan,
            "dynamic_cost_yr": np.nan,
            "static_vs_naive_%": np.nan,
            "dynamic_vs_static_%": np.nan,
            "modeled_annual_cost": np.nan,
            "is_primary": True,
            "notes": (
                "Final population: 30,490 SKUs. "
                "Frozen inventory-policy coverage: 30,368 SKUs. "
                "122 SKUs lack a frozen inventory-policy artifact. "
                "154 Tweedie-routed SKUs lack a frozen daily trajectory "
                "and are not replaced."
            ),
        }
    ]
)

# -----------------------------------------------------------------------------
# 8I. Standardize and save inventory results
# -----------------------------------------------------------------------------

inventory_columns = [
    "result_type",
    "regime",
    "service_level",
    "n_skus",
    "stockout_rate",
    "weekly_in_stock_rate",
    "fill_rate",
    "avg_inventory",
    "units_short",
    "median_wape",
    "p90_wape",
    "naive_cost_yr",
    "static_cost_yr",
    "dynamic_cost_yr",
    "static_vs_naive_%",
    "dynamic_vs_static_%",
    "modeled_annual_cost",
    "is_primary",
    "notes",
]

inventory_parts = [
    service_results,
    regime_inventory,
    overall_cost,
    regime_distribution,
    coverage,
]

for df in inventory_parts:
    for col in inventory_columns:
        if col not in df.columns:
            df[col] = np.nan

app_inventory_results = pd.concat(
    [df[inventory_columns] for df in inventory_parts],
    ignore_index=True,
    sort=False,
)

app_inventory_results.to_parquet(
    APP_DIR / "app_inventory_results.parquet",
    index=False,
)

# -----------------------------------------------------------------------------
# Final QA
# -----------------------------------------------------------------------------

saved_model = pd.read_parquet(
    APP_DIR / "app_model_results.parquet"
)

saved_inventory = pd.read_parquet(
    APP_DIR / "app_inventory_results.parquet"
)

assert APP_DIR.joinpath("app_model_results.parquet").exists()
assert APP_DIR.joinpath("app_inventory_results.parquet").exists()

# Model QA.
forecast_rows = saved_model[
    saved_model["result_type"].isin(
        ["forecast_model", "model_variant"]
    )
]

assert set(forecast_rows["model"]) >= {
    "Naive",
    "SARIMA",
    "Prophet",
    "XGBoost",
    "LightGBM / Tweedie",
}

assert (
    (
        (saved_model["result_type"] == "model_variant")
        & (saved_model["selected"])
        & (saved_model["model"] == "LightGBM / Tweedie")
        & (saved_model["variant"] == "Raw")
    ).sum()
    == 1
)

assert (
    saved_model.loc[
        saved_model["result_type"] == "stability",
        "selected",
    ]
    == False
).all()

# Inventory QA.
saved_service = saved_inventory[
    saved_inventory["result_type"] == "service_level"
]

assert set(saved_service["service_level"]) == expected_levels
assert (saved_service["n_skus"] == 30_368).all()

saved_regime = saved_inventory[
    saved_inventory["result_type"] == "regime_performance"
]

assert len(saved_regime) == 4
assert set(saved_regime["regime"]) == {
    "smooth",
    "erratic",
    "intermittent",
    "lumpy",
}

assert len(
    saved_inventory[
        saved_inventory["result_type"] == "overall_policy"
    ]
) == 1

assert len(
    saved_inventory[
        saved_inventory["result_type"] == "regime_distribution"
    ]
) == 4

# -----------------------------------------------------------------------------
# Final output
# -----------------------------------------------------------------------------

print()
print("═" * 100)
print("SECTION 8 QA")
print("═" * 100)

print(f"Model-result rows: {len(saved_model):,}")
print(
    "  Forecast/model rows:",
    len(forecast_rows),
)
print(
    "  Stability rows:",
    (saved_model["result_type"] == "stability").sum(),
)

print(f"Inventory-result rows: {len(saved_inventory):,}")

print(
    "Service-level coverage:",
    saved_service[
        ["service_level", "n_skus"]
    ]
    .sort_values("service_level")
    .to_dict("records"),
)

print(
    "Regime distribution:",
    regime_distribution[
        ["regime", "n_skus"]
    ]
    .sort_values("regime")
    .to_dict("records"),
)

print(
    "Selected production model:",
    saved_model.loc[
        saved_model["selected"] == True,
        ["model", "variant"],
    ]
    .drop_duplicates()
    .to_dict("records"),
)

print(
    "Pooled q80 modeled annual cost:",
    f"${pooled_dynamic_cost:,.0f}",
)

print(
    "SKUs represented in pooled cost artifact:",
    f"{pooled_policy_skus:,}",
)

print("Final SKU population: 30,490")
print("Frozen inventory-policy coverage: 30,368")
print("Primary inventory service level: q80")
print("Scenario levels: q50, q75, q90, q95, q99")
print("Simulation artifact: q80 only, stored as numeric 0.8")
print("Weekly in-stock rate: 1 - simulated stockout rate")
print("Fill rate: separate unit-based metric")
print("Lumpy WAPE: unavailable because n_real_forecast = 0")
print("Cost-sensitivity scenarios: retained for Section 9")
print(
    "No model training, tuning, recalibration, replacement forecasting, "
    "or new simulation performed."
)
print("✓ Section 8 complete")

════════════════════════════════════════════════════════════════════════════════════════════════════
SECTION 8: HISTORICAL MODEL PERFORMANCE
════════════════════════════════════════════════════════════════════════════════════════════════════
Frozen performance artifacts
Regime summary rows: 4
Pooled inventory rows: 4
Full-population rows: 29,671
Naive baseline rows: 30,368
Service-level rows: 182,208
Simulation rows: 30,368
Cost-sensitivity scenarios: 24
Static-vs-dynamic scenarios: 24
Stability rows: 12

════════════════════════════════════════════════════════════════════════════════════════════════════
SECTION 8 QA
════════════════════════════════════════════════════════════════════════════════════════════════════
Model-result rows: 21
  Forecast/model rows: 9
  Stability rows: 12
Inventory-result rows: 16
Service-level coverage: [{'service_level': 'q50', 'n_skus': 30368}, {'service_level': 'q75', 'n_skus': 30368}, {'service_level': 'q80', 'n_skus': 30368}, {'service_level': 'q90', '

# Section 8 — Historical Model Performance

## Purpose

Package existing frozen forecasting and inventory-evaluation results into compact app-ready artifacts.

This section is historical evaluation only. It does not train models, tune hyperparameters, generate replacement forecasts, recalibrate predictions, or run new simulations.

---

## Forecasting Model Results

`app_model_results.parquet` packages historical model-selection evidence across the forecasting approaches used in the project.

Classic statistical benchmarks retain their original evaluation scope:

* Naive — aggregate held-out 15-month benchmark
* SARIMA — aggregate held-out 15-month benchmark
* Prophet — aggregate held-out 15-month benchmark

XGBoost and LightGBM/Tweedie retain the frozen Fold 2 SKU-level model-selection results, including aggregate WAPE/MAPE/RMSE/bias, median per-SKU WAPE, and demand-ratio diagnostics across the stored raw, suppressed, and calibrated variants.

The final selected forecasting variant is **LightGBM/Tweedie Raw**.

The historical results are not presented as a single apples-to-apples leaderboard because the statistical and ML models were evaluated at different scopes.

---

## Fold 2 / Fold 3 Stability

The existing `fold3_stability_check.parquet` results are packaged into the same model-results artifact.

These rows document the existing Fold 2 versus Fold 3 stability checks for the final production system.

No new stability computation is performed.

---

## Inventory Policy Performance

`app_inventory_results.parquet` packages frozen inventory simulation and policy-evaluation results.

The primary inventory configuration is:

* Service level: **q80**
* Lead time: **7 days**
* Review period: **1 week**

The frozen service-level sweep contains six policy scenarios:

`q50`, `q75`, `q80`, `q90`, `q95`, `q99`

Each scenario covers **30,368 SKUs**.

The q80 simulation artifact is stored separately with `service_level = 0.8` and contains **30,368 SKUs**. It supplies the q80 regime-level inventory metrics.

Weekly in-stock rate is represented as:

`1 − simulated stockout rate`

This is distinct from unit-based fill rate.

---

## Regime-Level Evaluation

Fold 3 regime results include:

* SKU count
* median WAPE
* p90 WAPE
* real-forecast coverage
* stockout rate
* weekly in-stock rate
* average inventory
* unit fill rate
* units short
* modeled policy cost
* static-versus-dynamic policy comparison metrics

Lumpy demand has no real frozen forecast coverage in Fold 3 (`n_real_forecast = 0`), so WAPE and p90 WAPE are left unavailable rather than treating its historical-policy estimate as a forecast.

---

## Regime Distribution

The final frozen population contains **30,490 SKUs**:

* Smooth: 8,111
* Intermittent: 17,303
* Lumpy: 4,170
* Erratic: 906

The regime distribution is packaged for application filtering and portfolio reporting.

---

## Cost Interpretation

The pooled q80 modeled annual cost is packaged from the frozen pooled policy artifact and represents the **29,555 SKUs contained in that pooled artifact**.

It is a modeled historical policy result, not realized retailer savings or observed current performance.

Detailed cost-sensitivity scenarios remain in their source artifact for **Section 9** rather than being duplicated here.

---

## QA

Section 8 verifies:

* final population = 30,490 SKUs
* inventory-policy coverage = 30,368 SKUs
* six service-level scenarios with 30,368 SKUs each
* q80 simulation coverage = 30,368 SKUs
* four final regimes
* selected production model = LightGBM/Tweedie Raw
* lumpy WAPE remains unavailable where no real forecast exists
* no new modeling or simulation is performed

### Outputs

* `app_model_results.parquet`
* `app_inventory_results.parquet`


# Section 9 — Cost Sensitivity & Policy Trade-offs

Package the frozen cost-sensitivity and static-versus-dynamic policy results for app use.

The analysis covers 24 historical scenarios across six stockout-cost multipliers and four carrying-cost rates.

The results are modeled historical scenarios, not realized savings or future financial forecasts.

Outputs:

* `app_cost_sensitivity.parquet`
* `app_policy_tradeoffs.parquet`

No new optimization, forecasting, or simulation is performed.


In [63]:
# =============================================================================
# SECTION 9: COST SENSITIVITY & POLICY TRADE-OFFS
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd

PREDICTIONS_DIR = Path("../data/processed/predictions")
APP_DIR = PREDICTIONS_DIR / "app"
APP_DIR.mkdir(parents=True, exist_ok=True)

print("═" * 100)
print("SECTION 9: COST SENSITIVITY & POLICY TRADE-OFFS")
print("═" * 100)

# -----------------------------------------------------------------------------
# Load frozen artifacts
# -----------------------------------------------------------------------------

cost_sensitivity = pd.read_parquet(
    PREDICTIONS_DIR / "dynamic_cost_sensitivity_fold3.parquet"
)

head_to_head = pd.read_parquet(
    PREDICTIONS_DIR / "static_vs_dynamic_headtohead_fold3.parquet"
)

print()
print("Frozen cost artifacts")
print(f"Cost-sensitivity scenarios: {len(cost_sensitivity):,}")
print(f"Static-vs-dynamic scenarios: {len(head_to_head):,}")

# -----------------------------------------------------------------------------
# Frozen validation
# -----------------------------------------------------------------------------

assert len(cost_sensitivity) == 24
assert len(head_to_head) == 24

expected_stockout_mult = {1.5, 2.0, 3.0, 4.0, 6.0, 8.0}
expected_carry_rate = {0.15, 0.20, 0.25, 0.30}

assert set(cost_sensitivity["stockout_mult"]) == expected_stockout_mult
assert set(head_to_head["stockout_mult"]) == expected_stockout_mult

assert set(cost_sensitivity["carry_rate"]) == expected_carry_rate
assert set(head_to_head["carry_rate"]) == expected_carry_rate

assert len(
    cost_sensitivity[["stockout_mult", "carry_rate"]].drop_duplicates()
) == 24

assert len(
    head_to_head[["stockout_mult", "carry_rate"]].drop_duplicates()
) == 24

assert set(head_to_head["static_optimal_level"]) == {"q99"}

# -----------------------------------------------------------------------------
# Scenario IDs
# -----------------------------------------------------------------------------

def add_scenario_id(df):
    out = df.copy()

    out["scenario_id"] = (
        "sm"
        + out["stockout_mult"].map(lambda x: f"{x:g}")
        + "_cr"
        + out["carry_rate"].map(lambda x: f"{x:.2f}")
    )

    out["scenario_description"] = (
        "Stockout cost "
        + out["stockout_mult"].map(lambda x: f"{x:g}x")
        + "; carrying cost "
        + (out["carry_rate"] * 100).map(lambda x: f"{x:.0f}%")
        + "; ratio "
        + out["ratio"].map(lambda x: f"{x:.1f}")
    )

    return out

# -----------------------------------------------------------------------------
# 9A. Cost-sensitivity artifact
# -----------------------------------------------------------------------------

app_cost_sensitivity = cost_sensitivity.rename(
    columns={
        "stockout_mult": "stockout_cost_multiplier",
        "carry_rate": "carrying_cost_rate",
        "naive_cost_$": "naive_cost_yr",
        "static_cost_$": "static_cost_yr",
        "dynamic_cost_$": "dynamic_cost_yr",
        "dynamic_vs_static_%": "dynamic_cost_reduction_vs_static_pct",
        "dynamic_vs_naive_%": "dynamic_cost_reduction_vs_naive_pct",
    }
).copy()

app_cost_sensitivity = add_scenario_id(
    app_cost_sensitivity.rename(
        columns={
            "stockout_cost_multiplier": "stockout_mult",
            "carrying_cost_rate": "carry_rate",
        }
    )
).rename(
    columns={
        "stockout_mult": "stockout_cost_multiplier",
        "carry_rate": "carrying_cost_rate",
    }
)

app_cost_sensitivity["result_type"] = "cost_sensitivity"

app_cost_sensitivity["dynamic_cost_pct_of_static"] = (
    app_cost_sensitivity["dynamic_cost_yr"]
    / app_cost_sensitivity["static_cost_yr"]
    * 100
)

app_cost_sensitivity["dynamic_cost_pct_of_naive"] = (
    app_cost_sensitivity["dynamic_cost_yr"]
    / app_cost_sensitivity["naive_cost_yr"]
    * 100
)

app_cost_sensitivity["static_cost_pct_of_naive"] = (
    app_cost_sensitivity["static_cost_yr"]
    / app_cost_sensitivity["naive_cost_yr"]
    * 100
)

app_cost_sensitivity = app_cost_sensitivity[
    [
        "result_type",
        "scenario_id",
        "scenario_description",
        "stockout_cost_multiplier",
        "carrying_cost_rate",
        "ratio",
        "naive_cost_yr",
        "static_cost_yr",
        "dynamic_cost_yr",
        "dynamic_cost_reduction_vs_static_pct",
        "dynamic_cost_reduction_vs_naive_pct",
        "dynamic_cost_pct_of_static",
        "dynamic_cost_pct_of_naive",
        "static_cost_pct_of_naive",
    ]
].sort_values(
    ["stockout_cost_multiplier", "carrying_cost_rate"]
).reset_index(drop=True)

# -----------------------------------------------------------------------------
# 9B. Static vs dynamic policy trade-offs
# -----------------------------------------------------------------------------

app_policy_tradeoffs = head_to_head.rename(
    columns={
        "stockout_mult": "stockout_cost_multiplier",
        "carry_rate": "carrying_cost_rate",
        "static_optimal_cost_$": "static_optimal_cost_yr",
        "naive_cost_$": "naive_cost_yr",
        "dynamic_cost_$": "dynamic_cost_yr",
        "dynamic_vs_static_optimal_%": (
            "dynamic_cost_reduction_vs_static_optimal_pct"
        ),
        "static_optimal_vs_naive_%": (
            "static_optimal_cost_reduction_vs_naive_pct"
        ),
    }
).copy()

app_policy_tradeoffs = add_scenario_id(
    app_policy_tradeoffs.rename(
        columns={
            "stockout_cost_multiplier": "stockout_mult",
            "carrying_cost_rate": "carry_rate",
        }
    )
).rename(
    columns={
        "stockout_mult": "stockout_cost_multiplier",
        "carry_rate": "carrying_cost_rate",
    }
)

app_policy_tradeoffs["result_type"] = "policy_tradeoff"

app_policy_tradeoffs["dynamic_cost_pct_of_static_optimal"] = (
    app_policy_tradeoffs["dynamic_cost_yr"]
    / app_policy_tradeoffs["static_optimal_cost_yr"]
    * 100
)

app_policy_tradeoffs["dynamic_cost_pct_of_naive"] = (
    app_policy_tradeoffs["dynamic_cost_yr"]
    / app_policy_tradeoffs["naive_cost_yr"]
    * 100
)

app_policy_tradeoffs["static_optimal_cost_pct_of_naive"] = (
    app_policy_tradeoffs["static_optimal_cost_yr"]
    / app_policy_tradeoffs["naive_cost_yr"]
    * 100
)

app_policy_tradeoffs = app_policy_tradeoffs[
    [
        "result_type",
        "scenario_id",
        "scenario_description",
        "stockout_cost_multiplier",
        "carrying_cost_rate",
        "ratio",
        "static_optimal_level",
        "naive_cost_yr",
        "static_optimal_cost_yr",
        "dynamic_cost_yr",
        "dynamic_cost_reduction_vs_static_optimal_pct",
        "static_optimal_cost_reduction_vs_naive_pct",
        "dynamic_cost_pct_of_static_optimal",
        "dynamic_cost_pct_of_naive",
        "static_optimal_cost_pct_of_naive",
    ]
].sort_values(
    ["stockout_cost_multiplier", "carrying_cost_rate"]
).reset_index(drop=True)

# -----------------------------------------------------------------------------
# 9C. Save app artifacts
# -----------------------------------------------------------------------------

app_cost_sensitivity.to_parquet(
    APP_DIR / "app_cost_sensitivity.parquet",
    index=False,
)

app_policy_tradeoffs.to_parquet(
    APP_DIR / "app_policy_tradeoffs.parquet",
    index=False,
)

# -----------------------------------------------------------------------------
# Final QA
# -----------------------------------------------------------------------------

saved_sensitivity = pd.read_parquet(
    APP_DIR / "app_cost_sensitivity.parquet"
)

saved_tradeoffs = pd.read_parquet(
    APP_DIR / "app_policy_tradeoffs.parquet"
)

assert APP_DIR.joinpath("app_cost_sensitivity.parquet").exists()
assert APP_DIR.joinpath("app_policy_tradeoffs.parquet").exists()

# Exact scenario coverage.
assert len(saved_sensitivity) == 24
assert len(saved_tradeoffs) == 24

assert saved_sensitivity["scenario_id"].nunique() == 24
assert saved_tradeoffs["scenario_id"].nunique() == 24

# Parameter coverage.
assert set(
    saved_sensitivity["stockout_cost_multiplier"]
) == expected_stockout_mult

assert set(
    saved_tradeoffs["stockout_cost_multiplier"]
) == expected_stockout_mult

assert set(
    saved_sensitivity["carrying_cost_rate"]
) == expected_carry_rate

assert set(
    saved_tradeoffs["carrying_cost_rate"]
) == expected_carry_rate

# Static optimum preserved exactly from frozen artifact.
assert set(
    saved_tradeoffs["static_optimal_level"]
) == {"q99"}

# No invalid modeled costs.
for df in [saved_sensitivity, saved_tradeoffs]:
    cost_cols = [
        col for col in df.columns
        if col.endswith("_cost_yr")
    ]

    for col in cost_cols:
        assert df[col].notna().all()
        assert (df[col] >= 0).all()

# -----------------------------------------------------------------------------
# Final output summary
# -----------------------------------------------------------------------------

print()
print("═" * 100)
print("SECTION 9 QA")
print("═" * 100)

print(
    "Cost-sensitivity rows:",
    f"{len(saved_sensitivity):,}",
)

print(
    "Policy-tradeoff rows:",
    f"{len(saved_tradeoffs):,}",
)

print(
    "Stockout-cost multipliers:",
    sorted(saved_sensitivity["stockout_cost_multiplier"].unique()),
)

print(
    "Carrying-cost rates:",
    sorted(saved_sensitivity["carrying_cost_rate"].unique()),
)

print(
    "Static optimal levels:",
    sorted(saved_tradeoffs["static_optimal_level"].unique()),
)

print(
    "Dynamic modeled cost range:",
    f"${saved_sensitivity['dynamic_cost_yr'].min():,.0f}"
    f" – ${saved_sensitivity['dynamic_cost_yr'].max():,.0f}",
)

print(
    "Static modeled cost range:",
    f"${saved_sensitivity['static_cost_yr'].min():,.0f}"
    f" – ${saved_sensitivity['static_cost_yr'].max():,.0f}",
)

print(
    "Naive modeled cost range:",
    f"${saved_sensitivity['naive_cost_yr'].min():,.0f}"
    f" – ${saved_sensitivity['naive_cost_yr'].max():,.0f}",
)

print(
    "Scenario grid: 6 stockout-cost multipliers × "
    "4 carrying-cost rates = 24 scenarios"
)

print(
    "Static optimal level in frozen head-to-head artifact: q99 "
    "for all 24 tested scenarios"
)

print(
    "Modeled historical scenario analysis only; "
    "not realized savings or future financial performance."
)

print(
    "No new optimization, forecasting, inventory simulation, "
    "or policy search performed."
)

print("✓ Section 9 complete")

════════════════════════════════════════════════════════════════════════════════════════════════════
SECTION 9: COST SENSITIVITY & POLICY TRADE-OFFS
════════════════════════════════════════════════════════════════════════════════════════════════════

Frozen cost artifacts
Cost-sensitivity scenarios: 24
Static-vs-dynamic scenarios: 24

════════════════════════════════════════════════════════════════════════════════════════════════════
SECTION 9 QA
════════════════════════════════════════════════════════════════════════════════════════════════════
Cost-sensitivity rows: 24
Policy-tradeoff rows: 24
Stockout-cost multipliers: [np.float64(1.5), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(6.0), np.float64(8.0)]
Carrying-cost rates: [np.float64(0.15), np.float64(0.2), np.float64(0.25), np.float64(0.3)]
Static optimal levels: ['q99']
Dynamic modeled cost range: $1,403,290 – $6,303,232
Static modeled cost range: $15,239,337 – $80,717,242
Naive modeled cost range: $24,090,816 –

# Section 9 — Completion

Section 9 is complete.

The application now contains the frozen 24-scenario cost-sensitivity analysis and the frozen 24-scenario static-versus-dynamic policy comparison. These results are historical modeled scenarios and are not presented as realized savings or future financial forecasts.

No new forecasting, optimization, inventory simulation, or policy search was performed.


# Section 10 — Final App Package & Release QA

Final integrity gate for the historical M5 application dataset.

This section verifies that all app-ready artifacts exist, expected SKU populations reconcile, required scenario coverage is present, and the selected production model and primary inventory policy are preserved.

No modeling, forecasting, optimization, or simulation is performed.

After all checks pass, the app manifest is marked finalized.


In [69]:
# =============================================================================
# SECTION 10: FINAL APP PACKAGE & RELEASE QA
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

APP_DIR = Path("../data/processed/predictions/app")

print("═" * 100)
print("SECTION 10: FINAL APP PACKAGE & RELEASE QA")
print("═" * 100)

# -----------------------------------------------------------------------------
# Required app artifacts
# -----------------------------------------------------------------------------

required_files = {
    "manifest": APP_DIR / "app_manifest.json",
    "sku_metadata": APP_DIR / "app_sku_metadata.parquet",
    "forecasts": APP_DIR / "app_forecasts.parquet",
    "regime_forecasts": APP_DIR / "app_regime_forecasts.parquet",
    "inventory_policy": APP_DIR / "app_inventory_policy.parquet",
    "explainability": APP_DIR / "app_explainability.parquet",
    "model_results": APP_DIR / "app_model_results.parquet",
    "inventory_results": APP_DIR / "app_inventory_results.parquet",
    "cost_sensitivity": APP_DIR / "app_cost_sensitivity.parquet",
    "policy_tradeoffs": APP_DIR / "app_policy_tradeoffs.parquet",
}

# Discover the actual Section 7 risk artifact rather than assuming a filename.
risk_candidates = sorted(
    p for p in APP_DIR.glob("*risk*.parquet")
    if p.is_file()
)

assert len(risk_candidates) == 1, (
    f"Expected exactly one risk parquet artifact; "
    f"found {len(risk_candidates)}: {risk_candidates}"
)

required_files["risk"] = risk_candidates[0]

print()
print("Artifact existence")

for name, path in required_files.items():
    assert path.exists(), f"Missing app artifact: {path}"
    print(f"✓ {name}: {path.name}")

# -----------------------------------------------------------------------------
# Load artifacts
# -----------------------------------------------------------------------------

with open(required_files["manifest"], "r", encoding="utf-8") as f:
    manifest = json.load(f)

sku_metadata = pd.read_parquet(required_files["sku_metadata"])
forecasts = pd.read_parquet(required_files["forecasts"])
regime_forecasts = pd.read_parquet(required_files["regime_forecasts"])
inventory_policy = pd.read_parquet(required_files["inventory_policy"])
explainability = pd.read_parquet(required_files["explainability"])
risk = pd.read_parquet(required_files["risk"])
model_results = pd.read_parquet(required_files["model_results"])
inventory_results = pd.read_parquet(required_files["inventory_results"])
cost_sensitivity = pd.read_parquet(required_files["cost_sensitivity"])
policy_tradeoffs = pd.read_parquet(required_files["policy_tradeoffs"])

# -----------------------------------------------------------------------------
# Manifest
# -----------------------------------------------------------------------------

assert isinstance(manifest, dict)
assert "finalized" in manifest

print()
print("Manifest")
print(f"Current finalized flag: {manifest['finalized']}")

# -----------------------------------------------------------------------------
# Core population reconciliation
# -----------------------------------------------------------------------------

assert len(sku_metadata) == 30_490
assert sku_metadata["id"].nunique() == 30_490

assert len(regime_forecasts) == 30_490
assert regime_forecasts["id"].nunique() == 30_490

assert len(explainability) == 66_558
assert explainability["id"].nunique() == 30_490

assert len(risk) == 30_490
assert risk["id"].nunique() == 30_490

print()
print("Core population reconciliation")
print("✓ SKU master: 30,490")
print("✓ Regime forecasts: 30,490")
print("✓ Explainability: 30,490")
print("✓ Risk artifact: 30,490")

# -----------------------------------------------------------------------------
# Forecast checks
# -----------------------------------------------------------------------------

assert len(forecasts) == 3_234_995
assert forecasts["id"].nunique() == 8_863

print()
print("Forecast coverage")
print("✓ Daily forecast rows: 3,234,995")
print("✓ Forecast SKUs: 8,863")
print("✓ Frozen missing trajectories preserved")

# -----------------------------------------------------------------------------
# Inventory-policy checks
#
# app_inventory_policy stores numeric service levels:
#   0.80 = validated primary configuration
#   0.90 / 0.95 / 0.99 = scenario alternatives
# -----------------------------------------------------------------------------

assert len(inventory_policy) == 121_472
assert inventory_policy["id"].nunique() == 30_368

expected_policy_levels = {0.80, 0.90, 0.95, 0.99}

assert set(
    inventory_policy["service_level"].unique()
) == expected_policy_levels

for level in expected_policy_levels:
    assert (
        inventory_policy.loc[
            inventory_policy["service_level"] == level,
            "id",
        ].nunique()
        == 30_368
    )

assert inventory_policy["lead_time_days"].nunique() == 1
assert int(inventory_policy["lead_time_days"].iloc[0]) == 7

assert inventory_policy["review_period_weeks"].nunique() == 1
assert int(inventory_policy["review_period_weeks"].iloc[0]) == 1

# Validated configuration is true only for q80.
assert inventory_policy["validated_configuration"].dtype == bool

validated = inventory_policy[
    inventory_policy["validated_configuration"]
]

scenario_rows = inventory_policy[
    ~inventory_policy["validated_configuration"]
]

assert len(validated) == 30_368
assert validated["id"].nunique() == 30_368
assert set(validated["service_level"].unique()) == {0.80}

assert len(scenario_rows) == 91_104
assert set(scenario_rows["service_level"].unique()) == {
    0.90,
    0.95,
    0.99,
}

assert not validated.empty
assert validated["lead_time_days"].eq(7).all()
assert validated["review_period_weeks"].eq(1).all()

print()
print("Inventory policy")
print("✓ Rows: 121,472")
print("✓ SKUs: 30,368")
print("✓ q80 validated configuration: 30,368")
print("✓ q90/q95/q99 scenario rows: 91,104")
print("✓ Lead time: 7 days")
print("✓ Review period: 1 week")
print("✓ validated_configuration semantics preserved")

# -----------------------------------------------------------------------------
# Explainability
# -----------------------------------------------------------------------------

assert set(
    explainability["explanation_type"].dropna().unique()
) <= {
    "shap",
    "routing",
}

print()
print("Explainability")
print("✓ Rows: 66,558")
print("✓ SKUs: 30,490")
print("✓ Frozen SHAP/routing explanations")

# -----------------------------------------------------------------------------
# Portfolio risk
# -----------------------------------------------------------------------------

print()
print("Portfolio risk")
print(f"✓ Artifact: {risk_candidates[0].name}")
print(f"✓ Rows: {len(risk):,}")
print(f"✓ SKUs: {risk['id'].nunique():,}")

# -----------------------------------------------------------------------------
# Model-result checks
# -----------------------------------------------------------------------------

forecast_rows = model_results[
    model_results["result_type"].isin(
        ["forecast_model", "model_variant"]
    )
]

stability_rows = model_results[
    model_results["result_type"] == "stability"
]

selected_rows = model_results[
    (model_results["selected"] == True)
    & (
        model_results["result_type"].isin(
            ["forecast_model", "model_variant"]
        )
    )
]

assert len(forecast_rows) == 9
assert len(stability_rows) == 12

assert (
    selected_rows[
        ["model", "variant"]
    ]
    .drop_duplicates()
    .shape[0]
    == 1
)

selected_model = selected_rows[
    ["model", "variant"]
].drop_duplicates()

assert selected_model.iloc[0]["model"] == "LightGBM / Tweedie"
assert selected_model.iloc[0]["variant"] == "Raw"

assert (
    model_results.loc[
        model_results["result_type"] == "stability",
        "selected",
    ]
    == False
).all()

print()
print("Model results")
print("✓ Forecast/model rows: 9")
print("✓ Stability rows: 12")
print("✓ Selected model: LightGBM / Tweedie Raw")

# -----------------------------------------------------------------------------
# Inventory-result checks
# -----------------------------------------------------------------------------

service_rows = inventory_results[
    inventory_results["result_type"] == "service_level"
]

regime_rows = inventory_results[
    inventory_results["result_type"] == "regime_performance"
]

distribution_rows = inventory_results[
    inventory_results["result_type"] == "regime_distribution"
]

overall_rows = inventory_results[
    inventory_results["result_type"] == "overall_policy"
]

assert len(service_rows) == 6

assert set(service_rows["service_level"]) == {
    "q50",
    "q75",
    "q80",
    "q90",
    "q95",
    "q99",
}

assert (service_rows["n_skus"] == 30_368).all()

assert len(regime_rows) == 4
assert len(distribution_rows) == 4
assert len(overall_rows) == 1

assert distribution_rows["n_skus"].sum() == 30_490

print()
print("Inventory results")
print("✓ Six service-level scenario rows")
print("✓ 30,368 SKUs per service level")
print("✓ Four regime-performance rows")
print("✓ Four regime-distribution rows")
print("✓ One overall policy row")

# -----------------------------------------------------------------------------
# Cost / policy trade-off checks
# -----------------------------------------------------------------------------

assert len(cost_sensitivity) == 24
assert cost_sensitivity["scenario_id"].nunique() == 24

assert len(policy_tradeoffs) == 24
assert policy_tradeoffs["scenario_id"].nunique() == 24

expected_stockout_mult = {
    1.5,
    2.0,
    3.0,
    4.0,
    6.0,
    8.0,
}

expected_carry_rate = {
    0.15,
    0.20,
    0.25,
    0.30,
}

assert set(
    cost_sensitivity["stockout_cost_multiplier"]
) == expected_stockout_mult

assert set(
    cost_sensitivity["carrying_cost_rate"]
) == expected_carry_rate

assert set(
    policy_tradeoffs["stockout_cost_multiplier"]
) == expected_stockout_mult

assert set(
    policy_tradeoffs["carrying_cost_rate"]
) == expected_carry_rate

assert set(
    policy_tradeoffs["static_optimal_level"]
) == {"q99"}

print()
print("Cost sensitivity")
print("✓ 24 scenarios")

print("Policy trade-offs")
print("✓ 24 scenarios")

# -----------------------------------------------------------------------------
# Historical snapshot checks
# -----------------------------------------------------------------------------

for df_name, df in {
    "sku_metadata": sku_metadata,
    "regime_forecasts": regime_forecasts,
    "explainability": explainability,
    "risk": risk,
}.items():
    if "date" in df.columns:
        dates = pd.to_datetime(
            df["date"],
            errors="coerce",
        ).dropna()

        if len(dates):
            assert (
                dates.max().strftime("%Y-%m-%d")
                == "2016-01-31"
            )

print()
print("Historical snapshot")
print("✓ Application snapshot: 2016-01-31")

# -----------------------------------------------------------------------------
# Finalize manifest
# -----------------------------------------------------------------------------

manifest["finalized"] = True

with open(
    required_files["manifest"],
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False,
    )

with open(
    required_files["manifest"],
    "r",
    encoding="utf-8",
) as f:
    finalized_manifest = json.load(f)

assert finalized_manifest["finalized"] is True

# -----------------------------------------------------------------------------
# Release QA record
# -----------------------------------------------------------------------------

qa_record = {
    "status": "PASS",
    "finalized": True,
    "finalized_at_utc": datetime.now(timezone.utc).isoformat(),
    "final_sku_population": 30_490,
    "inventory_policy_sku_population": 30_368,
    "inventory_policy_rows": 121_472,
    "validated_q80_rows": 30_368,
    "scenario_policy_rows": 91_104,
    "forecast_sku_population": 8_863,
    "forecast_rows": 3_234_995,
    "explainability_rows": 66_558,
    "risk_rows": 30_490,
    "model_result_rows": len(model_results),
    "inventory_result_rows": len(inventory_results),
    "cost_sensitivity_rows": 24,
    "policy_tradeoff_rows": 24,
    "selected_model": "LightGBM / Tweedie Raw",
    "primary_service_level": "q80",
    "forecast_data_as_of": "2016-01-31",
    "risk_artifact": required_files["risk"].name,
    "inventory_policy_service_levels": [
        0.80,
        0.90,
        0.95,
        0.99,
    ],
    "inventory_policy_lead_time_days": 7,
    "inventory_policy_review_period_weeks": 1,
    "validated_configuration": "q80 only",
    "no_new_modeling": True,
    "no_new_simulation": True,
    "no_new_optimization": True,
}

with open(
    APP_DIR / "app_release_qa.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        qa_record,
        f,
        indent=2,
        ensure_ascii=False,
    )

# -----------------------------------------------------------------------------
# Final output
# -----------------------------------------------------------------------------

print()
print("═" * 100)
print("SECTION 10 QA")
print("═" * 100)

print("✓ All required app artifacts exist")
print("✓ Final populations reconcile")
print("✓ Forecast coverage reconciles")
print("✓ Inventory-policy coverage reconciles")
print("✓ q80 is the validated configuration")
print("✓ q90/q95/q99 are preserved as scenario alternatives")
print("✓ Inventory-policy lead time/review period validated")
print("✓ Explainability coverage reconciles")
print("✓ Portfolio risk coverage reconciles")
print("✓ Historical model results reconcile")
print("✓ Inventory scenarios reconcile")
print("✓ Cost-sensitivity scenarios reconcile")
print("✓ Policy-tradeoff scenarios reconcile")
print("✓ Historical snapshot = 2016-01-31")
print("✓ Selected model = LightGBM / Tweedie Raw")
print("✓ Primary service level = q80")
print("✓ Manifest finalized = True")
print("✓ app_release_qa.json written")
print("✓ No new modeling, simulation, or optimization performed")
print()
print("APP PACKAGE STATUS: READY")
print("✓ Section 10 complete")

════════════════════════════════════════════════════════════════════════════════════════════════════
SECTION 10: FINAL APP PACKAGE & RELEASE QA
════════════════════════════════════════════════════════════════════════════════════════════════════

Artifact existence
✓ manifest: app_manifest.json
✓ sku_metadata: app_sku_metadata.parquet
✓ forecasts: app_forecasts.parquet
✓ regime_forecasts: app_regime_forecasts.parquet
✓ inventory_policy: app_inventory_policy.parquet
✓ explainability: app_explainability.parquet
✓ model_results: app_model_results.parquet
✓ inventory_results: app_inventory_results.parquet
✓ cost_sensitivity: app_cost_sensitivity.parquet
✓ policy_tradeoffs: app_policy_tradeoffs.parquet
✓ risk: app_portfolio_risk.parquet

Manifest
Current finalized flag: False

Core population reconciliation
✓ SKU master: 30,490
✓ Regime forecasts: 30,490
✓ Explainability: 30,490
✓ Risk artifact: 30,490

Forecast coverage
✓ Daily forecast rows: 3,234,995
✓ Forecast SKUs: 8,863
✓ Frozen missin

# Section 10 — Final App Package & Release QA

Final integrity gate for the app-ready historical dataset.

This section verifies artifact existence, population reconciliation, frozen model and policy configuration, scenario coverage, historical snapshot consistency, and manifest finalization.

No new forecasting, modeling, optimization, calibration, or simulation is performed.

The package is marked ready only after all release checks pass.


# Section 11 — Final App Handoff Summary

Provide a compact final inventory of the app-ready package for the Streamlit application.

This section does not transform the modeling data or create new forecasts. It records the final artifact inventory, core populations, frozen production configuration, historical snapshot, and package status for handoff to the application layer.

The app should load the finalized manifest first and then consume the frozen app artifacts.


In [71]:
# =============================================================================
# SECTION 11: FINAL APP HANDOFF SUMMARY
# =============================================================================

from pathlib import Path
import json
import pandas as pd

APP_DIR = Path("../data/processed/predictions/app")

print("═" * 100)
print("SECTION 11: FINAL APP HANDOFF SUMMARY")
print("═" * 100)

# -----------------------------------------------------------------------------
# Load finalized release QA
# -----------------------------------------------------------------------------

qa_path = APP_DIR / "app_release_qa.json"

assert qa_path.exists(), f"Missing release QA record: {qa_path}"

with open(qa_path, "r", encoding="utf-8") as f:
    qa = json.load(f)

assert qa["status"] == "PASS"
assert qa["finalized"] is True

# -----------------------------------------------------------------------------
# Load final manifest
# -----------------------------------------------------------------------------

manifest_path = APP_DIR / "app_manifest.json"

assert manifest_path.exists()

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

assert manifest.get("finalized") is True

# -----------------------------------------------------------------------------
# Load final app artifacts
# -----------------------------------------------------------------------------

sku_metadata = pd.read_parquet(
    APP_DIR / "app_sku_metadata.parquet"
)

regime_forecasts = pd.read_parquet(
    APP_DIR / "app_regime_forecasts.parquet"
)

forecasts = pd.read_parquet(
    APP_DIR / "app_forecasts.parquet"
)

inventory_policy = pd.read_parquet(
    APP_DIR / "app_inventory_policy.parquet"
)

explainability = pd.read_parquet(
    APP_DIR / "app_explainability.parquet"
)

risk_candidates = sorted(
    p for p in APP_DIR.glob("*risk*.parquet")
    if p.is_file()
)

assert len(risk_candidates) == 1

risk = pd.read_parquet(risk_candidates[0])

model_results = pd.read_parquet(
    APP_DIR / "app_model_results.parquet"
)

inventory_results = pd.read_parquet(
    APP_DIR / "app_inventory_results.parquet"
)

cost_sensitivity = pd.read_parquet(
    APP_DIR / "app_cost_sensitivity.parquet"
)

policy_tradeoffs = pd.read_parquet(
    APP_DIR / "app_policy_tradeoffs.parquet"
)

# -----------------------------------------------------------------------------
# Final population summary
# -----------------------------------------------------------------------------

summary = {
    "final_sku_population": int(
        sku_metadata["id"].nunique()
    ),
    "regime_forecast_population": int(
        regime_forecasts["id"].nunique()
    ),
    "forecast_population": int(
        forecasts["id"].nunique()
    ),
    "forecast_rows": int(
        len(forecasts)
    ),
    "inventory_policy_population": int(
        inventory_policy["id"].nunique()
    ),
    "inventory_policy_rows": int(
        len(inventory_policy)
    ),
    "explainability_population": int(
        explainability["id"].nunique()
    ),
    "explainability_rows": int(
        len(explainability)
    ),
    "risk_population": int(
        risk["id"].nunique()
    ),
    "model_result_rows": int(
        len(model_results)
    ),
    "inventory_result_rows": int(
        len(inventory_results)
    ),
    "cost_sensitivity_rows": int(
        len(cost_sensitivity)
    ),
    "policy_tradeoff_rows": int(
        len(policy_tradeoffs)
    ),
}

# -----------------------------------------------------------------------------
# Final configuration
# -----------------------------------------------------------------------------

selected_model = model_results[
    (
        model_results["selected"] == True
    )
    & (
        model_results["result_type"].isin(
            ["forecast_model", "model_variant"]
        )
    )
][
    ["model", "variant"]
].drop_duplicates()

assert len(selected_model) == 1

selected_model_name = (
    selected_model.iloc[0]["model"]
    + " "
    + selected_model.iloc[0]["variant"]
)

validated_policy = inventory_policy[
    inventory_policy["validated_configuration"]
]

assert len(validated_policy) == 30_368
assert set(
    validated_policy["service_level"].unique()
) == {0.80}

configuration = {
    "selected_model": selected_model_name,
    "primary_service_level": "q80",
    "validated_policy_service_level": 0.80,
    "lead_time_days": int(
        validated_policy["lead_time_days"].iloc[0]
    ),
    "review_period_weeks": int(
        validated_policy["review_period_weeks"].iloc[0]
    ),
    "forecast_data_as_of": qa["forecast_data_as_of"],
}

# -----------------------------------------------------------------------------
# Final regime distribution
# -----------------------------------------------------------------------------

regime_distribution = (
    sku_metadata["regime"]
    .value_counts()
    .sort_index()
    .to_dict()
)

assert sum(regime_distribution.values()) == 30_490

# -----------------------------------------------------------------------------
# Build handoff summary
# -----------------------------------------------------------------------------

handoff_summary = {
    "package_status": "READY",
    "manifest_finalized": True,
    "release_qa_status": qa["status"],
    "configuration": configuration,
    "populations": summary,
    "regime_distribution": regime_distribution,
    "risk_artifact": risk_candidates[0].name,
    "notes": [
        "Historical M5 decision-support package.",
        "Forecast data as of 2016-01-31.",
        "No new modeling or simulation performed in Sections 8-11.",
        "q80 is the validated primary inventory configuration.",
        "q90/q95/q99 remain scenario alternatives.",
    ],
}

# -----------------------------------------------------------------------------
# Write handoff summary FIRST
# -----------------------------------------------------------------------------

handoff_path = APP_DIR / "app_handoff_summary.json"

with open(
    handoff_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        handoff_summary,
        f,
        indent=2,
        ensure_ascii=False,
    )

assert handoff_path.exists()
assert handoff_path.stat().st_size > 0

# Verify JSON can be read back.
with open(
    handoff_path,
    "r",
    encoding="utf-8",
) as f:
    saved_handoff = json.load(f)

assert saved_handoff["package_status"] == "READY"
assert saved_handoff["manifest_finalized"] is True
assert saved_handoff["configuration"]["primary_service_level"] == "q80"
assert saved_handoff["configuration"]["lead_time_days"] == 7
assert saved_handoff["configuration"]["review_period_weeks"] == 1

assert (
    saved_handoff["populations"]["final_sku_population"]
    == 30_490
)

assert (
    saved_handoff["populations"]["inventory_policy_population"]
    == 30_368
)

assert (
    saved_handoff["populations"]["forecast_population"]
    == 8_863
)

# -----------------------------------------------------------------------------
# Build FINAL artifact inventory AFTER writing handoff JSON
# -----------------------------------------------------------------------------

app_files = sorted(
    p.name
    for p in APP_DIR.iterdir()
    if p.is_file()
)

required_final_files = {
    "app_manifest.json",
    "app_release_qa.json",
    "app_handoff_summary.json",
    "app_sku_metadata.parquet",
    "app_forecasts.parquet",
    "app_regime_forecasts.parquet",
    "app_inventory_policy.parquet",
    "app_explainability.parquet",
    "app_model_results.parquet",
    "app_inventory_results.parquet",
    "app_cost_sensitivity.parquet",
    "app_policy_tradeoffs.parquet",
    risk_candidates[0].name,
}

assert required_final_files.issubset(set(app_files))

# -----------------------------------------------------------------------------
# Final QA
# -----------------------------------------------------------------------------

print()
print("═" * 100)
print("FINAL APP ARTIFACTS")
print("═" * 100)

for filename in app_files:
    print(f"✓ {filename}")

print()
print(f"Final artifact count: {len(app_files)}")
print(
    "✓ Handoff JSON:",
    handoff_path.resolve(),
)

print()
print("═" * 100)
print("SECTION 11 QA")
print("═" * 100)

print("✓ Release QA status: PASS")
print("✓ Manifest finalized: True")
print("✓ Package status: READY")
print("✓ Final SKU population: 30,490")
print("✓ Inventory-policy population: 30,368")
print("✓ Forecast population: 8,863")
print("✓ Forecast data as of: 2016-01-31")
print("✓ Selected model: LightGBM / Tweedie Raw")
print("✓ Primary inventory configuration: q80")
print("✓ Lead time: 7 days")
print("✓ Review period: 1 week")
print("✓ q90/q95/q99 retained as scenarios")
print("✓ Regime populations reconcile to 30,490")
print("✓ app_handoff_summary.json exists and was verified")
print(f"✓ Handoff path: {handoff_path.resolve()}")
print("✓ No new forecasting, modeling, optimization, or simulation performed")
print()
print("FINAL APP HANDOFF STATUS: READY")
print("✓ Section 11 complete")

════════════════════════════════════════════════════════════════════════════════════════════════════
SECTION 11: FINAL APP HANDOFF SUMMARY
════════════════════════════════════════════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════════════════════════════════════════════
FINAL APP ARTIFACTS
════════════════════════════════════════════════════════════════════════════════════════════════════
✓ app_cost_sensitivity.parquet
✓ app_explainability.parquet
✓ app_forecasts.parquet
✓ app_handoff_summary.json
✓ app_inventory_policy.parquet
✓ app_inventory_results.parquet
✓ app_manifest.json
✓ app_model_results.parquet
✓ app_policy_tradeoffs.parquet
✓ app_portfolio_risk.parquet
✓ app_regime_forecasts.parquet
✓ app_release_qa.json
✓ app_sku_metadata.parquet

Final artifact count: 13
✓ Handoff JSON: C:\Apps\Expense-Time-Series\data\processed\predictions\app\app_handoff_summary.json

════════════════════════════════════════════════

# Section 11 — Final App Handoff Summary

The app-ready package is finalized and passed release QA.

The final package contains the frozen SKU metadata, forecasts, regime forecasts, inventory policies, explainability, portfolio risk, historical model results, inventory results, cost-sensitivity scenarios, and policy trade-offs.

Final coverage and configuration:

* 30,490 final SKUs
* 30,368 SKUs with frozen inventory-policy coverage
* 8,863 SKUs with frozen daily forecast trajectories
* Forecast data as of January 31, 2016
* LightGBM/Tweedie Raw as the selected production forecasting model
* q80 as the validated primary inventory configuration
* 7-day lead time
* 1-week review period
* q90/q95/q99 retained as scenario alternatives

The package is historical decision-support data rather than a live retailer deployment. No additional forecasting, modeling, optimization, calibration, or simulation is performed in the app-preparation stage.

The final handoff record is written to:

`app_handoff_summary.json`

Package status: **READY**.
